<div style="background: linear-gradient(135deg, #0b1021, #14213d, #1b2a4a); border-radius: 16px; padding: 36px 40px; margin-bottom: 8px;">
  <h1 style="color: #8ecae6; font-size: 2.3em; font-weight: 800; margin: 0 0 10px 0; letter-spacing: 0.5px;">
    &#9889; iTransformer &middot; Walk-Forward BTCUSDT 1h
  </h1>
  <p style="color: #ffb703; font-size: 1.12em; margin: 0 0 18px 0; font-weight: 500;">
    Nominal Variates or Effective Dimensionality? &mdash; 15 origins &middot; 684 runs &middot; 2 &times; T4
  </p>
  <hr style="border: none; border-top: 1px solid #2a4365; margin: 16px 0;">
  <p style="color: #a8c0dd; font-size: 0.97em; margin: 0 0 12px 0;">
    <strong>Self-contained.</strong> This notebook needs exactly two things: itself, and
    <code>BTCUSDT_1h.parquet</code> attached as a Kaggle Dataset. Every definition it uses is
    <em>in</em> it &mdash; the cells below are ordinary <code>def</code>, <code>class</code> and
    constant bodies, run top to bottom. Nothing is written to disk to be imported back, there is
    no <code>itransformer_btc</code> package on this machine, and no <code>src/</code> on
    <code>sys.path</code>.
  </p>
  <p style="color: #a8c0dd; font-size: 0.97em; margin: 0 0 12px 0;">
    <strong>It is still a launcher, not a program.</strong> Every definition &mdash; the twelve
    variates, the segment law, the window semantics, the scaler, the model, the metrics &mdash; is
    authored in <code>src/itransformer_btc/</code> and unit-tested on CPU; the cells below are a
    transcription, not an authoring surface. <strong>Do not hand-edit the <em>Definitions</em>
    cells:</strong> they are generated by <code>tools/build_notebook.py</code>,
    <code>tests/test_notebook_sync.py</code> fails the moment they diverge from the package, and
    the next generator run reverts the edit.
  </p>
  <p style="color: #7f9cc0; font-size: 0.93em; margin: 0;">
    Answers <strong>RQ1</strong> (does benefit track K or K<sub>eff</sub>?),
    <strong>RQ2</strong> (does the multivariate gap narrow with model age?),
    <strong>RQ3</strong> (what retraining cadence?) &mdash; all three pre-registered before any
    model ran, and none of them changeable now without declaring a new experiment.
  </p>
</div>

<div style="background: linear-gradient(90deg, #0b1021, #112240); border-left: 4px solid #8ecae6; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #8ecae6; margin: 0 0 8px 0;">&#128295; 0 &middot; Setup</h2>
  <p style="color: #b8c7e0; margin: 0;">Find the immutable artifact by globbing, never by dataset slug. Install only what the Kaggle image lacks.</p>
  <p style="color: #7f9cc0; margin: 12px 0 0 0; font-size: 0.9em;">Kaggle ships its own torch and
    pyarrow; pinning them against a local venv is forbidden. <code>/kaggle/input</code> is read-only,
    everything is written to <code>/kaggle/working</code>. The parquet is <strong>not</strong>
    re-downloaded here even though Stage 1 could: a fresh download is a new vintage, and &sect;12
    forbids numbers from two vintages sharing a table.</p>
</div>

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

# Kaggle's 12 h wall runs from HERE, not from the moment the grid starts. The
# budget guard counts from whatever it is handed, so the prelude — data, K_eff,
# invariants, the twelve pilot runs — would sit outside the budget entirely and
# the two clocks would drift apart by however long it took. The grid cell
# subtracts this. Losing /kaggle/working to the wall costs the whole session's
# runs, so the margin is not somewhere to be approximate.
SESSION_T0 = time.perf_counter()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()).resolve()
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Relative paths inside the definitions below resolve against the process working
# directory, so artifacts have to land beside it whichever machine this is.
# Kaggle already starts in /kaggle/working; a local run started from notebooks/
# does not. Nothing is added to sys.path — there is no package to import, and an
# entry there could only serve to shadow these cells with someone else's copy.
if Path.cwd().resolve() != WORK:
    os.chdir(WORK)


def ensure(module: str, pip_name: str | None = None) -> None:
    """Install only what is genuinely missing.

    Kaggle ships torch, numpy, pyarrow and usually polars. Pinning any of them
    against a local venv is forbidden by root section 16 — the image wins.
    """
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or module]
        )


for _mod in ("polars", "pyarrow", "numpy", "torch"):
    ensure(_mod)


def find_parquet() -> Path:
    """Locate BTCUSDT_1h.parquet by globbing — never by Kaggle dataset slug.

    Root section 10.5: discovery is by glob so the Dataset can be renamed without
    editing anything. Both upload shapes are covered — the four Stage 1 files
    uploaded flat, and the whole repository uploaded with data/raw/ inside it.
    """
    patterns = (
        "data/raw/BTCUSDT_1h.parquet",
        "*/data/raw/BTCUSDT_1h.parquet",
        "BTCUSDT_1h.parquet",
        "*/BTCUSDT_1h.parquet",
        "*/*/BTCUSDT_1h.parquet",
    )
    roots = [WORK, Path("/kaggle/input")] if ON_KAGGLE else [WORK, WORK.parent]
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for hit in sorted(root.glob(pattern)):
                return hit.resolve()
    raise FileNotFoundError(
        "BTCUSDT_1h.parquet not found under "
        f"{[str(r) for r in roots]}. Attach data/raw/ as a Kaggle Dataset. "
        "It is NOT re-downloaded here on purpose: a fresh download is a new "
        "vintage, and section 12 forbids numbers from two vintages sharing a table."
    )


PARQUET = find_parquet()
# Every meta/*.json records the digest of the artifact its run consumed (section
# 12), and can only find it if told.
os.environ["ITBTC_PARQUET"] = str(PARQUET)

import numpy as np
import polars as pl
import torch

print(f"work      {WORK}")
print(f"parquet   {PARQUET}  ({PARQUET.stat().st_size / 1e6:.1f} MB)")
print(f"artifacts {ARTIFACTS}")
print(f"polars {pl.__version__} | torch {torch.__version__} | numpy {np.__version__}")
print(f"CUDA devices: {torch.cuda.device_count()}")
for _i in range(torch.cuda.device_count()):
    _cap = torch.cuda.get_device_capability(_i)
    print(f"  cuda:{_i}  {torch.cuda.get_device_name(_i)}  sm_{_cap[0]}{_cap[1]}")

# Root section 10.3: never gate precision on torch.cuda.is_bf16_supported(). It
# defaults to including_emulation=True and returns True on a T4 (sm_75),
# selecting an emulated bf16 path *slower than fp32*. Gate on capability.
if torch.cuda.is_available():
    print(f"native bf16: {torch.cuda.get_device_capability(0)[0] >= 8}  "
          f"(is_bf16_supported() says {torch.cuda.is_bf16_supported()} "
          f"and is not to be trusted here)")


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 4px solid #52b788; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #52b788; margin: 0 0 8px 0;">&#128230; 0b &middot; Definitions</h2>
  <p style="color: #b8c7e0; margin: 0;">Thirteen cells, one per module, in dependency order. Plain definitions &mdash; run them and every name the rest of the notebook calls exists. Generated from <code>src/</code>; do not hand-edit.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li><strong>Definitions, not files.</strong> The earlier launcher wrote these modules to
    disk with <code>%%writefile</code> and imported them back, because the grid ran as two
    subprocesses and a subprocess inherits none of this kernel's namespace. Measured on Kaggle, the
    completed grid was <strong>534 runs in 2.31 h at ~30 s per run</strong> against a &sect;10.3
    estimate of 60&ndash;100 s and 10&ndash;20 h (<code>D57</code>), so one process running the grid
    in sequence is ~4.5 h &mdash; inside the 11 h budget with room. The subprocesses stopped paying
    for themselves, and the files existed only to feed them.</li>
    <li><strong>Order is load-bearing now.</strong> These cells are <em>executed</em>, not merely
    written: decorators run, dataclass field types resolve, module constants evaluate. A cell naming
    something a later cell defines fails at once rather than at call time. <em>Save Version &rarr;
    Save &amp; Run All</em> runs them in order, which is the only order that works.</li>
    <li><strong>Two edits, and only two.</strong> Intra-package imports are removed &mdash; the names
    are already in this namespace, and the import would fail for want of a package. And
    <code>runner</code>'s <code>if __name__ == "__main__":</code> guard is removed: in a notebook
    cell <code>__name__</code> <em>is</em> <code>"__main__"</code>, so it would launch the entire
    grid the instant its definition cell ran. Everything else is the package, character for
    character.</li>
    <li><strong>The digest is the provenance.</strong> There is no git repository on Kaggle and now
    no package files either, so the cell after these pins <code>code_sha256</code> to the digest of
    <code>src/itransformer_btc/</code> taken at generation time &mdash; the same number a local
    checkout of the same source reports (<code>D54b</code>), so a notebook run and a repository run
    do not look like different code vintages.</li>
  </ul>
</div>

In [ ]:
# ═══ config.py ════════
"""Design constants and the walk-forward origin grid.

Every number here is fixed by ``CLAUDE.md`` and none of it may be tuned. The
module exists so that no magic number is buried in pipeline code (root §16) and
so that a change to the design is a one-line diff with a visible blast radius.

The origin grid is *derived*, never transcribed: transcribing it is how the
13-origin figure survived in four documents after `D26` replaced it.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Final

# -- data window (root §4.1) -------------------------------------------------

DATA_START: Final = datetime(2018, 1, 1, tzinfo=timezone.utc)
DATA_END: Final = datetime(2026, 8, 1, tzinfo=timezone.utc)  # EXCLUSIVE

BARS_EXPECTED: Final = 75_216
BARS_ACTUAL: Final = 75_094
MISSING_BARS: Final = 122
GAP_BLOCKS: Final = 27

# -- model geometry (root §6.2) ----------------------------------------------

SEQ_LEN: Final = 96   # L — 4 days of lookback
PRED_LEN: Final = 24  # H — headline horizon

#: A window occupies ``L + H`` consecutive bars, so a break inside a span
#: destroys ``L + H - 1`` *start positions*, not ``L + H``. Root §4.3 turns on
#: this off-by-one: across 30 breaks it is a 30-window difference in the
#: assertion target, the same size as the drift the assertion exists to catch.
WINDOW_SPAN: Final = SEQ_LEN + PRED_LEN         # 120
STARTS_LOST_PER_BREAK: Final = WINDOW_SPAN - 1  # 119

# -- walk-forward protocol (root §8.1) ---------------------------------------

TRAIN_MONTHS: Final = 24      # fixed rolling window, never expanding
VAL_MONTHS: Final = 3         # final 3 months of the training window
TRAIN_SUB_MONTHS: Final = TRAIN_MONTHS - VAL_MONTHS  # 21 — where the scaler is fit
TEST_BLOCKS: Final = 6
BLOCK_DAYS: Final = 30
BLOCK_HOURS: Final = BLOCK_DAYS * 24  # 720 window starts per block

#: 5, not 6. The calendar month a block lands on is ``m0 + s*i + (b-1) mod 12``,
#: so for fixed ``b`` the months visited form a coset of size ``12/gcd(s,12)``.
#: At s=6 that is 2 months, and a significant beta1 becomes observationally
#: equivalent to "February and August are harder" — a bias no post-hoc analysis
#: removes. Only s coprime to 12 fully decouples; 5 maximises the origin count
#: among those. Root §8.1 / `D26`.
ORIGIN_SPACING_MONTHS: Final = 5

FIRST_ORIGIN: Final = datetime(2020, 1, 1, tzinfo=timezone.utc)

# -- variate ladder (root §5.2) ----------------------------------------------

K_LADDER: Final = (1, 4, 8, 12)
SEEDS: Final = (42, 43, 44, 45, 46)

#: Horizons swept in root §10.2's 192-run arm. H=24 is the headline.
HORIZONS: Final = (1, 3, 24, 168)

#: Origins the horizon sweep runs at, **named in advance** (`D48`). Choosing
#: them after the main grid would be origin selection.
SWEEP_ORIGIN_INDICES: Final = (1, 5, 10, 15)

#: Offset of the falsification arm's fresh model (root §8.1). Exactly 90 days,
#: not 3 calendar months: test blocks are 30 **days**, so only 90 days lands the
#: fresh origin precisely on the aged model's block-4 boundary, which is what
#: makes "the *same* calendar blocks 4-6" true rather than approximately true.
FRESH_OFFSET_DAYS: Final = 90


def add_months(when: datetime, months: int) -> datetime:
    """Shift ``when`` by whole calendar months, keeping the day of month.

    Every boundary in this study falls on the first of a month, so the
    day-clamping question a general implementation must answer never arises.
    This raises rather than clamping if it ever does: a silently clamped
    boundary moves a split by a day and no assertion downstream would notice.

    Args:
        when: A timezone-aware datetime on the first of some month.
        months: Whole months to add; may be negative.

    Returns:
        The shifted datetime, same tzinfo and time of day.

    Raises:
        ValueError: If ``when`` is not on the first of a month.
    """
    if when.day != 1:
        raise ValueError(
            f"add_months is only used on month boundaries in this study; got "
            f"day={when.day}. Clamping rules would silently move a split."
        )
    total = when.month - 1 + months
    return when.replace(year=when.year + total // 12, month=total % 12 + 1)


@dataclass(frozen=True, slots=True)
class Origin:
    """One walk-forward origin and every boundary derived from it.

    All boundaries are half-open ``[start, end)``. The origin is both the end of
    validation and the start of testing: a forecaster standing at ``o`` has seen
    everything before ``o`` and nothing after it.
    """

    index: int
    origin: datetime

    @property
    def train_start(self) -> datetime:
        """Start of the 24-month rolling training window."""
        return add_months(self.origin, -TRAIN_MONTHS)

    @property
    def train_sub_end(self) -> datetime:
        """End of the 21-month sub-block; equivalently ``val_start``.

        The scaler is fit on this sub-block and nothing else, and training
        windows are enumerated over it and nothing else (`D25`). The 24-month
        count — ~17,400 windows, ~80 MB — is what you get by training on the
        validation months too, which is `D24`'s leak wearing a sample-count
        disguise.
        """
        return add_months(self.origin, -VAL_MONTHS)

    @property
    def val_start(self) -> datetime:
        return self.train_sub_end

    @property
    def val_end(self) -> datetime:
        return self.origin

    @property
    def test_start(self) -> datetime:
        return self.origin

    @property
    def test_end(self) -> datetime:
        return self.origin + timedelta(days=BLOCK_DAYS * TEST_BLOCKS)

    def block(self, b: int) -> tuple[datetime, datetime]:
        """Half-open bounds of test block ``b``, one-indexed as in the paper."""
        if not 1 <= b <= TEST_BLOCKS:
            raise ValueError(f"block index must be in 1..{TEST_BLOCKS}, got {b}")
        start = self.origin + timedelta(days=BLOCK_DAYS * (b - 1))
        return start, start + timedelta(days=BLOCK_DAYS)

    def blocks(self) -> list[tuple[int, datetime, datetime]]:
        """Every test block as ``(label, start, end)``, label one-indexed.

        The label is carried rather than inferred from position because the
        falsification arm evaluates blocks 4-6 and nothing else: there, the
        first tensor in the tuple is block **4**, and writing it out as block 1
        would silently re-index the arm the comparison depends on.
        """
        return [(b, *self.block(b)) for b in range(1, TEST_BLOCKS + 1)]

    @property
    def label(self) -> str:
        """``YYYY-MM`` — the form used in every table and figure."""
        return self.origin.strftime("%Y-%m")


@dataclass(frozen=True, slots=True)
class FalsificationOrigin:
    """A model trained fresh at ``o_i + 90 days``, scored on blocks 4-6.

    Root §8.1's pre-registered falsification arm, and **the only design in the
    study that identifies decay directly**. If the aged-minus-fresh gap is zero
    while beta1 < 0, then beta1 is calendar, not age — the aged model is not
    decaying, the market simply got harder in months 4-6, and RQ2's headline
    would be an artefact.

    Every training boundary is the base origin's, shifted by the same 90 days,
    so the fresh model trains on a window of **identical duration** to the aged
    one. Re-deriving the window from a 24-month subtraction instead would land
    on 2020-03-31-style dates, where the day-of-month clamping question
    :func:`add_months` refuses to answer silently would arise for the first time
    in this study — and a clamped boundary moves a split by a day with no
    assertion downstream to notice.

    Its validation sub-block overlaps the aged model's test blocks 1-3. That is
    not a leak: this is a *different* model, standing at a later origin, and a
    forecaster there has legitimately seen everything before ``o_i + 90 days``.
    """

    base: Origin
    offset_days: int = FRESH_OFFSET_DAYS

    @property
    def index(self) -> int:
        return self.base.index

    @property
    def _shift(self) -> timedelta:
        return timedelta(days=self.offset_days)

    @property
    def origin(self) -> datetime:
        return self.base.origin + self._shift

    @property
    def train_start(self) -> datetime:
        return self.base.train_start + self._shift

    @property
    def train_sub_end(self) -> datetime:
        return self.base.train_sub_end + self._shift

    @property
    def val_start(self) -> datetime:
        return self.train_sub_end

    @property
    def val_end(self) -> datetime:
        return self.origin

    @property
    def test_start(self) -> datetime:
        return self.origin

    @property
    def test_end(self) -> datetime:
        return self.base.test_end

    def blocks(self) -> list[tuple[int, datetime, datetime]]:
        """The **base** origin's blocks 4-6, keeping their original labels.

        ``o_i + 90 days`` is exactly where base block 4 opens, so these are the
        same calendar hours the aged model was scored on — which is the entire
        content of the comparison.
        """
        return [(b, *self.base.block(b)) for b in (4, 5, 6)]

    @property
    def label(self) -> str:
        return f"{self.base.label}+{self.offset_days}d"


#: Anything :func:`itransformer_btc.splits.build_origin_tensors` accepts. The
#: two share an interface rather than an inheritance chain because they share no
#: implementation: one derives its boundaries from calendar months, the other by
#: shifting another origin's.
OriginLike = Origin | FalsificationOrigin


def origin_grid(
    first: datetime = FIRST_ORIGIN,
    spacing_months: int = ORIGIN_SPACING_MONTHS,
    data_start: datetime = DATA_START,
    data_end: datetime = DATA_END,
) -> list[Origin]:
    """Derive every origin that fits inside the data window.

    An origin is admissible when its training window starts no earlier than the
    data and its sixth test block ends no later than the data:
    ``o - 24 months >= data_start`` and ``o + 180 days <= data_end``.

    Under the committed constants this yields **15** origins, 2020-01 … 2025-11.
    The count is derived rather than written down so that changing the spacing
    changes the grid instead of leaving a stale integer behind — which is
    exactly what happened to the superseded 13.

    Returns:
        Origins in chronological order, ``index`` one-based.

    Raises:
        ValueError: If the first origin would need data from before the window.
    """
    grid: list[Origin] = []
    candidate = first
    while True:
        origin = Origin(index=len(grid) + 1, origin=candidate)
        if origin.train_start < data_start:
            raise ValueError(
                f"origin {origin.label} needs training data from "
                f"{origin.train_start.date()}, before the data window opens at "
                f"{data_start.date()}"
            )
        if origin.test_end > data_end:
            break
        grid.append(origin)
        candidate = add_months(candidate, spacing_months)
    return grid


#: Materialised once. Import this rather than rebuilding it — a second call with
#: different arguments silently produces a different study.
ORIGINS: Final = origin_grid()


In [ ]:
# ═══ __init__.py ════════
"""Data plane and experiment scaffolding for the spot-only iTransformer study.

Root ``CLAUDE.md`` is the project law, and it is the *only* one: the
subdirectory files were deleted on 2026-08-06 because a rule that loads solely
when a file in its subtree is opened is absent exactly when an agent reasons
about the area without opening one. Two of those rules shape every module here:

* **polars only.** Its rolling API is backward-closed by construction, so the
  ``center=True`` leak class is unrepresentable. Stage 1 ingest
  (``spot_klines_btc.py``, at the repository root) is the one documented
  exemption and lives outside this package.
* **Fail loudly.** A schema mismatch, a window-count mismatch or a hash mismatch
  raises. The anti-leakage checklist is ``assert``s wherever it can be, because
  a checklist that lives only in prose is a checklist nobody runs.
"""

from __future__ import annotations


__all__ = [
    "ORIGINS",
    "Origin",
    "PRED_LEN",
    "SEQ_LEN",
    "STARTS_LOST_PER_BREAK",
    "WINDOW_SPAN",
    "origin_grid",
]


In [ ]:
# ═══ segments.py ════════
"""The segment law: what breaks the series, and where.

A **segment** is a maximal run of contiguous, usable hourly bars. Root §4.3
breaks the series at two kinds of position:

1. **any missing bar** — 27 downtime blocks, 122 bars. When the exchange is down
   no price forms, so there is nothing to infer and imputation is *undefined*,
   not merely risky (root §4.2);
2. **any zero-volume or ``high == low`` bar** — it carries no trade information,
   exactly like downtime, so it gets the same treatment. This is what makes
   ``(VWAP - C)/(H - L)`` and ``log(volume)`` total rather than partial
   functions (`D14`), and why the F2 estimators are strictly positive and their
   logs total (root §5.1).

The ``high == low`` count has never been measured — root §4.3 and
``docs/ORIGIN_WINDOW_BUDGET.md`` both flag it as assumed. :func:`break_summary`
measures it, which is why this module exists before any model does.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Final

import polars as pl


#: One hour in the integer domain every timestamp comparison uses. Root §2:
#: "Every timestamp is epoch-based and compared as an integer."
HOUR_MS: Final = 3_600_000

DEFAULT_PARQUET: Final = Path("data/raw/BTCUSDT_1h.parquet")


@dataclass(frozen=True, slots=True)
class Segment:
    """A maximal run of contiguous usable bars, as half-open row indices.

    Attributes:
        start_row: First row index into the *usable* frame, inclusive.
        end_row: One past the last row index, exclusive.
        start_ts: Epoch ms of the first bar.
        end_ts: Epoch ms of the last bar — inclusive, because this is a bar and
            not a bound.
    """

    start_row: int
    end_row: int
    start_ts: int
    end_ts: int

    @property
    def n_bars(self) -> int:
        return self.end_row - self.start_row

    def window_starts(self, span: int) -> int:
        """How many ``span``-bar windows start inside this segment.

        A segment shorter than ``span`` contributes *zero*, never a negative
        number. The closed-form budget arithmetic in root §4.3 quietly assumes
        every segment clears ``span``; where it does not, the two disagree and
        :mod:`itransformer_btc.budget` reports the disagreement rather than
        papering over it.
        """
        return max(0, self.n_bars - span + 1)


def load_bars(path: Path | str = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the immutable Stage 1 artifact, sorted, with an epoch-ms column.

    Args:
        path: Parquet written by ``spot_klines_btc.py``.

    Returns:
        Every original column plus ``ts_ms`` (Int64 epoch milliseconds), sorted
        ascending.

    Raises:
        FileNotFoundError: If the artifact is absent.
        ValueError: If the frame is empty, carries duplicate timestamps, or
            reaches outside the declared half-open data window. That last check
            is the runnable form of `D33`: the boundary bar at
            ``2026-08-01T00:00`` sat one hour past the window and shifted every
            count derived from ``len(df)``.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. The four Stage 1 artifacts live in data/raw/ "
            f"(`D33`); regenerate with "
            f"`python spot_klines_btc.py --rebuild-only --outdir ./data/raw`."
        )

    frame = (
        pl.read_parquet(path)
        .with_columns(pl.col("open_time").dt.epoch("ms").alias("ts_ms"))
        .sort("ts_ms")
    )

    if frame.height == 0:
        raise ValueError(f"{path} is empty")

    n_unique = frame.select(pl.col("ts_ms").n_unique()).item()
    if n_unique != frame.height:
        raise ValueError(
            f"{path} carries {frame.height - n_unique} duplicate timestamps; "
            f"Stage 1 de-duplicates, so this is not Stage 1 output"
        )

    lo, hi = frame.select(
        pl.col("ts_ms").min().alias("lo"), pl.col("ts_ms").max().alias("hi")
    ).row(0)
    window_lo = int(DATA_START.timestamp() * 1000)
    window_hi = int(DATA_END.timestamp() * 1000)
    if lo < window_lo or hi >= window_hi:
        raise ValueError(
            f"{path} reaches outside the half-open data window "
            f"[{DATA_START.isoformat()}, {DATA_END.isoformat()}): "
            f"first={lo} last={hi}. `D33` — re-emit with --rebuild-only, which "
            f"applies clip_to_window()."
        )
    return frame


def usable_mask(frame: pl.DataFrame) -> pl.DataFrame:
    """Flag each bar usable or not, with the reason attached.

    A bar is unusable when it carries no trade information: zero volume, or
    ``high == low`` (no intrabar range). Both are treated exactly like downtime
    by root §4.3 — excluded, and the series splits there.

    ``zero_trades`` is measured but does **not** by itself mark a bar unusable:
    root §4.3 names only zero-volume and ``H == L``. It is carried so the open
    question in ``docs/ORIGIN_WINDOW_BUDGET.md`` — whether the 3 zero-volume and
    3 zero-trade bars are the same 3 bars — can be answered rather than assumed.

    Returns:
        The input frame plus boolean ``zero_volume``, ``flat_bar``,
        ``zero_trades`` and ``usable``.
    """
    return frame.with_columns(
        (pl.col("volume") <= 0).alias("zero_volume"),
        (pl.col("high") <= pl.col("low")).alias("flat_bar"),
        (pl.col("trades") <= 0).alias("zero_trades"),
    ).with_columns(
        (~pl.col("zero_volume") & ~pl.col("flat_bar")).alias("usable")
    )


def build_segments(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
) -> list[Segment]:
    """Split the usable bars of ``[start, end)`` into contiguous segments.

    A new segment begins wherever the previous usable bar is not exactly one
    hour earlier. That covers downtime and exclusion alike: an excluded bar has
    already been filtered out, so it shows up here as a time jump.

    Args:
        frame: Output of :func:`usable_mask`, or anything carrying ``ts_ms`` and
            ``usable``.
        start: Inclusive lower bound; ``None`` for unbounded.
        end: Exclusive upper bound; ``None`` for unbounded.

    Returns:
        Segments in chronological order; empty if the span holds no usable bar.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    span = frame.filter(pl.col("usable"))
    if start is not None:
        span = span.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        span = span.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    if span.height == 0:
        return []

    ts = span.get_column("ts_ms").to_list()
    segments: list[Segment] = []
    seg_start = 0
    for i in range(1, len(ts)):
        if ts[i] - ts[i - 1] != HOUR_MS:
            segments.append(Segment(seg_start, i, ts[seg_start], ts[i - 1]))
            seg_start = i
    segments.append(Segment(seg_start, len(ts), ts[seg_start], ts[-1]))
    return segments


@dataclass(frozen=True, slots=True)
class BreakSummary:
    """Measured break profile of a span. Every field is counted, not assumed."""

    calendar_hours: int
    bars_present: int
    bars_usable: int
    missing_bars: int
    zero_volume_bars: int
    flat_bars: int
    zero_trade_bars: int
    excluded_positions: int
    break_runs: int

    @property
    def segments(self) -> int:
        """Segments the span splits into.

        ``break_runs + 1`` only when every run is interior. A run touching
        either edge of the span produces one fewer segment, so this counts
        segments directly from the run structure rather than assuming.
        """
        return max(1, self.break_runs + 1)

    @property
    def window_starts_lost(self) -> int:
        """``119 x break_runs + excluded_positions`` — root §4.3's cost model."""
        return STARTS_LOST_PER_BREAK * self.break_runs + self.excluded_positions


def break_summary(
    frame: pl.DataFrame,
    start: datetime,
    end: datetime,
) -> BreakSummary:
    """Measure every break-inducing condition in ``[start, end)``.

    A **break run** is a maximal contiguous stretch of excluded calendar
    positions, whether excluded because the bar is missing or because it is
    unusable. Runs, not bars, are what the cost model charges 119 window starts
    to — so a zero-volume bar adjacent to a downtime block joins that block into
    one run instead of adding a second charge. Counting bars here instead of
    runs would overstate the loss by 119 per adjacency.

    This is the function that answers the two quantities
    ``docs/ORIGIN_WINDOW_BUDGET.md`` lists under "Not yet measured".
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    lo = int(start.timestamp() * 1000)
    hi = int(end.timestamp() * 1000)
    span = frame.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))

    calendar_hours = (hi - lo) // HOUR_MS
    usable = int(span.select(pl.col("usable").sum()).item())
    counts = span.select(
        pl.col("zero_volume").sum().alias("zv"),
        pl.col("flat_bar").sum().alias("fb"),
        pl.col("zero_trades").sum().alias("zt"),
    ).row(0)

    # Walk the calendar, not the rows: a missing bar has no row to inspect, and
    # a run mixing missing with unusable positions must count once.
    usable_ts = set(span.filter(pl.col("usable")).get_column("ts_ms").to_list())
    break_runs = 0
    in_run = False
    for t in range(lo, hi, HOUR_MS):
        if t in usable_ts:
            in_run = False
        else:
            if not in_run:
                break_runs += 1
            in_run = True

    return BreakSummary(
        calendar_hours=calendar_hours,
        bars_present=span.height,
        bars_usable=usable,
        missing_bars=calendar_hours - span.height,
        zero_volume_bars=int(counts[0]),
        flat_bars=int(counts[1]),
        zero_trade_bars=int(counts[2]),
        excluded_positions=calendar_hours - usable,
        break_runs=break_runs,
    )


In [ ]:
# ═══ windows.py ════════
"""Window enumeration, validated by timestamp rather than by position.

Importers: ``itransformer_btc.budget`` and ``tests/test_data_plane.py``.
Reads no file directly — it consumes a frame produced by
:mod:`itransformer_btc.segments` and writes nothing.

Root §4.3 names this the highest-probability silent bug in the pipeline: after
any row drop, positional sliding closes gaps invisibly, and a window that spans
a two-day outage looks identical to one that does not. The rule is therefore

    window [s, s+L+H) is valid  <=>  t[s+L+H-1] - t[s] == (L+H-1) hours

and it is checked on every emitted window, not sampled. The check is cheap; the
failure it prevents is a leak no metric would reveal, because a model trained
across a gap looks *better*, not worse.

**The purge is structural here, not a separate step.** A window occupies
``[s, s+L+H)`` and its target is the final ``H`` bars. Enumerating only windows
that lie wholly inside a span therefore guarantees the last target ends exactly
at the span boundary — which is what root §8.2 asks for at *both* boundaries,
train→validation and train→test (`D24`). There is no separate purge to forget.
"""

from __future__ import annotations

from datetime import datetime

import polars as pl



def enumerate_windows(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> list[int]:
    """Every valid window start inside ``[start, end)``, as epoch ms.

    Windows are built *inside* segments and never across them, so no window can
    span a break. Enumerating inside ``[start, end)`` also applies the purge:
    the last window's target ends at ``end``, never past it.

    Args:
        frame: Bars carrying ``ts_ms``; ``usable`` is derived if absent.
        start: Inclusive lower bound of the span.
        end: Exclusive upper bound of the span.
        seq_len: Lookback ``L``.
        pred_len: Horizon ``H``.

    Returns:
        Window-start timestamps in ascending order.

    Raises:
        ValueError: If any emitted window fails the timestamp identity. That is
            an assertion about the segment builder, not about the data, so a
            failure means the pipeline is broken rather than the market.
    """
    span = seq_len + pred_len
    segments = build_segments(frame, start, end)
    if not segments:
        return []

    rows = (frame if "usable" in frame.columns else usable_mask(frame)).filter(
        pl.col("usable")
    )
    if start is not None:
        rows = rows.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        rows = rows.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    ts = rows.get_column("ts_ms").to_list()

    starts: list[int] = []
    for seg in segments:
        for s in range(seg.start_row, seg.end_row - span + 1):
            last = s + span - 1
            if ts[last] - ts[s] != (span - 1) * HOUR_MS:
                raise ValueError(
                    f"window at ts={ts[s]} spans a break: "
                    f"t[{last}] - t[{s}] = {ts[last] - ts[s]} ms, expected "
                    f"{(span - 1) * HOUR_MS} ms. The segment builder is wrong; "
                    f"do not relax this check."
                )
            starts.append(ts[s])
    return starts


def count_windows(
    segments: list[Segment],
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> int:
    """Total window starts across segments — the measured truth.

    Uses ``max(0, n - span + 1)`` per segment, so a segment shorter than one
    window contributes nothing rather than a negative count. Root §4.3's closed
    form ``(bars - 119) - [119 x breaks + missing]`` is algebraically identical
    to this **only while every segment clears the span**; where one does not,
    this is right and the closed form is not.
    """
    span = seq_len + pred_len
    return sum(seg.window_starts(span) for seg in segments)


In [ ]:
# ═══ budget.py ════════
"""Per-origin window accounting — the assertion target of root §11.

Importers: ``tests/test_data_plane.py``, and the Stage 2 launcher in
``notebooks/``. Reads ``data/raw/BTCUSDT_1h.parquet`` only through
:func:`itransformer_btc.segments.load_bars`; writes nothing.

``docs/ORIGIN_WINDOW_BUDGET.md`` was committed *before* any run so the pipeline
has something to be checked **against** rather than something to be tuned
**to**. This module measures the same quantities from the artifact and compares.

A divergence is a finding, not a nuisance. The committed table is derived from
the 27 downtime blocks alone, while the segment law (root §4.3) also breaks at
zero-volume and ``high == low`` bars, whose count has never been measured. If
those bars fall inside training spans, measured windows are **lower** than the
table and the table needs regenerating.

Root §11 requires an **exact equality per origin**, never a comparison against
the pooled 4.9% figure — asserted pooled, it fires spuriously at fourteen of
fifteen origins, gets loosened until it passes, and then can no longer
distinguish positional drift from ordinary between-origin variation.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Final

import polars as pl


#: Origin label to ``(break_runs, excluded_positions, windows_kept)`` for the
#: 21-month training sub-block. **Measured from the artifact on 2026-08-06**,
#: superseding the derived table `D26` shipped with (`D51`). It is pinned here
#: rather than recomputed inside the test so the test can catch *drift*: with
#: both sides computed the same way, a regression would agree with itself and
#: pass. Regenerate with :func:`format_markdown` and update both this dict and
#: ``docs/ORIGIN_WINDOW_BUDGET.md`` together, never one alone.
#:
#: The derived table it replaces read, for the first four origins,
#: ``(10, 86, 13_917) (12, 62, 13_727) (11, 47, 13_861) (12, 40, 13_797)``.
#: It diverged at twelve of fifteen origins for two reasons, both structural:
#: it never counted the three unusable bars, and its closed form charges a
#: short segment a negative window count. See `D51`.
COMMITTED_TRAIN_BUDGET: Final[dict[str, tuple[int, int, int]]] = {
    "2020-01": (11, 87, 13_934),
    "2020-06": (13, 63, 13_701),
    "2020-11": (12, 48, 13_741),
    "2021-04": (13, 41, 13_716),
    "2021-09": (14, 30, 13_560),
    "2022-02": (14, 32, 13_558),
    "2022-07": (9, 20, 14_165),
    "2022-12": (8, 19, 14_285),
    "2023-05": (2, 6, 15_021),
    "2023-10": (1, 2, 15_072),
    "2024-03": (1, 2, 15_120),
    "2024-08": (1, 2, 15_096),
    "2025-01": (1, 2, 15_096),
    "2025-06": (0, 0, 15_217),
    "2025-11": (0, 0, 15_217),
}


@dataclass(frozen=True, slots=True)
class OriginBudget:
    """Measured window accounting for one origin's training sub-block."""

    origin: Origin
    summary: BreakSummary
    windows_measured: int
    windows_closed_form: int
    test_block_starts: tuple[int, ...]

    @property
    def label(self) -> str:
        return self.origin.label

    @property
    def loss_pct(self) -> float:
        """Window starts lost to breaks, as a percentage of a gap-free span."""
        ceiling = self.summary.calendar_hours - STARTS_LOST_PER_BREAK
        return 100.0 * (1.0 - self.windows_measured / ceiling) if ceiling else 0.0

    @property
    def closed_form_agrees(self) -> bool:
        """Whether root §4.3's arithmetic matches the segment-wise truth.

        Disagreement means some segment is shorter than one window, so the
        closed form has gone negative somewhere and been silently absorbed.
        """
        return self.windows_measured == self.windows_closed_form


def surviving_block_starts(frame: pl.DataFrame, origin: Origin, b: int) -> int:
    """Window starts surviving inside test block ``b`` — out of 720.

    **Test blocks do not use the training semantics, and the difference is 119
    windows per block.** A training window must lie wholly inside its span,
    because its target may not cross into validation (root §8.2). A *test*
    window may not: root §8.3 states explicitly that a window's 96-bar input
    reaching back across the boundary is past information legitimately available
    to a forecaster at that moment, and blocking it "would make the evaluation
    unrealistically pessimistic". So every one of the block's 720 hours is an
    admissible forecast origin; what disqualifies one is a break inside the 120
    bars it spans, wherever those bars fall.

    Counting test blocks the training way yields 601 out of 720 even for a
    perfectly clean block — a 16.5% phantom loss that would be read as outage
    damage and would enter §9.2's block-coverage covariate as pure noise.
    """
    lo, hi = origin.block(b)
    lo_ms = int(lo.timestamp() * 1000)
    hi_ms = int(hi.timestamp() * 1000)
    span_ms = (WINDOW_SPAN - 1) * HOUR_MS

    usable = set(
        frame.filter(pl.col("usable")).get_column("ts_ms").to_list()
    )
    survivors = 0
    for start in range(lo_ms, hi_ms, HOUR_MS):
        # The window is contiguous exactly when every hour it spans is usable.
        if all((start + k * HOUR_MS) in usable for k in range(WINDOW_SPAN)):
            survivors += 1
    return survivors


def origin_budget(frame: pl.DataFrame, origin: Origin) -> OriginBudget:
    """Measure one origin's training sub-block and its six test blocks.

    The sub-block is ``[train_start, train_sub_end)`` — 21 months, not 24
    (`D25`). Test-block figures are surviving window *starts* inside each block.
    """
    summary = break_summary(frame, origin.train_start, origin.train_sub_end)
    segments = build_segments(frame, origin.train_start, origin.train_sub_end)

    ceiling = summary.calendar_hours - STARTS_LOST_PER_BREAK
    blocks = [surviving_block_starts(frame, origin, b) for b in range(1, TEST_BLOCKS + 1)]

    return OriginBudget(
        origin=origin,
        summary=summary,
        windows_measured=count_windows(segments, SEQ_LEN, PRED_LEN),
        windows_closed_form=ceiling - summary.window_starts_lost,
        test_block_starts=tuple(blocks),
    )


def budget_table(frame: pl.DataFrame) -> list[OriginBudget]:
    """Measure every origin in the committed grid."""
    return [origin_budget(frame, origin) for origin in ORIGINS]


def format_markdown(budgets: list[OriginBudget]) -> str:
    """Render the measured table in the shape of ``ORIGIN_WINDOW_BUDGET.md``.

    Used to regenerate the document when measurement supersedes derivation —
    never to silently overwrite it. Root §12: a number that cannot be
    regenerated is a documented failure, not a footnote.
    """
    head = (
        "| # | Origin | Training sub-block | Breaks | Excluded | Windows kept "
        "| Loss | Test-block starts B1…B6 |\n"
        "|---:|---|---|---:|---:|---:|---:|---|\n"
    )
    rows = [
        f"| {b.origin.index:>2} | {b.origin.origin:%Y-%m-%d} "
        f"| {b.origin.train_start:%Y-%m-%d} → {b.origin.train_sub_end:%Y-%m-%d} "
        f"| {b.summary.break_runs} | {b.summary.excluded_positions} "
        f"| {b.windows_measured:,} | {b.loss_pct:.1f}% "
        f"| {' / '.join(str(n) for n in b.test_block_starts)} |"
        for b in budgets
    ]
    return head + "\n".join(rows) + "\n"


In [ ]:
# ═══ features.py ════════
"""The twelve variates, and the K ladder cut over them.

All twelve are **engineered**; none is a raw kline column. What is excluded is a
class, not a list: technical indicators, multi-bar rolling statistics, calendar
dummies, cross-asset, on-chain, macro. Root §5.3 carries the argument — K is
RQ1's independent variable, and anything outside families F1–F5 breaks the
taxonomy that makes K_eff interpretable.

**No variate uses a rolling window.** Every one is a pure per-bar function of
the current bar, except ``r``, which uses the current and previous close. That
is a structural safety property rather than a style choice: with no rolling
window anywhere, the ``center=True`` leak class is unrepresentable (root §5.3).

**The ladder is cumulative and its order is load-bearing.** Column order is
ladder order, so rung K is exactly the first K columns and ``r`` is channel 0 at
every rung. Root §6.2 requires the loss be MSE on the **target channel only** at
every rung; with ``r`` pinned at index 0 that is one constant, not a lookup.
"""

from __future__ import annotations

import math
from typing import Final

import polars as pl


#: Ladder order. Rung K is ``VARIATE_ORDER[:K]``. Root §5.2's unique consistent
#: cut (`D01` — the source specification's K=8 rung summed to nine and
#: double-assigned ``log_mean_trade_size``).
VARIATE_ORDER: Final[tuple[str, ...]] = (
    # F1 price trajectory — K=1 is `r` alone
    "r",
    "upper_shadow",
    "lower_shadow",
    # F3 intensity, first member — completes K=4
    "log_quote_volume",
    # K=8: intensity, order flow, intrabar location
    "log_trade_count",
    "taker_buy_ratio",
    "signed_flow",
    "vwap_location",
    # K=12: the F2 volatility estimators + the dependent intensity member
    "log_parkinson",
    "log_garman_klass",
    "log_rogers_satchell",
    "log_mean_trade_size",
)

TARGET: Final = "r"
TARGET_INDEX: Final = 0

#: Parkinson's normaliser, ``1 / (4 ln 2)``.
_PARKINSON_C: Final = 1.0 / (4.0 * math.log(2.0))
#: Garman–Klass's second-term coefficient, ``2 ln 2 - 1`` ≈ 0.386. Strictly
#: below 0.5, which is what keeps the estimator positive: ``|ln(C/O)| <=
#: ln(H/L)`` because C and O both lie in ``[L, H]``, so GK >= 0.114 (ln H/L)^2.
_GK_C: Final = 2.0 * math.log(2.0) - 1.0

#: Stabiliser for Rogers–Satchell only (`D52`).
#:
#: Root §5.1 claims all three F2 estimators are "strictly positive once H == L
#: bars are excluded". That holds for Parkinson (proportional to ``(ln H/L)^2``)
#: and for Garman–Klass (bounded below by ``0.114 (ln H/L)^2``), but **not** for
#: Rogers–Satchell, which is
#:
#:     ln(H/C) ln(H/O) + ln(L/C) ln(L/O)
#:
#: and vanishes on any bar with no shadows at all — H equal to one of O/C and L
#: equal to the other. Such a bar has H > L, carries real trade information, and
#: passes the segment law; it is a marubozu, not a degenerate bar. Measured: 33
#: of 75,091 usable bars, 0.044%.
#:
#: ``1e-9`` is chosen so ``log(kappa) = -20.7`` lands **inside the measured
#: support** of ``log RS`` — median -10.91, 0.1st percentile -17.57, minimum
#: -23.5 — in the low tail where a shadowless bar belongs. A hard floor far
#: below support (1e-12 gives -27.6, about -11 sigma) would instead create 33
#: spikes that distort the instance normalisation of every window containing
#: one, and would smuggle a categorical marubozu flag into a continuous
#: variate — the convenience-variate failure root §5.2 forbids. The shift it
#: applies to a typical bar is negligible: at the median RS of 1.8e-5, adding
#: 1e-9 moves ``log RS`` by 5e-5.
#:
#: Deliberately **not** applied to Parkinson or Garman–Klass: both are provably
#: positive, their measured minima are 1.16e-8 and 1.48e-8, and adding kappa
#: there would shift the smallest values by roughly 8% for no reason.
_RS_STABILISER: Final = 1e-9


def ladder_columns(k: int) -> list[str]:
    """The variate names at rung ``k``.

    Raises:
        ValueError: If ``k`` is not one of the pre-registered rungs. Rungs are
            fixed before any model runs (root §3); an ad-hoc K is a new
            experiment and must be declared as one.
    """
    if k not in (1, 4, 8, 12):
        raise ValueError(f"K must be a pre-registered rung 1/4/8/12, got {k}")
    return list(VARIATE_ORDER[:k])


def build_features(frame: pl.DataFrame) -> pl.DataFrame:
    """Compute all twelve variates, per segment, dropping what is undefined.

    ``r`` is computed **within** each segment. Computing it on a concatenated
    series would inject giant cross-gap returns into ``mu_g`` and ``sigma_g``
    before any window is excluded (root §4.3): the 2018-02-08 outage would book
    a 33-hour move as a one-hour return, and the scaler would be fitted on it.

    The first bar of each segment therefore yields a null ``r`` and is dropped.
    That shortens every segment by one bar, which the window enumerator picks up
    for free — the dropped bar leaves a two-hour jump at the segment head, and
    :func:`itransformer_btc.segments.build_segments` splits on any jump.

    Args:
        frame: Bars carrying ``ts_ms`` and the raw kline columns. ``usable`` is
            derived if absent.

    Returns:
        ``ts_ms``, ``usable`` (all True), and the twelve variates in ladder
        order as ``Float64``, with no nulls.

    Raises:
        ValueError: If any variate is null or non-finite. Both are impossible
            once the segment law has excluded zero-volume and ``H == L`` bars,
            so either means the exclusion did not run.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    rows = frame.filter(pl.col("usable")).sort("ts_ms")

    # Segment identity from the timestamp alone. Excluded bars are already gone,
    # so they show up here as jumps, exactly as downtime does.
    rows = rows.with_columns(
        (pl.col("ts_ms").diff().fill_null(HOUR_MS) != HOUR_MS).cum_sum().alias("_seg")
    )

    log_h_l = (pl.col("high") / pl.col("low")).log()
    log_c_o = (pl.col("close") / pl.col("open")).log()
    vwap = pl.col("quote_volume") / pl.col("volume")

    out = rows.with_columns(
        # -- F1 price trajectory, 3 dof -------------------------------------
        (pl.col("close").log() - pl.col("close").log().shift(1).over("_seg")).alias("r"),
        (pl.col("high") / pl.max_horizontal("open", "close")).log().alias("upper_shadow"),
        (pl.min_horizontal("open", "close") / pl.col("low")).log().alias("lower_shadow"),

        # -- F3 intensity, 2 dof — the third is the difference of the first two
        pl.col("quote_volume").log().alias("log_quote_volume"),
        pl.col("trades").log().alias("log_trade_count"),
        (pl.col("quote_volume") / pl.col("trades")).log().alias("log_mean_trade_size"),

        # -- F4 order flow, 1–2 dof -----------------------------------------
        # Base-denominated: the canonical buyer-initiated volume share (`D12`).
        # The quote-denominated variant is a robustness check, not the default.
        (pl.col("taker_buy_base") / pl.col("volume")).alias("taker_buy_ratio"),

        # -- F5 intrabar location, 1 dof ------------------------------------
        # A total function, not a partial one, because H == L bars are segment
        # breaks (`D14`). Without that exclusion this divides by zero.
        ((vwap - pl.col("close")) / (pl.col("high") - pl.col("low"))).alias("vwap_location"),

        # -- F2 volatility estimators, ~1 dof, redundant by construction -----
        # Per-bar with no trailing average (`D13`). Pre-smoothing over 24 bars
        # is strictly less informative: the model can compute that average
        # itself and cannot recover what smoothing destroyed (root §5.3).
        # Parkinson and Garman-Klass are provably positive once H > L, so their
        # logs are total. Rogers-Satchell is NOT — it vanishes on shadowless
        # bars — hence the stabiliser, and only there (`D52`).
        (_PARKINSON_C * log_h_l.pow(2)).log().alias("log_parkinson"),
        (0.5 * log_h_l.pow(2) - _GK_C * log_c_o.pow(2)).log().alias("log_garman_klass"),
        (
            (pl.col("high") / pl.col("close")).log() * (pl.col("high") / pl.col("open")).log()
            + (pl.col("low") / pl.col("close")).log() * (pl.col("low") / pl.col("open")).log()
            + _RS_STABILISER
        ).log().alias("log_rogers_satchell"),
    ).with_columns(
        # A deterministic product of two other K=8 members. Kept, with the
        # dependence disclosed (`D12`): it weakens the claim that K=8 is the
        # rung of maximum effective rank, and the measured participation ratio
        # settles that question rather than the argument doing so.
        (
            (2.0 * pl.col("taker_buy_ratio") - 1.0) * pl.col("log_quote_volume")
        ).alias("signed_flow"),
    )

    # The first bar of each segment has no predecessor inside its segment.
    out = out.filter(pl.col("r").is_not_null())

    out = out.select(["ts_ms", "usable", *VARIATE_ORDER]).with_columns(
        [pl.col(c).cast(pl.Float64) for c in VARIATE_ORDER]
    )

    offenders = {
        name: n
        for name in VARIATE_ORDER
        if (
            n := int(
                out.select(
                    (~pl.col(name).is_finite() | pl.col(name).is_null()).sum()
                ).item()
            )
        )
    }
    if offenders:
        raise ValueError(
            f"non-finite variate values: {offenders}. Every variate is total "
            f"once zero-volume and H == L bars are excluded by the segment law "
            f"(root §4.3 / `D14`), so this means the exclusion did not run."
        )
    return out


In [ ]:
# ═══ splits.py ════════
"""Per-origin splits, the scaler, and the tensors the training loop slices.

Three things live here because they are one decision: which windows exist, what
standardises them, and how they reach the device.

**Window semantics differ by split, and the difference is 119 windows (`D51`).**
A *training* or *validation* window must lie wholly inside its span — its H-step
target may not cross the boundary, which is the purge at both boundaries (root
§8.2 / `D24`). A *test* window may not: root §8.3 states that its 96-bar
lookback reaching back across the boundary is past information legitimately
available to a forecaster, and that blocking it would make the evaluation
unrealistically pessimistic. Every hour of a test block is an admissible
forecast origin.

**The scaler is fitted on the 21-month sub-block and nothing else**, at every
origin. Moving ``train_end`` is a leak, not a mismatch (root §8.2).
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from typing import Literal

import numpy as np
import polars as pl


Semantics = Literal["contained", "origin"]


def window_starts(
    ts: np.ndarray,
    start: datetime,
    end: datetime,
    semantics: Semantics,
    span: int = WINDOW_SPAN,
) -> np.ndarray:
    """Absolute row indices of valid window starts in ``[start, end)``.

    Args:
        ts: Epoch-ms timestamps of the full feature frame, ascending.
        start: Inclusive lower bound.
        end: Exclusive upper bound.
        semantics: ``"contained"`` requires the whole window inside the span —
            training and validation, where the target may not cross the
            boundary. ``"origin"`` requires only the *start* inside — test
            blocks, where the lookback may reach back across it.
        span: ``L + H``.

    Returns:
        Row indices, ascending; empty if the span admits no window.
    """
    lo = int(start.timestamp() * 1000)
    hi = int(end.timestamp() * 1000)

    first = np.arange(len(ts) - span + 1)
    if len(first) == 0:
        return np.empty(0, dtype=np.int64)

    # Contiguity: the window covers `span` consecutive hours with no break.
    contiguous = (ts[first + span - 1] - ts[first]) == (span - 1) * HOUR_MS

    if semantics == "contained":
        inside = (ts[first] >= lo) & (ts[first + span - 1] < hi)
    else:
        inside = (ts[first] >= lo) & (ts[first] < hi)

    return first[contiguous & inside]


@dataclass(frozen=True, slots=True)
class Scaler:
    """Per-channel standardiser fitted on the training sub-block only.

    Root §6.3: the outer affine scaler **cancels algebraically** under instance
    normalisation, because ``(z - m)/s`` recovers ``(x - mean_t)/std_t`` with
    ``mu_g`` and ``sigma_g`` dropping out. What it still controls is the
    *reporting scale* of every metric, and learning for the baselines that have
    no internal normalisation. StandardScaler is chosen for literature
    comparability, inertness under ``use_norm=True``, and cross-model
    consistency — not because it changes what the transformer learns.
    """

    mean: np.ndarray
    std: np.ndarray
    columns: tuple[str, ...]

    @classmethod
    def fit(cls, values: np.ndarray, columns: list[str]) -> "Scaler":
        std = values.std(axis=0, ddof=0)
        if not np.all(np.isfinite(std)) or np.any(std <= 0):
            raise ValueError(
                f"degenerate channel std in the training sub-block: "
                f"{dict(zip(columns, std))}"
            )
        return cls(values.mean(axis=0), std, tuple(columns))

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (values - self.mean) / self.std

    @property
    def target_mu_over_sigma(self) -> float:
        """``mu_g / sigma_g`` on the target channel — the Naive-RW offset.

        Root §7 / `D31`: a random walk in price implies ``y_raw = 0``, but the
        metrics live on standardised returns, so ``y_z = 0`` would silently mean
        ``r_hat = mu_g``, the training-window mean hourly return — a constant
        drift model wearing the EMH baseline's name. The baseline is mapped as
        ``y_z = -mu_g / sigma_g`` instead, and this value is logged per origin
        so the size of the tilt is auditable.
        """
        return float(self.mean[TARGET_INDEX] / self.std[TARGET_INDEX])


@dataclass(frozen=True, slots=True)
class SplitTensors:
    """One split's inputs, targets, and the timestamps they were issued at.

    ``y`` and ``y_all`` are the same block of future bars read at two widths, and
    both exist because the study now trains two kinds of model (`D56`). Root §6.2
    / `D39` fixes the iTransformer loss on the **target channel only** at every
    rung, so the ladder trains against ``y``. The channel-independent baselines —
    DLinear, PatchTST — carry their published all-channel objective, and that is
    the only thing making their ``K = 8`` label true: a channel-independent
    forecast for one channel depends on that channel's history alone, so
    supervision on all eight through shared weights is the sole route by which
    the other seven reach the target's forecast at all. Trained against ``y``
    they would be K=1 wearing a K=8 label — exactly the collapse `D40` exists to
    prevent.

    ``preds/*.parquet`` holds the target channel for **every** model regardless
    (root §10.4), so nothing downstream needs to know which width was trained on.
    """

    x: np.ndarray      # (n, L, K) float32, standardised
    y: np.ndarray      # (n, H)    float32, standardised target channel
    y_all: np.ndarray  # (n, H, K) float32, every channel's H-step target
    ts: np.ndarray     # (n,)      int64, window start — for traceability

    def __len__(self) -> int:
        return len(self.ts)


@dataclass(frozen=True, slots=True)
class OriginTensors:
    """Everything one training run consumes, already standardised."""

    origin: OriginLike
    k: int
    scaler: Scaler
    train: SplitTensors
    val: SplitTensors
    test_blocks: tuple[SplitTensors, ...]
    #: One-indexed block label per entry of ``test_blocks``. ``(1,…,6)`` for a
    #: normal origin, ``(4, 5, 6)`` for the falsification arm — which is why the
    #: label is stored rather than recovered from position.
    block_labels: tuple[int, ...]

    @property
    def naive_rw_z(self) -> float:
        """The Naive-RW prediction in scaler space (`D31`)."""
        return -self.scaler.target_mu_over_sigma


def _gather(
    values: np.ndarray, starts: np.ndarray, ts: np.ndarray, seq_len: int, pred_len: int
) -> SplitTensors:
    """Slice windows out of a standardised array by index arithmetic.

    No ``Dataset``, no ``DataLoader``, no per-item Python. Root §10.3: at ~280k
    parameters the run is dominated by data movement and interpreter overhead,
    which a per-item loader maximises — the naive path costs roughly 10x and
    puts the grid outside the weekly GPU quota outright.
    """
    if len(starts) == 0:
        return SplitTensors(
            x=np.empty((0, seq_len, values.shape[1]), np.float32),
            y=np.empty((0, pred_len), np.float32),
            y_all=np.empty((0, pred_len, values.shape[1]), np.float32),
            ts=np.empty(0, np.int64),
        )
    rows = starts[:, None] + np.arange(seq_len)[None, :]
    tgt = starts[:, None] + seq_len + np.arange(pred_len)[None, :]
    targets = values[tgt].astype(np.float32, copy=False)
    return SplitTensors(
        x=values[rows].astype(np.float32, copy=False),
        # Copied out rather than left as a strided view of `targets`: `y` is what
        # the whole pipeline reads, and a non-contiguous array of it would make
        # every downstream `from_numpy` and `reshape` behave differently for a
        # reason nobody would think to look for.
        y=np.ascontiguousarray(targets[:, :, TARGET_INDEX]),
        y_all=targets,
        ts=ts[starts],
    )


def build_origin_tensors(
    features: pl.DataFrame,
    origin: OriginLike,
    k: int,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> OriginTensors:
    """Build every split for one (origin, K) cell.

    The scaler is fitted on the **rows** of the 21-month sub-block, before any
    window is cut, and then applied to every split. Fitting it on validation or
    test rows is the leak root §11 calls fatal.

    Raises:
        ValueError: If the training split is empty, or if the last training
            window's target reaches at or past ``val_start`` — the purge
            assertion (`D24`), checked here rather than trusted.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()
    span = seq_len + pred_len

    train_idx = window_starts(ts, origin.train_start, origin.train_sub_end,
                              "contained", span)
    val_idx = window_starts(ts, origin.val_start, origin.val_end, "contained", span)
    if len(train_idx) == 0:
        raise ValueError(f"origin {origin.label}: empty training split")

    last_train_target = ts[train_idx[-1] + span - 1]
    if last_train_target >= int(origin.val_start.timestamp() * 1000):
        raise ValueError(
            f"origin {origin.label}: a training target reaches into validation "
            f"({last_train_target}); the purge did not hold"
        )

    scaler = Scaler.fit(values[train_idx[0] : train_idx[-1] + span], columns)
    scaled = scaler.transform(values)

    blocks = origin.blocks()
    return OriginTensors(
        origin=origin,
        k=k,
        scaler=scaler,
        train=_gather(scaled, train_idx, ts, seq_len, pred_len),
        val=_gather(scaled, val_idx, ts, seq_len, pred_len),
        test_blocks=tuple(
            _gather(scaled, window_starts(ts, lo, hi, "origin", span),
                    ts, seq_len, pred_len)
            for _, lo, hi in blocks
        ),
        block_labels=tuple(label for label, _, _ in blocks),
    )


In [ ]:
# ═══ model.py ════════
"""Encoder-only iTransformer: each variate is a token, attention runs across them.

Root §6.1. The inversion is the whole point — attention operates over the
**variate** axis, not the time axis, so the sequence the attention sees is
``N <= 12`` long rather than ``L = 96``. That is why ``d_model = 128`` and not
the reference implementation's 512: at ~14,000 training samples per origin, 512
over-parameterises badly, and the usual justification for a wide model — a long
attention sequence — does not apply here (`D25`).

**No causal mask.** Masking applies to the time axis; this attention runs over
the variate axis, where all tokens are contemporaneous. Causality is enforced
upstream, in the features and the windowing.

**K=1 is a designed control, not a degenerate bug** (`D50`). At ``N = 1``,
softmax over a single token returns weight 1, so attention reduces to
``W_O W_V x + x`` — the value and output projections and the residual **remain**;
it is not a bare identity. Parameter count is identical at every rung, because K
changes the token count and not a single weight shape. Say so in the
methodology: unexplained, an examiner reads it as an implementation error.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch import Tensor, nn


@dataclass(frozen=True, slots=True)
class ITransformerConfig:
    """Hyperparameters, adopted unchanged from Liu et al. (2024) bar ``d_model``.

    **Nothing here is tuned, deliberately** (`D38`). Holding capacity fixed is
    what makes the rungs comparable; per-rung tuning would confound the ladder
    with model selection. Root §11's checklist item on validation-based
    hyperparameter selection therefore applies to ARIMA order and ridge alpha
    only — those are the two models where selection actually happens.
    """

    seq_len: int = 96
    pred_len: int = 24
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    use_norm: bool = True
    #: Force attention weights uniform — the third main-grid arm (`D50`).
    #: K=1 vs K=8 differs in *information* and in *whether attention is active*
    #: at the same time, so a decaying A(b) is equally consistent with a
    #: capacity story as with the information story RQ2 claims. This separates
    #: them, at runs Figure 5 needs anyway.
    uniform_attention: bool = False

    # -- the Architecture protocol (`D56`) ----------------------------------
    #
    # Methods, not fields. ``write_artifacts`` records ``asdict(cfg)``, so
    # anything added here as a *field* would enter every iTransformer
    # ``meta/*.json`` and change bytes the 534-run grid has already produced.

    def build(self) -> "ITransformer":
        """A fresh module for this configuration."""
        return ITransformer(self)

    def loss_target(self) -> str:
        """``"target"`` — MSE on the target channel only, at every rung (`D39`).

        A constant rather than a field, deliberately. Standard iTransformer
        implementations compute the loss over all N channels, which would make
        K=12 a 12-task problem and K=1 a 1-task problem: auxiliary supervision
        varying with the study's own independent variable, and K=1 no longer the
        stated control but a different learning problem. Root §11 carries this as
        a verifiable assertion, and a field would be one edit away from failing
        it silently.
        """
        return "target"

    def fit(
        self,
        tensors: "OriginTensors",
        spec: "RunSpec",
        *,
        device: "torch.device | None" = None,
    ) -> tuple["ITransformer", "ITransformerConfig", "TrainOutcome"]:
        """Train one cell and hand back this config **unchanged**.

        Nothing is selected here: every hyperparameter is fixed a priori and
        identical at every rung (`D38`), which is what makes the rungs
        comparable. Ridge is the contrast — its alpha is chosen on the validation
        sub-block, so its ``fit`` returns a different config than it received.

        The import is deferred because ``train`` owns the protocol this
        satisfies, and importing it at module scope would be a cycle. In the
        flattened notebook there are no modules at all and ``train_one`` is
        simply a name a later cell defines, bound by the time this is called.
        """

        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


class InvertedEmbedding(nn.Module):
    """Embed each variate's entire lookback: ``Linear(L -> d_model)``.

    With ``d_model = 128 > L = 96`` this projection is generically injective, so
    the whole lookback survives it. Root §5.3 leans on that: the reason for
    excluding moving averages is not that they are unrecoverable — a linear
    function of the lookback is recoverable in principle — but that adding one
    raises nominal K without raising information, which is precisely the
    phenomenon RQ1 exists to measure.
    """

    def __init__(self, seq_len: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.projection = nn.Linear(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, N, d_model)``."""
        return self.dropout(self.projection(x.permute(0, 2, 1)))


class VariateAttention(nn.Module):
    """Multi-head attention over the variate axis, optionally forced uniform."""

    def __init__(self, d_model: int, n_heads: int, dropout: float, uniform: bool) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError(f"d_model {d_model} not divisible by n_heads {n_heads}")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.uniform = uniform
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        b, n, d = x.shape
        shape = (b, n, self.n_heads, self.head_dim)
        v = self.v(x).view(shape).transpose(1, 2)

        if self.uniform:
            # Every variate attends equally to every variate. W_V, W_O and the
            # parameter count stay intact, so the arm isolates *what attention
            # selects* rather than how much capacity the model has.
            context = v.mean(dim=2, keepdim=True).expand(-1, -1, n, -1)
        else:
            q = self.q(x).view(shape).transpose(1, 2)
            k = self.k(x).view(shape).transpose(1, 2)
            scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
            context = self.dropout(torch.softmax(scores, dim=-1)) @ v

        return self.out(context.transpose(1, 2).reshape(b, n, d))


class EncoderLayer(nn.Module):
    """Attention over variates, then a position-wise FFN. Post-norm, as in the paper."""

    def __init__(self, cfg: ITransformerConfig) -> None:
        super().__init__()
        self.attention = VariateAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, cfg.uniform_attention
        )
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = self.norm1(x + self.dropout(self.attention(x)))
        return self.norm2(x + self.dropout(self.ffn(x)))


class ITransformer(nn.Module):
    """``(B, L, N) -> (B, H)`` on the target channel.

    The output is the target channel alone even though the projection produces
    all N, because the **loss is single-channel** (`D39`). Standard
    implementations compute it over all N, which would make K=12 a 12-task
    problem and K=1 a 1-task problem — auxiliary supervision varying with the
    study's own independent variable, and K=1 no longer the stated control but a
    different learning problem. The reference implementation defaults to the
    option that breaks the design, so root §11 carries this as an assertion.
    """

    def __init__(self, cfg: ITransformerConfig, target_index: int = 0) -> None:
        super().__init__()
        self.cfg = cfg
        self.target_index = target_index
        self.embedding = InvertedEmbedding(cfg.seq_len, cfg.d_model, cfg.dropout)
        self.layers = nn.ModuleList(EncoderLayer(cfg) for _ in range(cfg.e_layers))
        self.projection = nn.Linear(cfg.d_model, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        """Args: ``x`` of shape ``(B, L, N)``. Returns ``(B, H)``."""
        mean = std = None
        if self.cfg.use_norm:
            # Per-channel instance normalisation over time. Root §6.3: this is
            # what makes the outer StandardScaler cancel algebraically — and it
            # is itself a nonlinearity, which is why the F2 estimators
            # contribute *shape* and not *level*, the confound `D04` requires be
            # disclosed in Limitations whatever RQ1 returns.
            mean = x.mean(dim=1, keepdim=True)
            x = x - mean
            std = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5)
            x = x / std

        h = self.embedding(x)
        for layer in self.layers:
            h = layer(h)
        out = self.projection(h).permute(0, 2, 1)  # (B, H, N)

        if self.cfg.use_norm:
            out = out * std[:, 0, :].unsqueeze(1) + mean[:, 0, :].unsqueeze(1)

        return out[:, :, self.target_index]

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, H)`` — here, identical to ``forward``.

        The method exists because ``preds/*.parquet`` holds the target channel
        for **every** model in the study, and a channel-independent baseline's
        ``forward`` returns all N (`D56`). Declaring the projection on the model
        that wrote the file beats sniffing the rank of an output tensor: the
        prediction file's meaning then rests on something a model said, not on a
        shape a reader has to reverse-engineer.
        """
        return self(x)

    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [ ]:
# ═══ train.py ════════
"""Training loop, run identity, and the two files every run must leave behind.

Root §10.3's regime is the load-bearing part: **load the whole split to the
device once, then batch by index-slicing that tensor.** No ``Dataset``, no
``DataLoader``, no workers. At ~280k parameters the compute is trivial and the
run is dominated entirely by data movement and Python overhead, which a per-item
loader maximises — the naive path costs roughly 10x and puts the 837-run grid
outside the 30 h weekly quota outright.

Root §10.4: **persist raw predictions, always.** They are required for the
Diebold-Mariano test, the per-regime analysis and the economic evaluation.
Re-running the grid because only metrics were saved is an expensive, avoidable
mistake.
"""

from __future__ import annotations

import hashlib
import json
import os
import random
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Protocol

import numpy as np
import polars as pl
import torch
from torch import Tensor, nn


ARTIFACTS: Path = Path("artifacts")

DEFAULT_PARQUET: Path = Path("data/raw/BTCUSDT_1h.parquet")

#: Environment variable naming the input artifact actually consumed.
#:
#: Root §12 forbids comparing numbers produced under different input-artifact
#: hashes, so every ``meta/*.json`` must name the vintage it read. The
#: repository-relative default is right locally and **wrong everywhere the grid
#: actually runs**: on Kaggle the artifact arrives as an attached Dataset under
#: ``/kaggle/input/<slug>/``, and root §10.5 forbids hard-coding that slug. With
#: the path unresolvable the digest logged as ``"unknown"`` on every Kaggle run,
#: which is §12 unenforceable at precisely the place the grid executes. The
#: launcher and the worker CLI both set this from the path they were handed.
INPUT_PARQUET_ENV: str = "ITBTC_PARQUET"

#: Digest supplied by a launcher that has no package files to hash.
#:
#: :func:`code_sha256` normally hashes ``*.py`` beside this module, which needs
#: a ``__file__`` — and a notebook that carries the package as **plain
#: definition cells** rather than materialised files has none. The generator
#: computes the identical digest from ``src/itransformer_btc/`` and sets this,
#: so the number in ``meta/*.json`` still names the code that ran (root §12,
#: `D54b`) and still matches the digest a local checkout of the same source
#: produces. Left ``None`` in every file-based context, where the real hash is
#: strictly better because nothing has to be told to keep it honest.
CODE_SHA256_OVERRIDE: str | None = None


def set_seed(seed: int) -> None:
    """Seed every source of nondeterminism root §16 names.

    ``cudnn.deterministic`` costs throughput and is set anyway: a run that
    cannot be reproduced cannot enter the manuscript (root §12).
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    """Prefer CUDA; fall back to CPU."""
    return torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")


def supports_native_bf16(device: torch.device) -> bool:
    """True only on sm_80+.

    Never gate on ``torch.cuda.is_bf16_supported()``: it defaults to
    ``including_emulation=True`` and returns **True** on a T4 (sm_75), selecting
    an emulated bf16 path slower than fp32 (root §10.3).
    """
    if device.type != "cuda":
        return False
    return torch.cuda.get_device_capability(device.index or 0)[0] >= 8


@dataclass(frozen=True, slots=True)
class RunSpec:
    """Deterministic, human-readable run identity (root §10.4).

    Changing any component deliberately **orphans** prior outputs rather than
    silently reusing a mismatched result.
    """

    model: str
    origin_index: int
    k: int
    pred_len: int
    seed: int

    @property
    def run_id(self) -> str:
        return (
            f"{self.model}_o{self.origin_index:02d}_K{self.k:02d}"
            f"_H{self.pred_len:03d}_s{self.seed}"
        )


@dataclass(frozen=True, slots=True)
class TrainOutcome:
    """What one completed run produced, beyond its two artifacts."""

    run_id: str
    epochs_run: int
    best_val_mse: float
    train_loss: float
    wall_time_s: float
    n_parameters: int
    device: str


class Architecture(Protocol):
    """What the trainer, the runner and the artifact writer need of a config.

    :func:`write_artifacts` is the **only** definition of the ``meta/*.json``
    schema — root §12's traceability contract expressed as code — and it was
    bound to ``ITransformer``/``ITransformerConfig`` until the §7 baselines
    arrived (`D56`). A second writer in ``baselines.py`` would have made two
    definitions of one contract, which is the drift surface `D54d` exists to
    prevent, so the writer was widened to this protocol instead of copied.

    **Everything here is a method, never a field, and that is load-bearing.**
    ``write_artifacts`` records ``asdict(cfg)``, so a field added merely to steer
    dispatch would appear in every iTransformer ``meta/*.json`` and change bytes
    the study has already produced. Methods are invisible to
    :func:`dataclasses.asdict`; fields are not.
    """

    pred_len: int

    def build(self) -> nn.Module:
        """A fresh, untrained module for this configuration."""

    def loss_target(self) -> str:
        """``"target"`` or ``"all"`` — which target tensor the loss reads.

        See :class:`itransformer_btc.splits.SplitTensors`: the ladder is
        single-channel by `D39`, the channel-independent baselines are
        all-channel by their own published objective, and that difference is what
        makes their K label mean anything.
        """

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple[nn.Module, "Architecture", TrainOutcome]:
        """Fit one cell: the model, the **resolved** config, and the outcome.

        The config comes back because selection happens for one model and not the
        others. Every iTransformer hyperparameter is fixed a priori and identical
        at every rung (`D38`), so its ``fit`` returns what it was given; ridge's
        alpha is chosen on the validation sub-block, and with ARIMA outside the
        minimal set that is the **only** hyperparameter selected anywhere in this
        study (root §11). A chosen value that never reached ``meta['config']``
        would be a number the manuscript could not regenerate.
        """


class Forecaster(Protocol):
    """What the artifact writer needs of a fitted model."""

    cfg: Architecture

    def eval(self) -> "Forecaster":
        """Inference mode — dropout off."""

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)`` on the target channel: what ``preds/`` holds."""


def _to_device(
    split: SplitTensors, device: torch.device, *, target: str = "target"
) -> tuple[Tensor, Tensor]:
    """Move one split's inputs and its loss target to the device.

    ``target`` selects the width, not the content: ``"target"`` is the ``r``
    channel the ladder is scored on, ``"all"`` is every channel, which is what a
    channel-independent baseline's published objective supervises.
    """
    y = split.y if target == "target" else split.y_all
    return (
        torch.from_numpy(split.x).to(device, non_blocking=True),
        torch.from_numpy(y).to(device, non_blocking=True),
    )


@torch.no_grad()
def _mean_loss(model: nn.Module, x: Tensor, y: Tensor, batch: int = 512) -> float:
    """Mean MSE over a split, batched to bound peak memory rather than for speed.

    The divisor is every element of one sample's target, so this is the mean over
    ``(H,)`` for a single-channel target and over ``(H, N)`` for an all-channel
    one — the same quantity ``mse_loss`` returns, computed in pieces.
    """
    if len(x) == 0:
        return float("nan")
    model.eval()
    total = 0.0
    for i in range(0, len(x), batch):
        total += nn.functional.mse_loss(
            model(x[i : i + batch]), y[i : i + batch], reduction="sum"
        ).item()
    return total / (len(x) * int(np.prod(y.shape[1:])))


@torch.no_grad()
def predict(model: Forecaster, x: Tensor, batch: int = 512) -> np.ndarray:
    """The target channel's H-step forecasts for every window, batched.

    Routed through ``forecast_target`` rather than ``__call__`` because a
    channel-independent baseline trained on its published all-channel objective
    returns ``(B, H, N)`` from ``forward`` while ``preds/`` holds one channel
    (root §10.4). Sniffing the rank of the output instead would leave the
    prediction file's meaning resting on a shape no model ever declared.
    """
    model.eval()
    if len(x) == 0:
        return np.empty((0, model.cfg.pred_len), np.float32)
    return np.concatenate(
        [
            model.forecast_target(x[i : i + batch]).cpu().numpy()
            for i in range(0, len(x), batch)
        ]
    )


def train_one(
    tensors: OriginTensors,
    spec: RunSpec,
    cfg: Architecture,
    *,
    device: torch.device | None = None,
    batch_size: int = 32,
    max_epochs: int = 30,
    patience: int = 5,
    lr: float = 1e-4,
    lr_halve_every: int = 4,
) -> tuple[nn.Module, TrainOutcome]:
    """Train one (origin, K, seed) cell and return the best-validation model.

    The learning rate halves every **four** epochs, not every epoch (`D47`):
    per-epoch halving reaches ~4e-7 by epoch 9, which makes the 30-epoch budget
    and the patience-5 early stop decorative — the model stops moving long
    before either can bind.

    ``epochs_run`` and the final training loss come back because root §6.2
    requires them logged per rung: they are how a reader tells a flat 8->12 rung
    from an under-trained one.

    **One trainer serves every gradient model in the study**, iTransformer and
    the §7 baselines alike (`D56`). The schedule, the patience and the early-stop
    rule are root §6.2's, and duplicating them per model would be a second
    definition of the training protocol — the same drift `D54d` names for the
    artifact schema. The config supplies only what genuinely differs: the module
    to build, and the width of the target its loss reads.
    """
    device = device or pick_device()
    set_seed(spec.seed)

    model = cfg.build().to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    schedule = torch.optim.lr_scheduler.StepLR(
        optimiser, step_size=lr_halve_every, gamma=0.5
    )

    # The whole split, resident. Index-slicing it is the entire batching
    # strategy; shuffling permutes an index tensor on device, never the data.
    loss_target = cfg.loss_target()
    x_tr, y_tr = _to_device(tensors.train, device, target=loss_target)
    x_va, y_va = _to_device(tensors.val, device, target=loss_target)

    best_val = float("inf")
    best_state: dict[str, Tensor] | None = None
    epochs_run = 0
    train_loss = float("nan")
    stale = 0
    started = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(x_tr), device=device)
        running = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i : i + batch_size]
            optimiser.zero_grad(set_to_none=True)
            # For the ladder this is MSE on the target channel only, at every
            # rung (`D39`): `ITransformerConfig.loss_target` is the constant
            # "target" and `ITransformer.forward` returns that channel alone, so
            # it cannot drift to the all-channel loss the reference
            # implementation defaults to — which would make K=12 a 12-task
            # problem and K=1 a 1-task one, varying supervision with the study's
            # own independent variable. A channel-independent baseline says
            # "all" and gets the all-channel objective it is published with.
            loss = nn.functional.mse_loss(model(x_tr[idx]), y_tr[idx])
            loss.backward()
            optimiser.step()
            running += loss.item() * len(idx)
        schedule.step()

        epochs_run = epoch
        train_loss = running / len(x_tr)
        val = _mean_loss(model, x_va, y_va)

        if val < best_val - 1e-9:
            best_val, stale = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, TrainOutcome(
        run_id=spec.run_id,
        epochs_run=epochs_run,
        best_val_mse=best_val,
        train_loss=train_loss,
        wall_time_s=time.perf_counter() - started,
        n_parameters=model.n_parameters(),
        device=str(device),
    )


def scale_invariance_check(
    model: Forecaster, x: Tensor, y: Tensor, c: float = 100.0
) -> tuple[float, float]:
    """Root §6.3's corrected ``use_norm`` invariant (`D03`).

    The source specification said to multiply the input by 100 and assert
    identical losses. That **cannot pass**: the target is a channel of the same
    array, so it scales too and the loss scales by ``c^2``. The invariant that
    does hold is ``MSE(c x) / c^2 == MSE(x)``.

    Returns:
        ``(MSE(x), MSE(c x) / c^2)`` — equal to floating-point tolerance while
        ``use_norm`` is active, and visibly unequal the moment it is not.
    """
    return _mean_loss(model, x, y), _mean_loss(model, x * c, y * c) / (c * c)


def _git_sha() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return "unknown"


def code_sha256() -> str:
    """Digest of this package's own source — the git sha's stand-in off-repo.

    Root §12 asks a run to name the code that produced it, and names the git sha
    as the way. **There is no git repository on Kaggle**, so the sha logs as
    ``"unknown"`` there and the traceability contract loses its code half exactly
    where the grid runs. Hashing the package source answers the same question and
    answers it better: it identifies the code that ran, not the commit someone
    happened to be standing on with a dirty tree.

    Line endings are normalised, so a CRLF checkout on Windows and the LF copy a
    notebook materialises give the **same** digest for identical logic. Without
    that, every Kaggle run would appear to be a different code vintage from the
    local run of the same commit — a false positive on the one check §12 exists
    to make possible.

    ``CODE_SHA256_OVERRIDE`` short-circuits this where there are no files to
    hash — a notebook carrying the package as plain definition cells. The
    ``__file__`` lookup below sits *after* that check on purpose: in such a
    launcher there is no module file at all, so reaching it would raise rather
    than return the digest the traceability contract asks for.
    """
    if CODE_SHA256_OVERRIDE is not None:
        return CODE_SHA256_OVERRIDE
    root = Path(__file__).resolve().parent
    digest = hashlib.sha256()
    for path in sorted(root.glob("*.py")):
        digest.update(path.name.encode("utf-8"))
        digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
    return digest.hexdigest()


def resolve_input_parquet(parquet: Path | str | None = None) -> Path:
    """The input artifact this process consumed: argument, then env, then default."""
    if parquet is not None:
        return Path(parquet)
    from_env = os.environ.get(INPUT_PARQUET_ENV)
    return Path(from_env) if from_env else DEFAULT_PARQUET


def _input_sha256(parquet: Path | str | None = None) -> tuple[str, str]:
    """``(digest, provenance)`` for the parquet actually read.

    The Stage 1 report sitting beside the artifact is preferred, because that
    digest was written by the ingest script over the bytes it emitted and is the
    figure root §4.1 pins. Where the report did not travel — an attached Kaggle
    Dataset holding the parquet alone — the file is hashed directly, which yields
    the same number by construction.

    Returns ``("unknown", "unresolved")`` rather than raising: a run that cannot
    name its input is a documented failure under §12, and failing the run instead
    would lose 90 s of GPU time to a bookkeeping problem.
    """
    path = resolve_input_parquet(parquet)
    report = path.with_name(f"{path.stem}_report.json")
    try:
        return json.loads(report.read_text())["artifact_sha256"][path.name], "report"
    except Exception:
        pass
    try:
        return hashlib.sha256(path.read_bytes()).hexdigest(), "file-digest"
    except Exception:
        return "unknown", "unresolved"


def write_artifacts(
    model: Forecaster,
    tensors: OriginTensors,
    spec: RunSpec,
    cfg: Architecture,
    outcome: TrainOutcome,
    device: torch.device,
    root: Path = ARTIFACTS,
) -> tuple[Path, Path]:
    """Write ``preds/{run_id}.parquet`` and ``meta/{run_id}.json``.

    A run is complete **only when both files exist and ``status == "complete"``**
    (root §10.5). Anything else is re-run from scratch; intra-run checkpointing
    is deliberately omitted, since at ~30 s per run measured (`D57`) it costs far
    more complexity than it saves.

    **This function is the schema.** Root §12 admits no number into the
    manuscript that does not resolve to a prediction file, a config hash and a
    documented decision, and these two files are where all three live. Every
    model in the study — the ladder, ridge, DLinear, PatchTST — writes through
    here, typed against :class:`Forecaster` and :class:`Architecture` rather than
    against one model, so there is exactly one place the contract can be read and
    exactly one place it can be broken.
    """
    (root / "preds").mkdir(parents=True, exist_ok=True)
    (root / "meta").mkdir(parents=True, exist_ok=True)

    frames = []
    for b, split in zip(tensors.block_labels, tensors.test_blocks):
        # The label, not the position: the falsification arm's first tensor is
        # block 4, and re-indexing it to 1 would silently compare the fresh
        # model against the aged model's wrong blocks.
        if len(split) == 0:
            continue
        x, _ = _to_device(split, device)
        pred = predict(model, x)
        n, h = pred.shape
        frames.append(
            pl.DataFrame(
                {
                    "block": np.full(n * h, b, dtype=np.int8),
                    "step": np.tile(np.arange(1, h + 1, dtype=np.int16), n),
                    "timestamp": np.repeat(split.ts, h),
                    "y_true": split.y.reshape(-1),
                    "y_pred": pred.reshape(-1),
                }
            )
        )
    preds = (
        pl.concat(frames)
        if frames
        else pl.DataFrame(
            schema={
                "block": pl.Int8,
                "step": pl.Int16,
                "timestamp": pl.Int64,
                "y_true": pl.Float32,
                "y_pred": pl.Float32,
            }
        )
    )

    preds_path = root / "preds" / f"{spec.run_id}.parquet"
    meta_path = root / "meta" / f"{spec.run_id}.json"
    preds.write_parquet(preds_path)

    input_parquet = resolve_input_parquet()
    input_digest, input_provenance = _input_sha256(input_parquet)
    meta = {
        "run_id": spec.run_id,
        "spec": asdict(spec),
        "config": asdict(cfg),
        "origin": tensors.origin.label,
        "origin_index": tensors.origin.index,
        "block_labels": list(tensors.block_labels),
        "k": tensors.k,
        "variates": list(tensors.scaler.columns),
        "git_sha": _git_sha(),
        # Root §12's code half. `git_sha` is "unknown" off-repo, which is every
        # Kaggle session; `code_sha256` answers the same question there.
        "code_sha256": code_sha256(),
        "input_parquet": str(input_parquet),
        "input_sha256": input_digest,
        "input_sha256_source": input_provenance,
        "n_train": len(tensors.train),
        "n_val": len(tensors.val),
        "n_test_per_block": [len(s) for s in tensors.test_blocks],
        # Root §7 / `D31`: logged per origin so the drift tilt the Naive-RW
        # baseline carries in scaler space is auditable rather than assumed away.
        "mu_g": float(tensors.scaler.mean[0]),
        "sigma_g": float(tensors.scaler.std[0]),
        "mu_over_sigma": tensors.scaler.target_mu_over_sigma,
        "naive_rw_z": tensors.naive_rw_z,
        "epochs_run": outcome.epochs_run,
        "best_val_mse": outcome.best_val_mse,
        "train_loss": outcome.train_loss,
        "wall_time_s": outcome.wall_time_s,
        "n_parameters": outcome.n_parameters,
        "device": outcome.device,
        "status": "complete",
    }
    meta_path.write_text(json.dumps(meta, indent=2))
    return preds_path, meta_path


def is_complete(run_id: str, root: Path = ARTIFACTS) -> bool:
    """Root §10.5's idempotence rule, as one function."""
    preds = root / "preds" / f"{run_id}.parquet"
    meta = root / "meta" / f"{run_id}.json"
    if not (preds.exists() and meta.exists()):
        return False
    try:
        return json.loads(meta.read_text()).get("status") == "complete"
    except Exception:
        return False


In [ ]:
# ═══ keff.py ════════
"""Effective dimensionality — RQ1's independent variable, measured before training.

Root §5.4. These statistics run **before any model does**, and two of the study's
three claimed contributions rest on them: the separation of nominal K from
effective dimensionality, and the K-versus-K_eff horse race in §9.1.

Three constraints shape every function here, and each closes a leak the earlier
design left open:

* **`D44` — span.** Every reported K_eff declares its span, and the one that
  feeds RQ1's regression is computed **per origin on that origin's own 21-month
  training sub-block**. Nothing previously forbade computing it over 2018-2026,
  a span containing every origin's test blocks; the regressor would then be
  estimated on the same data as the outcome and RQ1's claim would be partly
  circular, while §11's fatal checklist item still passed because it audits only
  the *gate*. That was the one leakage path surviving every checklist item by
  construction.
* **`D44` — construct validity.** PR on the K x K *contemporaneous* correlation
  matrix is blind to cross-lag structure, while the model consumes a K x 96
  block and embeds each variate's entire lookback. Two variates can be
  near-uncorrelated contemporaneously yet near-redundant to a model with a
  96-hour lookback. A lookback-aware measure is therefore reported on the same
  rungs, and the divergence between them is reported whatever it turns out to
  be. If the construct does not correspond to what the architecture consumes,
  the second contribution is a measurement-validity failure rather than a
  finding — which is what a methods referee will spend the review on.
* **`D04` — the instance-normalisation confound.** ``use_norm=True`` divides each
  window by its own per-variate sigma over L, so the F2 estimators contribute
  *shape*, not *level*. The 8->12 rung can flatten for a reason that has nothing
  to do with redundancy. PR is therefore measured on window-normalised features
  as well as raw, both reported, and the confound disclosed in Limitations
  whatever RQ1 returns.

Provenance for the statistic itself: Laloux et al. (1999), Plerou et al. (2002).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import polars as pl


#: Root §8.5's Stage 3b trigger, pre-registered numerically **before** measuring:
#: a gate without a number stated in advance is not a gate (`D02`).
GATE_PR_FLOOR: float = 5.0

#: Windows sampled for the lookback-aware measures. The stable rank is a per
#: window SVD, so the cost is linear in this, and the sample is drawn by a fixed
#: stride — deterministic, not random, so the number is regenerable under root
#: §12 without carrying a seed.
LOOKBACK_SAMPLE: int = 2_000


def participation_ratio(eigenvalues: np.ndarray) -> float:
    """``PR = (sum lambda)^2 / sum lambda^2``, bounded in ``[1, K]``.

    PR is 1 when one eigenvalue carries everything and K when the spectrum is
    flat, so it reads directly as "how many independent directions are actually
    here". Negative eigenvalues from floating-point noise on a
    positive-semidefinite matrix are clipped to zero rather than dropped, since
    dropping them would change the trace and so the numerator.

    Raises:
        ValueError: If the spectrum sums to zero — a degenerate block that no
            downstream number could be computed from.
    """
    lam = np.clip(np.asarray(eigenvalues, dtype=np.float64), 0.0, None)
    total = lam.sum()
    if total <= 0:
        raise ValueError("degenerate spectrum: eigenvalues sum to zero")
    return float(total * total / np.square(lam).sum())


def contemporaneous_pr(values: np.ndarray) -> float:
    """PR of the ``K x K`` correlation matrix — Table 2b's first column.

    Args:
        values: ``(n, K)`` observations, one row per bar.

    The correlation matrix rather than the covariance, so the statistic is
    invariant to the arbitrary units the twelve variates carry: log-returns,
    log-volumes and a bounded ratio do not share a scale, and on the covariance
    the log-volume channel would dominate the spectrum for that reason alone.
    """
    corr = np.atleast_2d(
        np.corrcoef(np.asarray(values, dtype=np.float64), rowvar=False)
    )
    return participation_ratio(np.linalg.eigvalsh(corr))


def window_normalised_pr(windows: np.ndarray) -> float:
    """PR after per-window standardisation over L — `D04`'s required companion.

    Args:
        windows: ``(n, L, K)``.

    This reproduces exactly what ``use_norm=True`` hands the embedding: each
    channel of each window centred and divided by its own sigma over the
    lookback. Level information is gone by construction, so if the raw and
    normalised PR disagree at the K=12 rung, the F2 estimators' apparent
    redundancy is an artefact of the normalisation rather than a property of the
    data — and RQ1's axis is confounded in a way no post-hoc analysis removes.
    """
    x = np.asarray(windows, dtype=np.float64)
    mean = x.mean(axis=1, keepdims=True)
    std = np.sqrt(x.var(axis=1, keepdims=True) + 1e-12)
    return contemporaneous_pr(((x - mean) / std).reshape(-1, x.shape[2]))


def stable_rank(matrix: np.ndarray) -> float:
    """``||M||_F^2 / ||M||_2^2`` — bounded in ``[1, min(rows, cols)]``.

    The lookback-aware measure that is *commensurable* with the contemporaneous
    PR: applied to a ``K x L`` block with ``K <= 12 < 96`` it lives in the same
    ``[1, K]`` interval, so the two sit in one table and can be compared rung by
    rung. The ``K*L x K*L`` covariance PR below cannot — its ceiling is ``K*L``,
    which is 1,152 at K=12.
    """
    singular = np.linalg.svd(np.asarray(matrix, dtype=np.float64), compute_uv=False)
    if singular[0] <= 0:
        raise ValueError("degenerate window block: largest singular value is 0")
    return float(np.square(singular).sum() / (singular[0] ** 2))


def lookback_stable_rank(windows: np.ndarray, sample: int = LOOKBACK_SAMPLE) -> float:
    """Mean stable rank of each window's ``K x L`` block, **as the model sees it**.

    Args:
        windows: ``(n, L, K)``.
        sample: Windows to evaluate, taken by a fixed stride across the whole
            span so the sample spreads over the sub-block rather than
            concentrating at its head.

    Each channel is standardised **within its window** first — exactly what
    ``use_norm=True`` does before the embedding. Centring alone is not enough and
    the difference is not cosmetic: measured on origin 1, the merely-centred
    version returns 1.00 / 1.00 / 1.16 / 1.65 across the four rungs, because
    ``log_quote_volume`` deviations are two orders of magnitude larger than
    ``r`` deviations in absolute terms, so one row dominates both the Frobenius
    and the spectral norm and the statistic reports "one effective direction" at
    every rung. That is a units artefact, not a finding about the data.

    With standardisation the quantity has a closed form worth stating: the block
    is ``K x L`` with unit-variance rows, so ``||M||_F^2 = K L`` and
    ``sigma_1^2 = L lambda_1`` where ``lambda_1`` is the leading eigenvalue of
    the **within-window** correlation matrix of the K channels. The stable rank
    is therefore ``K / lambda_1`` — the reciprocal of the dominant direction's
    share, bounded in ``[1, K]`` and directly comparable to the contemporaneous
    PR beside it.
    """
    x = np.asarray(windows, dtype=np.float64)
    if len(x) == 0:
        raise ValueError("no windows to measure")
    stride = max(1, len(x) // sample)
    blocks = np.transpose(x[::stride][:sample], (0, 2, 1))   # (m, K, L)
    blocks = blocks - blocks.mean(axis=2, keepdims=True)
    blocks = blocks / np.sqrt(np.square(blocks).mean(axis=2, keepdims=True) + 1e-12)
    return float(np.mean([stable_rank(b) for b in blocks]))


def lookback_covariance_pr(windows: np.ndarray) -> float:
    """PR of the ``K*L x K*L`` **correlation** spectrum — §5.4's first alternative.

    The correlation matrix, not the covariance §5.4 names literally. On the raw
    covariance the statistic is dominated by whichever channel happens to carry
    the largest variance, and it stops being monotone in K: measured on origin 1
    it returned 92.1 / 3.0 / 44.0 / 8.8 across the four rungs, where the drop
    from K=1 to K=4 is entirely the arrival of ``log_quote_volume`` and says
    nothing about dimensionality. Standardising the ``K*L`` columns first makes
    it scale-free, exactly as :func:`contemporaneous_pr` uses the correlation
    matrix for the same reason.

    This is the only measure here that sees genuine **cross-lag** structure — the
    stable rank above sees cross-*variate* structure inside a window. Its ceiling
    is ``K*L``, so it is **not** on the contemporaneous PR's scale; report it as
    a fraction of that ceiling (:attr:`KeffRow.pr_lookback_ratio`) when comparing
    rungs.
    """
    x = np.asarray(windows, dtype=np.float64)
    flat = x.reshape(len(x), -1)
    flat = flat - flat.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.square(flat).mean(axis=0, keepdims=True) + 1e-24)
    flat = flat / scale
    gram = (flat.T @ flat) / max(1, len(flat) - 1)
    return participation_ratio(np.linalg.eigvalsh(gram))


@dataclass(frozen=True, slots=True)
class KeffRow:
    """One (origin, rung) cell of Table 2b."""

    origin: str
    origin_index: int
    k: int
    n_rows: int
    n_windows: int
    pr_raw: float
    pr_window_norm: float
    stable_rank_lookback: float
    pr_lookback_cov: float

    @property
    def pr_lookback_ratio(self) -> float:
        """``pr_lookback_cov / (K * L)`` — the cross-lag PR as a share of its ceiling.

        The raw value lives in ``[1, K*L]`` and so cannot be compared rung to
        rung; this can.
        """
        return self.pr_lookback_cov / (self.k * SEQ_LEN)

    @property
    def divergence(self) -> float:
        """``stable_rank - pr_raw`` — §5.4 requires this be reported as such.

        Positive means the lookback carries structure the contemporaneous
        correlation cannot see; negative means variates that look independent
        bar-to-bar turn redundant once 96 hours of each are in view. Either way
        it goes in §4.1b: if the effective-dimensionality construct does not
        correspond to what the architecture consumes, the study's second claimed
        contribution is a measurement-validity failure rather than a finding.
        """
        return self.stable_rank_lookback - self.pr_raw


def _training_windows(
    features: pl.DataFrame, origin: OriginLike, k: int, seq_len: int = SEQ_LEN
) -> tuple[np.ndarray, np.ndarray]:
    """Rows and windows of one origin's 21-month training sub-block.

    Both are cut from ``[train_start, train_sub_end)`` and nothing else — the
    span the scaler is fitted on. This is `D44`'s closure: RQ1's regressor may
    not see a single bar its outcome is measured on.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()

    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    rows = values[(ts >= lo) & (ts < hi)]

    starts = window_starts(
        ts, origin.train_start, origin.train_sub_end, "contained", WINDOW_SPAN
    )
    if len(starts) == 0:
        raise ValueError(f"origin {origin.label}: no training window to measure")
    idx = starts[:, None] + np.arange(seq_len)[None, :]
    return rows, values[idx]


def keff_row(features: pl.DataFrame, origin: OriginLike, k: int) -> KeffRow:
    """Measure every K_eff variant for one (origin, rung) cell."""
    rows, windows = _training_windows(features, origin, k)
    return KeffRow(
        origin=origin.label,
        origin_index=origin.index,
        k=k,
        n_rows=len(rows),
        n_windows=len(windows),
        pr_raw=contemporaneous_pr(rows),
        pr_window_norm=window_normalised_pr(windows),
        stable_rank_lookback=lookback_stable_rank(windows),
        pr_lookback_cov=lookback_covariance_pr(windows),
    )


def keff_table(
    features: pl.DataFrame,
    origins: list[OriginLike] | None = None,
    rungs: tuple[int, ...] = K_LADDER,
) -> pl.DataFrame:
    """Table 2b — every rung at every origin, on training spans only.

    Returns:
        One row per (origin, rung): the raw, window-normalised and two
        lookback-aware measures side by side, plus their divergence.

    K=1 is included even though its PR is identically 1. §9.1's horse race
    regresses on the rung's K_eff, and dropping the rung whose value is known in
    advance would unbalance the panel for no gain.
    """
    grid = list(origins if origins is not None else ORIGINS)
    return pl.DataFrame(
        [
            {
                "origin": row.origin,
                "origin_index": row.origin_index,
                "k": row.k,
                "n_rows": row.n_rows,
                "n_windows": row.n_windows,
                "pr_raw": row.pr_raw,
                "pr_window_norm": row.pr_window_norm,
                "stable_rank_lookback": row.stable_rank_lookback,
                "pr_lookback_cov": row.pr_lookback_cov,
                "pr_lookback_ratio": row.pr_lookback_ratio,
                "divergence": row.divergence,
            }
            for origin in grid
            for k in rungs
            for row in (keff_row(features, origin, k),)
        ]
    )


def corr_k_keff(table: pl.DataFrame, column: str = "pr_raw") -> float:
    """``corr(K, K_eff)`` across the rungs — §9.1 requires it in Table 2b.

    A reader is entitled to this before reading the non-nested comparison: if it
    sits near 1, the two theories are close to collinear and the horse race has
    little to separate them, whatever the reported p-value says.
    """
    means = table.group_by("k").agg(pl.col(column).mean().alias("keff")).sort("k")
    k = means.get_column("k").to_numpy().astype(np.float64)
    keff = means.get_column("keff").to_numpy()
    return float(np.corrcoef(k, keff)[0, 1])


def gate_pr(features: pl.DataFrame, k: int = 8) -> float:
    """Stage 3b's gate value — **pre-first-origin span only** (`D02`).

    Computed on ``[2018-01, 2020-01)``, which contains no origin's test block.
    The full-sample rolling PR is descriptive and may inform no design decision:
    every origin's test block lies inside it, so a ladder re-cut driven by it
    would be a design choice made with the answers already in hand.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    lo = int(DATA_START.timestamp() * 1000)
    hi = int(FIRST_ORIGIN.timestamp() * 1000)
    rows = features.select(columns).to_numpy()[(ts >= lo) & (ts < hi)]
    if len(rows) == 0:
        raise ValueError("the pre-first-origin span holds no usable bar")
    return contemporaneous_pr(rows)


def gate_verdict(measured: float, floor: float = GATE_PR_FLOOR) -> str:
    """Stage 3b's action, which is **disclosure, not a re-cut** (`D48`).

    §8.5 originally said a PR below the floor should re-cut the ladder, but
    `D01` establishes that exactly one consistent cut exists over F1-F5, so
    "re-cut" named no reachable alternative. The gate therefore reports, the
    grid proceeds unchanged, and the divergence from §5.2's expected values is
    disclosed in §4.1b. A gate whose only action is unreachable is not a gate.
    """
    if measured >= floor:
        return (
            f"PASS: measured PR at K=8 is {measured:.3f} >= {floor:.1f}. "
            f"Proceed; report the value in Table 2b."
        )
    return (
        f"DISCLOSE: measured PR at K=8 is {measured:.3f} < {floor:.1f}. "
        f"Root §8.5 / `D48` — proceed unchanged and disclose the divergence "
        f"from §5.2's expected K_eff in §4.1b. Do not re-cut the ladder: `D01` "
        f"leaves no second consistent cut over F1-F5."
    )


In [ ]:
# ═══ efficiency.py ════════
"""Root §4.5's preliminary market-efficiency tests.

Run once, reported in the Data section. The point is to convert "efficient
market" from an assumption into a finding: §4.5 forbids *claiming* the market is
efficient and requires stating that the evidence is mixed and time-varying
(Urquhart 2016; Nadarajah & Chu 2017; Bariviera 2017; Sensoy 2019), then
reporting our own numbers beside it.

Reported over two spans, because "time-varying" is a claim about variation and
one full-sample row cannot exhibit it: the whole sample, and each origin's
**21-month training sub-block** -- the same span the scaler is fitted on. Those
rows are descriptive and gate nothing, so the span rule `D44` imposes on K_eff
does not bind, but reading a test block merely to describe the data would still
be indefensible when avoiding it costs one filter.

``arch`` and ``statsmodels`` accept numpy arrays, so this module crosses root
§16's stats boundary without pandas ever entering the process.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Final

import numpy as np
import polars as pl


#: Lags for the Lo-MacKinlay variance ratio. Powers of two spanning two hours to
#: two thirds of a day, which brackets the horizons this study forecasts.
VR_LAGS: Final[tuple[int, ...]] = (2, 4, 8, 16)

#: Smallest R/S block. Below ~16 points the rescaled range is dominated by its
#: own small-sample bias and the log-log slope bends upward on white noise.
HURST_MIN_N: Final = 16

#: Blocks needed for the log-log regression to mean anything. Two points define a
#: line exactly and would report a slope with no residual to doubt it.
HURST_MIN_BLOCK_SIZES: Final = 3

#: "c" admits a non-zero drift in the random walk, which BTC plainly has. Passed
#: explicitly rather than left to the library default: root §16 forbids a magic
#: number, and a silent default is worse than one -- it is a magic number nobody
#: can see.
VR_TREND: Final = "c"


@dataclass(frozen=True, slots=True)
class VarianceRatioRow:
    """One Lo-MacKinlay variance ratio. ``vr`` near 1 is consistent with a random walk."""

    lag: int
    vr: float
    statistic: float
    p_value: float


@dataclass(frozen=True, slots=True)
class ADFRow:
    """Augmented Dickey-Fuller on log-returns, which should reject a unit root."""

    statistic: float
    p_value: float
    used_lag: int
    n_obs: int


def hurst_rs(x: np.ndarray, min_n: int = HURST_MIN_N, max_n: int | None = None) -> float:
    """Hurst exponent by rescaled range, read off the log-log plot as an OLS slope.

    ``H ~ 0.5`` on log-returns is the no-long-memory reading §4.5 pins. Block
    sizes are dyadic; at each size the series is cut into non-overlapping blocks
    and the mean R/S over them is the point that enters the regression.

    Args:
        x: The series to measure. Table 2 passes **log-returns**; passing a level
            series is the control that shows the estimator responds to memory at
            all, and returns ``H`` near 1.
        min_n: Smallest block. See :data:`HURST_MIN_N`.
        max_n: Largest block, defaulting to ``len(x) // 4`` so the largest size
            still averages over four blocks rather than reporting one range.

    Returns:
        The estimated Hurst exponent.

    Raises:
        ValueError: If fewer than :data:`HURST_MIN_BLOCK_SIZES` usable block
            sizes fit inside the series.
    """
    x = np.asarray(x, dtype=np.float64)
    n_total = len(x)
    if max_n is None:
        max_n = n_total // 4
    sizes = [n for n in (min_n * 2**i for i in range(64)) if n <= max_n]
    if len(sizes) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"R/S needs at least {HURST_MIN_BLOCK_SIZES} block sizes between "
            f"{min_n} and {max_n}; got {len(sizes)} at n={n_total}"
        )

    logs_n: list[float] = []
    logs_rs: list[float] = []
    for n in sizes:
        blocks = x[: (n_total // n) * n].reshape(-1, n)
        deviate = np.cumsum(blocks - blocks.mean(axis=1, keepdims=True), axis=1)
        spread = deviate.max(axis=1) - deviate.min(axis=1)
        sd = blocks.std(axis=1, ddof=1)
        # A constant block has no scale to rescale by. Dropping it is not
        # imputation -- nothing is invented, the block simply carries no R/S.
        keep = sd > 0
        if not keep.any():
            continue
        logs_n.append(float(np.log(n)))
        logs_rs.append(float(np.log(float((spread[keep] / sd[keep]).mean()))))

    if len(logs_n) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"only {len(logs_n)} block sizes carried a non-zero standard deviation"
        )
    slope, _ = np.polyfit(np.asarray(logs_n), np.asarray(logs_rs), 1)
    return float(slope)


def variance_ratios(
    r: np.ndarray, lags: tuple[int, ...] = VR_LAGS
) -> list[VarianceRatioRow]:
    """Lo-MacKinlay variance ratio at each lag, from a **return** series.

    ``arch.unitroot.VarianceRatio`` consumes a **level** series and differences
    it itself, so the returns are cumulated back to a level here rather than
    handed over raw. Fed the returns directly it reports ``VR = 1/lag`` -- 0.49,
    0.25, 0.12, 0.06 at lags 2, 4, 8, 16 on white noise -- the signature of
    over-differencing, and a p-value of 0.0000 that would read as decisive
    evidence against a random walk while being evidence of nothing at all. The
    unit tests pin both directions.

    The cumulation is safe across gaps and does not smuggle one in. ``r`` is
    computed **per segment** with each segment's first bar dropped (root §4.3),
    so the series contains no cross-gap return; the cumulative sum merely glues
    the segments into a pseudo-level whose first differences are exactly the
    returns the estimator then recovers. No value is fabricated at a boundary,
    which is what §2's no-imputation rule is about.

    Args:
        r: Log-returns.
        lags: Multi-period horizons for the numerator variance.

    Returns:
        One row per lag, in the order given.
    """
    from arch.unitroot import VarianceRatio  # root §16's named stats boundary

    level = np.cumsum(np.asarray(r, dtype=np.float64))
    rows: list[VarianceRatioRow] = []
    for lag in lags:
        ratio = VarianceRatio(level, lags=lag, trend=VR_TREND, overlap=True)
        rows.append(
            VarianceRatioRow(
                lag=lag,
                vr=float(ratio.vr),
                statistic=float(ratio.stat),
                p_value=float(ratio.pvalue),
            )
        )
    return rows


def adf(r: np.ndarray) -> ADFRow:
    """Augmented Dickey-Fuller with AIC lag selection, on log-returns."""
    from statsmodels.tsa.stattools import adfuller  # root §16's named stats boundary

    stat, p_value, used_lag, n_obs, *_ = adfuller(
        np.asarray(r, dtype=np.float64), autolag="AIC"
    )
    return ADFRow(
        statistic=float(stat),
        p_value=float(p_value),
        used_lag=int(used_lag),
        n_obs=int(n_obs),
    )


def _row(span: str, r: np.ndarray) -> dict[str, float | str | int]:
    """One Table 2 row: ADF, Hurst, and the variance ratio at every lag."""
    unit_root = adf(r)
    row: dict[str, float | str | int] = {
        "span": span,
        "n": int(len(r)),
        "adf_stat": unit_root.statistic,
        "adf_p": unit_root.p_value,
        "hurst": hurst_rs(r),
    }
    for ratio in variance_ratios(r):
        row[f"vr_{ratio.lag}"] = ratio.vr
        row[f"vr_p_{ratio.lag}"] = ratio.p_value
    return row


def _training_returns(features: pl.DataFrame, origin: OriginLike) -> np.ndarray:
    """Log-returns of one origin's 21-month training sub-block.

    ``[train_start, train_sub_end)``, half-open, matching
    :func:`itransformer_btc.keff._training_windows` exactly -- the two must cut
    the same span or Table 2 and Table 2b describe different data.
    """
    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    return (
        features.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))
        .get_column("r")
        .to_numpy()
    )


#: Shortest sub-block the R/S regression can describe: enough rows for
#: :data:`HURST_MIN_BLOCK_SIZES` dyadic sizes, each averaged over four blocks.
MIN_SPAN_ROWS: Final = HURST_MIN_N * 2 ** (HURST_MIN_BLOCK_SIZES - 1) * 4


def efficiency_table(
    features: pl.DataFrame, origins: list[OriginLike] | None = None
) -> pl.DataFrame:
    """Table 2 -- one row for the whole sample, one per origin's training sub-block.

    Args:
        features: The frame :func:`itransformer_btc.features.build_features`
            returns, carrying ``ts_ms`` and ``r``.
        origins: Defaults to the full walk-forward grid.

    Returns:
        ``span, n, adf_stat, adf_p, hurst`` plus a ``vr_{lag}`` / ``vr_p_{lag}``
        pair per lag. The ``"full"`` row comes first; origin rows follow in
        walk-forward order.

    An origin whose sub-block is shorter than :data:`MIN_SPAN_ROWS` is skipped.
    On the real artifact that never happens -- the shortest sub-block holds
    roughly 15,000 bars -- so the branch exists for synthetic and truncated
    frames, and a caller can tell it fired by comparing the row count against
    ``len(origins) + 1``.
    """
    grid = list(origins if origins is not None else ORIGINS)
    rows = [_row("full", features.get_column("r").to_numpy())]
    for origin in grid:
        returns = _training_returns(features, origin)
        if len(returns) < MIN_SPAN_ROWS:
            continue
        rows.append(_row(origin.label, returns))
    return pl.DataFrame(rows)


In [ ]:
# ═══ metrics.py ════════
"""Metrics, tests, and the three RQ estimators.

Root §9. Everything here consumes ``preds/{run_id}.parquet`` and
``meta/{run_id}.json`` and produces numbers that go straight into
``artifacts/paper_numbers.json`` — which is why nothing in this module reads a
model or a tensor. Root §12: a number that cannot be regenerated from a
persisted prediction file plus a config hash is a documented failure, not a
footnote.

Six decisions here are load-bearing, and each reverses something the source
design said:

* **`D31`** — Naive-RW is ``y_z = -mu_g/sigma_g``, never ``y_z = 0``. In scaler
  space zero means ``r_hat = mu_g``, the training-window mean hourly return: a
  constant-drift model wearing the EMH baseline's name. Measured, ``mu_g/sigma_g``
  spans -0.00818 … +0.01733 and **changes sign across origins**, so it is not a
  constant tilt a reader could mentally subtract.
* **`D23`** — ``D(i,b)`` lives on the **skill** scale, not on RelMSE. On RelMSE
  every pre-registered tau is arithmetically unreachable and RQ3 returns "no
  decay detected" by construction, before a single epoch runs.
* **`D05` follow-on** — the decay denominator is the **within-origin mean**, not
  block 1. One 30-day block under heavy tails would sit in the denominator of
  five quantities, making their errors perfectly correlated, and ``b*`` reads a
  threshold crossing straight off the series, so an unlucky block 1 moves the
  crossing by whole blocks.
* **`D42`** — every ratio metric is formed from **seed-averaged MSEs**, never
  from an average of per-seed ratios. The two differ by Jensen, and the second
  additionally requires pairing seed 42 at K=1 with seed 42 at K=8, which are
  independent training runs of different models: any of 5! orderings gives a
  different answer.
* **`D29`** — nested pairs get **Clark-West**, not Diebold-Mariano. Under the
  null with nested models and estimated parameters the loss differential has a
  mean shifted away from zero, so standard DM is systematically undersized
  against the alternative this study exists to establish.
* **`D34`** — the long-run variance estimator is **rectangular**, not Bartlett.
  Under the DM null, h-step optimal forecast errors are MA(h-1), so every
  autocovariance to lag 23 is genuinely nonzero and equally real; Bartlett
  weights shrink the lag-22 term by ~92%, understating the variance and
  producing exactly the over-optimistic p-values this module exists to prevent.
"""

from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import polars as pl


#: ``{model}_o{origin:02d}_K{K:02d}_H{H:03d}_s{seed}`` (root §10.4).
RUN_ID_PATTERN = re.compile(
    r"^(?P<model>[a-z0-9]+)_o(?P<origin>\d{2})_K(?P<k>\d{2})"
    r"_H(?P<h>\d{3})_s(?P<seed>\d+)$"
)

HOUR_MS = 3_600_000

#: Root §3's pre-registered thresholds. The headline is 5%; the rest are
#: sensitivities. Choosing tau after seeing the decay curve is p-hacking.
TAU_HEADLINE: float = 0.05
TAU_SENSITIVITY: tuple[float, ...] = (0.025, 0.05, 0.10, 0.50)

#: Declared so `DecayResult.b_star` keeps its columns when every origin is
#: excluded (`D55`). An inferred schema over zero rows yields a frame with no
#: columns at all, and the caller's ``bs["b_star"]`` then raises rather than
#: reporting the pre-registered null.
B_STAR_SCHEMA: dict[str, pl.DataType] = {
    "origin": pl.Utf8,
    "tau": pl.Float64,
    "b_star": pl.Int64,
    "event": pl.Boolean,
}


# -- artifact I/O ------------------------------------------------------------


def parse_run_id(run_id: str) -> dict[str, int | str]:
    """Decompose a ``run_id`` into its five components.

    Raises:
        ValueError: If it does not match root §10.4's pattern. A run whose id
            cannot be parsed cannot be placed in the grid, and skipping it
            silently would drop a cell from a table without saying so.
    """
    match = RUN_ID_PATTERN.match(run_id)
    if match is None:
        raise ValueError(f"{run_id!r} is not a root §10.4 run_id")
    g = match.groupdict()
    return {
        "model": g["model"],
        "origin_index": int(g["origin"]),
        "k": int(g["k"]),
        "pred_len": int(g["h"]),
        "seed": int(g["seed"]),
    }


def _locate(run_id: str, roots: list[Path], kind: str, suffix: str) -> Path:
    for root in roots:
        candidate = Path(root) / kind / f"{run_id}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{kind}/{run_id}{suffix} in none of {[str(r) for r in roots]}"
    )


def load_predictions(run_id: str, roots: list[Path]) -> pl.DataFrame:
    """Read one run's raw predictions.

    Roots are searched in order, so a working directory listed first shadows an
    older attached dataset — which is what lets a single re-run cell take effect
    without deleting the previous session's output.
    """
    return pl.read_parquet(_locate(run_id, roots, "preds", ".parquet"))


def load_meta(run_id: str, roots: list[Path]) -> dict:
    return json.loads(_locate(run_id, roots, "meta", ".json").read_text())


# -- point metrics -----------------------------------------------------------


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.square(np.asarray(y_true) - np.asarray(y_pred))))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rel_mse(model: float, naive: float) -> float:
    """``MSE_model / MSE_naive`` — controls for period difficulty (root §9.1)."""
    return model / naive


def r2_oos(model: float, naive: float) -> float:
    """``1 - RelMSE`` (`D20`) — the readable form of the same quantity.

    RelMSE near 1.00 is hard to read; ``R2_oos`` reads directly as skill against
    a random walk, and its sign is the whole question.
    """
    return 1.0 - rel_mse(model, naive)


def raw_rmse(mse_z: float, sigma_g: float) -> float:
    """RMSE back in raw log-return units — root §9.1's second reporting scale.

    "RMSE 0.0043 on hourly log-returns" tells a reader far more than "MSE 0.187
    on normalized data", and stating ``sigma_g`` is what lets the two reconcile.
    """
    return math.sqrt(mse_z) * sigma_g


def non_overlapping_mask(timestamps: np.ndarray) -> np.ndarray:
    """Window starts whose forecast period opens at **00:00 UTC** (`D46`).

    There are 24 admissible alignments of a non-overlapping daily partition and
    each gives a different Sharpe, MDD and turnover, so the phase is fixed in
    advance rather than chosen after seeing the equity curve.

    ``timestamp`` in the prediction file is the **window start**; the first
    target hour is ``start + L``. With ``L = 96`` a multiple of 24 the phase is
    preserved, so selecting starts at hour 0 selects targets opening at hour 0.
    """
    return (np.asarray(timestamps) // HOUR_MS) % 24 == 0


# -- directional accuracy and its testing regime (`D21`) ---------------------


def pesaran_timmermann(actual: np.ndarray, predicted: np.ndarray) -> tuple[float, float]:
    """Pesaran-Timmermann (1992) test of directional predictability.

    Returns:
        ``(statistic, one-sided p)`` against ``N(0,1)``. Without a null
        hypothesis, directional accuracy is a descriptive number; this supplies
        the null.

    Zero targets are excluded rather than assigned a direction: a zero
    log-return has no sign to predict, and assigning one would inflate the hit
    rate by whatever the model happened to output there.
    """
    a = np.sign(np.asarray(actual, dtype=np.float64))
    f = np.sign(np.asarray(predicted, dtype=np.float64))
    keep = a != 0
    a, f = a[keep], f[keep]
    n = len(a)
    if n < 2:
        return float("nan"), float("nan")

    hit = float(np.mean(a == f))
    py = float(np.mean(a > 0))
    px = float(np.mean(f > 0))
    p_star = py * px + (1 - py) * (1 - px)

    var_hit = p_star * (1 - p_star) / n
    var_star = (
        (2 * py - 1) ** 2 * px * (1 - px)
        + (2 * px - 1) ** 2 * py * (1 - py)
        + 4 * py * px * (1 - py) * (1 - px) / n
    ) / n
    denom = var_hit - var_star
    if denom <= 0:
        return float("nan"), float("nan")

    stat = (hit - p_star) / math.sqrt(denom)
    return stat, 0.5 * math.erfc(stat / math.sqrt(2.0))


def _hit_rate(actual: np.ndarray, predicted: np.ndarray) -> float:
    keep = np.sign(actual) != 0
    if not keep.any():
        return float("nan")
    return float(np.mean(np.sign(actual[keep]) == np.sign(predicted[keep])))


@dataclass(frozen=True, slots=True)
class DirectionalAccuracy:
    """DA at the three horizons §9.1 requires, with their testing regimes.

    ``da_h24`` and ``da_cum`` carry p-values **only** on the non-overlapping
    sample. On hourly spacing their targets overlap by 23 of 24 hours, giving
    lag-1 autocorrelation of about 23/24; Pesaran-Timmermann's variance is then
    far too small and the test over-rejects badly. The overlapping figures are
    reported descriptively, without p-values, and the resulting power loss
    (T = 30 per block) is stated rather than recovered by using the invalid
    sample.
    """

    da_h1: float
    p_h1: float
    da_hH_overlapping: float
    da_hH: float
    p_hH: float
    da_cum_overlapping: float
    da_cum: float
    p_cum: float
    n_h1: int
    n_non_overlapping: int


def directional_accuracy(frame: pl.DataFrame) -> DirectionalAccuracy:
    """Compute all three DA variants for one run's predictions."""
    last_step = int(frame.get_column("step").max())

    step1 = frame.filter(pl.col("step") == 1)
    a1 = step1.get_column("y_true").to_numpy()
    f1 = step1.get_column("y_pred").to_numpy()
    _, p1 = pesaran_timmermann(a1, f1)

    step_h = frame.filter(pl.col("step") == last_step)
    ts_h = step_h.get_column("timestamp").to_numpy()
    a_h = step_h.get_column("y_true").to_numpy()
    f_h = step_h.get_column("y_pred").to_numpy()
    keep_h = non_overlapping_mask(ts_h)
    _, p_h = pesaran_timmermann(a_h[keep_h], f_h[keep_h])

    cum = (
        frame.group_by("timestamp")
        .agg(pl.col("y_true").sum(), pl.col("y_pred").sum())
        .sort("timestamp")
    )
    ts_c = cum.get_column("timestamp").to_numpy()
    a_c = cum.get_column("y_true").to_numpy()
    f_c = cum.get_column("y_pred").to_numpy()
    keep_c = non_overlapping_mask(ts_c)
    _, p_c = pesaran_timmermann(a_c[keep_c], f_c[keep_c])

    return DirectionalAccuracy(
        da_h1=_hit_rate(a1, f1),
        p_h1=p1,
        da_hH_overlapping=_hit_rate(a_h, f_h),
        da_hH=_hit_rate(a_h[keep_h], f_h[keep_h]),
        p_hH=p_h,
        da_cum_overlapping=_hit_rate(a_c, f_c),
        da_cum=_hit_rate(a_c[keep_c], f_c[keep_c]),
        p_cum=p_c,
        n_h1=len(a1),
        n_non_overlapping=int(keep_c.sum()),
    )


# -- per-block tables --------------------------------------------------------


def assert_same_windows(left: pl.DataFrame, right: pl.DataFrame, what: str) -> None:
    """`D45` — two models may only be compared on identical evaluated windows.

    Naive-RW needs no 96-bar lookback, so unless it is restricted to the window
    set its comparator actually evaluated, RelMSE is a ratio across two different
    samples. Test-window survival is conditioned on *future* gaps and outages
    cluster on stress, so the two samples would differ precisely in their
    high-volatility content.

    Raises:
        ValueError: If the evaluated ``(block, timestamp)`` sets differ.
    """
    def _key(frame: pl.DataFrame) -> np.ndarray:
        pairs = frame.select(["block", "timestamp"]).unique().sort(["block", "timestamp"])
        return (
            pairs.get_column("block").to_numpy().astype(np.int64) * (1 << 44)
            + pairs.get_column("timestamp").to_numpy().astype(np.int64) // HOUR_MS
        )

    a, b = _key(left), _key(right)
    if len(a) != len(b) or not np.array_equal(a, b):
        raise ValueError(
            f"{what}: evaluated window sets differ ({len(a)} vs {len(b)} "
            f"windows). RelMSE across two samples is not a ratio."
        )


def block_metrics(frame: pl.DataFrame, naive_z: float) -> pl.DataFrame:
    """Per-block MSE, MAE, RelMSE and ``R2_oos`` for one run.

    Args:
        frame: One run's predictions.
        naive_z: ``-mu_g/sigma_g`` from ``meta.json`` (`D31`). Passing 0 here
            silently substitutes a constant-drift model for the EMH baseline.

    The Naive-RW error is computed on exactly the rows the model was scored on,
    so :func:`assert_same_windows` has nothing to check for this pair — the
    sample is shared by construction. It still applies across *models*.
    """
    return (
        frame.with_columns(
            (pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"),
            (pl.col("y_true") - pl.col("y_pred")).abs().alias("_ae"),
            (pl.col("y_true") - naive_z).pow(2).alias("_se_naive"),
        )
        .group_by("block")
        .agg(
            pl.col("timestamp").n_unique().alias("n_windows"),
            pl.col("_se").count().alias("n_points"),
            pl.col("_se").mean().alias("mse"),
            pl.col("_ae").mean().alias("mae"),
            pl.col("_se_naive").mean().alias("mse_naive"),
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort("block")
    )


def gather_grid(run_ids: list[str], roots: list[Path]) -> pl.DataFrame:
    """Per (run, block) metrics for many runs — the input to every RQ estimator.

    Returns:
        Long frame with ``run_id, model, origin_index, origin, k, pred_len,
        seed, block, n_windows, mse, mae, mse_naive, rel_mse, r2_oos, sigma_g``.

    Raises:
        FileNotFoundError: If any run is absent. A quietly short grid produces a
            table whose cells came from different arms, which is worse than no
            table.
    """
    rows: list[pl.DataFrame] = []
    missing: list[str] = []
    for run_id in run_ids:
        try:
            preds = load_predictions(run_id, roots)
            meta = load_meta(run_id, roots)
        except FileNotFoundError:
            missing.append(run_id)
            continue
        parts = parse_run_id(run_id)
        rows.append(
            block_metrics(preds, float(meta["naive_rw_z"])).with_columns(
                pl.lit(run_id).alias("run_id"),
                pl.lit(str(parts["model"])).alias("model"),
                pl.lit(int(parts["origin_index"])).cast(pl.Int32).alias("origin_index"),
                pl.lit(str(meta["origin"])).alias("origin"),
                pl.lit(int(parts["k"])).cast(pl.Int32).alias("k"),
                pl.lit(int(parts["pred_len"])).cast(pl.Int32).alias("pred_len"),
                pl.lit(int(parts["seed"])).cast(pl.Int32).alias("seed"),
                pl.lit(float(meta["sigma_g"])).alias("sigma_g"),
            )
        )
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} of {len(run_ids)} runs are absent, first few: "
            f"{missing[:5]}. Complete the grid or pass the subset explicitly — "
            f"a silently short grid mixes arms inside one table."
        )
    if not rows:
        raise ValueError("no runs gathered")
    return pl.concat(rows)


def seed_average(grid: pl.DataFrame) -> pl.DataFrame:
    """Average MSE across seeds **before** any ratio is formed (`D42`).

    Seeds are computational noise, not population draws. Averaging ratios
    instead would differ by Jensen and would additionally require pairing seed
    42 at K=1 with seed 42 at K=8 — independent training runs of different
    models, where any of 5! orderings gives a different answer.

    The cell mean still carries Monte-Carlo error, which enters as measurement
    error in the dependent variable: unbiased for beta1, but inflating residual
    variance. ``n_seeds`` and ``mse_seed_std`` are carried so §9.2's dispersion
    rule can bind the error bar to the aggregation level (`D30`) — seed std is a
    Monte-Carlo diagnostic, never the uncertainty on an origin-aggregated row.
    """
    return (
        grid.group_by(["model", "origin_index", "origin", "k", "pred_len", "block"])
        .agg(
            pl.col("mse").mean().alias("mse"),
            pl.col("mae").mean().alias("mae"),
            pl.col("mse_naive").mean().alias("mse_naive"),
            pl.col("mse").std().alias("mse_seed_std"),
            pl.col("n_windows").first().alias("n_windows"),
            pl.col("sigma_g").first().alias("sigma_g"),
            pl.col("mse").count().alias("n_seeds"),
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort(["model", "origin_index", "k", "pred_len", "block"])
    )


# -- RQ2: the amplification gap and its decay --------------------------------


def amplification(
    seed_avg: pl.DataFrame,
    k_small: int = 1,
    k_large: int = 8,
    model: str = "itr",
    pred_len: int = 24,
) -> pl.DataFrame:
    """``A(i,b) = [MSE_K1 - MSE_K8] / MSE_K1`` — RQ2's dependent variable.

    **K=8, never K=12.** K=12 carries deliberate redundancy (root §5.2), so
    using it would confound decay with that redundancy — which is why the pair
    is a parameter with a pre-registered default rather than a free choice.

    Both models are evaluated on the same block, so period difficulty cancels in
    the ratio, and it cancels *well*: ``MSE_model`` and ``MSE_naive`` on one
    block correlate near 1. That is the argument the whole ratio-metric design
    rests on.
    """
    base = seed_avg.filter(
        (pl.col("model") == model) & (pl.col("pred_len") == pred_len)
    )
    small = (
        base.filter(pl.col("k") == k_small)
        .select(["origin_index", "origin", "block", "mse", "n_windows"])
        .rename({"mse": "mse_small", "n_windows": "n_small"})
    )
    large = (
        base.filter(pl.col("k") == k_large)
        .select(["origin_index", "block", "mse", "n_windows"])
        .rename({"mse": "mse_large", "n_windows": "n_large"})
    )

    joined = small.join(large, on=["origin_index", "block"], how="inner")
    if joined.height != small.height:
        raise ValueError(
            f"K={k_small} has {small.height} cells but only {joined.height} "
            f"matched K={k_large}; the panel must be balanced before beta1"
        )
    mismatched = joined.filter(pl.col("n_small") != pl.col("n_large"))
    if mismatched.height:
        raise ValueError(
            f"`D45`: {mismatched.height} cells evaluate K={k_small} and "
            f"K={k_large} on different window counts; A would be a ratio across "
            f"two samples"
        )
    return joined.with_columns(
        ((pl.col("mse_small") - pl.col("mse_large")) / pl.col("mse_small")).alias("A")
    ).sort(["origin_index", "block"])


def attention_amplification(
    seed_avg: pl.DataFrame, k: int = 8, pred_len: int = 24
) -> pl.DataFrame:
    """``A_attn(i,b) = [MSE_uniformK8 - MSE_K8] / MSE_uniformK8`` (`D50`).

    K=1 versus K=8 does **not** isolate attention: the two arms differ in
    *information* and in *whether attention is active* simultaneously, so a
    decaying ``A(b)`` is equally consistent with "cross-variate attention
    overfits regime-specific structure" — a capacity story — as with the
    information story RQ2 claims. This contrast holds information fixed and
    varies only what attention selects, at runs Figure 5 needs anyway.
    """
    sel = seed_avg.filter((pl.col("k") == k) & (pl.col("pred_len") == pred_len))
    uniform = (
        sel.filter(pl.col("model") == "itru")
        .select(["origin_index", "origin", "block", "mse"])
        .rename({"mse": "mse_uniform"})
    )
    attended = (
        sel.filter(pl.col("model") == "itr")
        .select(["origin_index", "block", "mse"])
        .rename({"mse": "mse_attended"})
    )
    return (
        uniform.join(attended, on=["origin_index", "block"], how="inner")
        .with_columns(
            (
                (pl.col("mse_uniform") - pl.col("mse_attended"))
                / pl.col("mse_uniform")
            ).alias("A_attn")
        )
        .sort(["origin_index", "block"])
    )


# -- RQ3: skill decay and the retraining cadence -----------------------------


@dataclass(frozen=True, slots=True)
class DecayResult:
    """Per-origin ``D(i,b)`` and the censored ``b*`` it implies."""

    table: pl.DataFrame
    excluded_origins: tuple[str, ...]

    def b_star(self, tau: float) -> pl.DataFrame:
        """``b*(i) = min{b : D(i,b) > tau}``, **right-censored at 6** (`D41`).

        ``min{.}`` does not commute with averaging, so pooling MSEs across
        origins and *then* taking the minimum is a different estimand and is
        forbidden. Each origin contributes one observation, censored or not.

        The schema is declared rather than inferred (`D55`). When `decay`'s
        non-positive-skill guard excludes *every* origin, ``self.table`` is empty
        and an inferred schema yields a frame with no columns, so a caller's
        ``bs["b_star"]`` raises ``ColumnNotFoundError`` — which is what took the
        Kaggle notebook down at the exact moment its grid output was the only
        thing worth keeping. That guard firing is the **expected** outcome under
        non-positive skill, not an edge case: root §10.3's first measured run
        returned ``R2_oos = -0.0183`` and the completed grid returned it at all
        fifteen origins. `D54e` gates the estimators on grid *completeness*,
        which is a different failure and does not cover this path.

        An empty return means the estimand is **undefined** — there is no edge to
        lose a proportion of — which is not the same as every origin being
        censored at 6, where an edge exists and simply never decays past tau.
        Callers must report the two differently; ``excluded_origins`` is what
        tells them apart.
        """
        rows = []
        for key, part in self.table.group_by("origin", maintain_order=True):
            label = key[0] if isinstance(key, tuple) else key
            crossed = part.filter(pl.col("D") > tau).sort("block")
            rows.append(
                {
                    "origin": str(label),
                    "tau": tau,
                    "b_star": int(crossed.get_column("block")[0]) if crossed.height else 6,
                    "event": bool(crossed.height),
                }
            )
        return pl.DataFrame(rows, schema=B_STAR_SCHEMA)


def decay(
    seed_avg: pl.DataFrame,
    k: int = 8,
    model: str = "itr",
    pred_len: int = 24,
) -> DecayResult:
    """``D(i,b)`` on the **skill** scale (`D23`), normalised within origin (`D05`).

    ``D(i,b) = [mean_b' R2_oos(i,b') - R2_oos(i,b)] / mean_b' R2_oos(i,b')``

    On the RelMSE scale the pre-registered thresholds are unreachable: with
    ``RelMSE(1) = 0.996``, even *total* destruction of the model's edge gives
    ``D = 1/0.996 - 1 = 0.402%``, so tau = 2.5% would require the model to become
    2% worse than forecasting zero. RQ3 would return "no decay detected"
    regardless of the data — a result fixed by a units mismatch rather than by
    the market. On the skill scale ``D`` runs from 0 (no decay) through 1 (edge
    fully gone) and the taus are commensurate with it.

    The denominator is the within-origin **mean** rather than block 1's value
    (`D05` follow-on): block 1 is one 30-day block under heavy tails and
    volatility clustering, and it would otherwise sit in the denominator of five
    quantities and make their errors perfectly correlated.

    **Origins with non-positive mean skill are excluded and named**, never
    silently dropped: ``D`` is a proportion of an edge, and an origin with no
    edge has no proportion of one. Root §10.3's first measured run returned
    ``R2_oos = -0.0183``, so this guard may well be the common case rather than
    the edge case §9.1 assumed.
    """
    sel = seed_avg.filter(
        (pl.col("model") == model)
        & (pl.col("k") == k)
        & (pl.col("pred_len") == pred_len)
    ).sort(["origin_index", "block"])

    reference = sel.group_by("origin").agg(pl.col("r2_oos").mean().alias("r2_ref"))
    joined = sel.join(reference, on="origin", how="left")

    excluded = tuple(
        sorted(joined.filter(pl.col("r2_ref") <= 0).get_column("origin").unique().to_list())
    )
    kept = joined.filter(pl.col("r2_ref") > 0).with_columns(
        ((pl.col("r2_ref") - pl.col("r2_oos")) / pl.col("r2_ref")).alias("D")
    )
    return DecayResult(
        table=kept.select(
            ["origin", "origin_index", "block", "n_windows", "r2_oos", "r2_ref", "D"]
        ).sort(["origin_index", "block"]),
        excluded_origins=excluded,
    )


# -- survival analysis for b* (`D41`) ----------------------------------------


@dataclass(frozen=True, slots=True)
class SurvivalCurve:
    """Kaplan-Meier estimate on the 30-day block grid."""

    times: np.ndarray
    survival: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    n_events: int
    n_censored: int

    @property
    def median(self) -> float:
        """Smallest block with ``S(t) <= 0.5``; ``inf`` when never reached.

        ``inf`` is the honest answer, and root §3 fixes its wording: *"no decay
        detected within 180 days"* — a right-censored result, not a missing one.
        """
        below = self.times[self.survival <= 0.5]
        return float(below[0]) if len(below) else float("inf")

    @property
    def median_interval(self) -> tuple[float, float]:
        """Confidence set for the median: blocks whose CI band straddles 0.5.

        Table 5 carries this interval, never a bare integer, and the abstract's
        recommended cadence is this interval.
        """
        straddle = self.times[(self.lower <= 0.5) & (self.upper >= 0.5)]
        if not len(straddle):
            return (self.median, self.median)
        return (float(straddle[0]), float(straddle[-1]))


def _normal_quantile(p: float) -> float:
    """Acklam's inverse normal CDF — avoids a scipy import for one number."""
    a = [-3.969683028665376e+01, 2.209460984245205e+02, -2.759285104469687e+02,
         1.383577518672690e+02, -3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02, -1.556989798598866e+02,
         6.680131188771972e+01, -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00, 2.938163982698783e+00]
    d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e+00,
         3.754408661907416e+00]
    plow, phigh = 0.02425, 1 - 0.02425
    if p < plow:
        q = math.sqrt(-2 * math.log(p))
        return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
               ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    if p > phigh:
        q = math.sqrt(-2 * math.log(1 - p))
        return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
                ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    q, r = p - 0.5, (p - 0.5) ** 2
    return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q / \
           (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)


def _loglog_band(surv: float, var_sum: float, z: float) -> tuple[float, float]:
    """Log-log transformed Greenwood band — stays inside ``[0, 1]`` at small G.

    The plain Greenwood band routinely leaves the unit interval at G = 15, which
    would make the median interval unreadable at exactly the sample size this
    study has.
    """
    if surv <= 0.0 or surv >= 1.0 or var_sum <= 0.0:
        return surv, surv
    se = math.sqrt(var_sum) / abs(math.log(surv))
    lo = surv ** math.exp(z * se)
    hi = surv ** math.exp(-z * se)
    return float(min(lo, hi)), float(max(lo, hi))


def kaplan_meier(
    times: np.ndarray, events: np.ndarray, alpha: float = 0.05
) -> SurvivalCurve:
    """Kaplan-Meier with Greenwood log-log confidence bands.

    ``b*`` is right-censored survival data on a six-point grid: an origin that
    never crosses tau is censored at 6, not missing. Reporting a bare mean over
    the crossers would condition on the event and bias the recommended cadence
    downward — which is the number the abstract carries.
    """
    t = np.asarray(times, dtype=np.float64)
    e = np.asarray(events, dtype=bool)
    z = _normal_quantile(1 - alpha / 2)

    surv, var_sum = 1.0, 0.0
    out_t, out_s, out_lo, out_hi = [], [], [], []
    for time in np.unique(t):
        at_risk = int(np.sum(t >= time))
        died = int(np.sum((t == time) & e))
        if at_risk > 0 and died > 0:
            surv *= 1.0 - died / at_risk
            if at_risk > died:
                var_sum += died / (at_risk * (at_risk - died))
        lo, hi = _loglog_band(surv, var_sum, z)
        out_t.append(time)
        out_s.append(surv)
        out_lo.append(lo)
        out_hi.append(hi)

    return SurvivalCurve(
        times=np.array(out_t),
        survival=np.array(out_s),
        lower=np.array(out_lo),
        upper=np.array(out_hi),
        n_events=int(e.sum()),
        n_censored=int(len(e) - e.sum()),
    )


def logrank(
    times_a: np.ndarray, events_a: np.ndarray,
    times_b: np.ndarray, events_b: np.ndarray,
) -> tuple[float, float]:
    """Two-sample log-rank test — H3's "larger K decays faster" (`D41`).

    Returns:
        ``(chi-square statistic on 1 df, two-sided p)``.
    """
    t = np.concatenate([times_a, times_b]).astype(np.float64)
    e = np.concatenate([events_a, events_b]).astype(bool)
    g = np.concatenate([np.zeros(len(times_a)), np.ones(len(times_b))]).astype(bool)

    observed = expected = variance = 0.0
    for time in np.unique(t[e]):
        n_risk = int(np.sum(t >= time))
        n_risk_b = int(np.sum((t >= time) & g))
        d = int(np.sum((t == time) & e))
        d_b = int(np.sum((t == time) & e & g))
        if n_risk < 2 or d == 0:
            continue
        share = n_risk_b / n_risk
        observed += d_b
        expected += d * share
        variance += d * share * (1 - share) * (n_risk - d) / (n_risk - 1)
    if variance <= 0:
        return float("nan"), float("nan")
    chi2 = (observed - expected) ** 2 / variance
    return float(chi2), float(math.erfc(math.sqrt(chi2 / 2.0)))


# -- Diebold-Mariano and Clark-West (`D29`, `D34`) ---------------------------


def _rectangular_lrv(d: np.ndarray, h: int) -> float:
    """``[gamma_0 + 2 sum_{k=1}^{h-1} gamma_k] / T`` — the variance of ``d_bar``.

    Rectangular, not Bartlett (`D34`). Under the DM null, h-step optimal
    forecast errors are MA(h-1), so all autocovariances to lag ``h-1`` are
    genuinely nonzero and equally real. Bartlett weights shrink the lag-22 term
    by about 92%, understating the long-run variance and producing exactly the
    over-optimistic p-values this estimator exists to prevent. ``statsmodels``'
    ``cov_hac`` is Bartlett by default, so a literal reading of "Newey-West"
    fails validation against R's ``forecast::dm.test``.
    """
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * float(dm[k:] @ dm[:-k]) / t
    return total / t


def _bartlett_lrv(d: np.ndarray, h: int) -> float:
    """Only ever the fallback, and its use is reported (root §9.2)."""
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * (1.0 - k / h) * float(dm[k:] @ dm[:-k]) / t
    return total / t


@dataclass(frozen=True, slots=True)
class TestResult:
    """One forecast-comparison test, carrying everything needed to redo it."""

    name: str
    statistic: float
    p_value: float
    T: int
    h: int
    one_sided: bool
    fallback_fired: bool

    def __str__(self) -> str:
        tail = "  [Bartlett fallback fired]" if self.fallback_fired else ""
        side = "one-sided" if self.one_sided else "two-sided"
        return (
            f"{self.name}: S*={self.statistic:+.4f}  p={self.p_value:.4g} "
            f"({side})  T={self.T}  h={self.h}{tail}"
        )


def _upper_tail(stat: float, df: int) -> float:
    """``P(T_df > stat)`` — Student-t, falling back to the normal without scipy."""
    try:
        from scipy import stats as _stats  # root §16's named stats boundary

        return float(_stats.t.sf(stat, df=df))
    except ImportError:  # pragma: no cover - scipy ships with the Kaggle image
        return 0.5 * math.erfc(stat / math.sqrt(2.0))


def _hln_and_p(d: np.ndarray, h: int, name: str, one_sided: bool) -> TestResult:
    """Harvey-Leybourne-Newbold correction, referred to ``t(T-1)``.

    ``S* = S sqrt[(T + 1 - 2h + h(h-1)/T) / T]``, compared against Student-t
    with ``T-1`` degrees of freedom — **not** the standard normal. The factor is
    asserted positive before use: at ``h = 24`` it is exactly 0 at ``T = 24`` and
    0.047 at ``T = 30``, precisely the T a non-overlapping 30-day block would
    produce, so a silent negative would yield a complex statistic reported as a
    real one.
    """
    d = np.asarray(d, dtype=np.float64)
    t = len(d)
    if t < 2:
        raise ValueError(f"{name}: T={t} is too small for a loss differential")
    factor = (t + 1 - 2 * h + h * (h - 1) / t) / t
    if factor <= 0:
        raise ValueError(
            f"{name}: the HLN factor is {factor:.4f} <= 0 at T={t}, h={h}. "
            f"Root §9.2 refuses to report where it fails; state T instead."
        )

    variance = _rectangular_lrv(d, h)
    fallback = False
    if variance <= 0:
        # The rectangular estimator is not guaranteed positive in finite
        # samples. Root §9.2: fall back to Bartlett and *report that it fired*.
        variance = _bartlett_lrv(d, h)
        fallback = True
        if variance <= 0:
            raise ValueError(f"{name}: no positive long-run variance at T={t}")

    stat = float(d.mean() / math.sqrt(variance) * math.sqrt(factor))
    upper = _upper_tail(abs(stat), t - 1)
    if one_sided:
        p = upper if stat >= 0 else 1.0 - upper
    else:
        p = 2.0 * upper
    return TestResult(name, stat, float(min(p, 1.0)), t, h, one_sided, fallback)


def dm_test(
    loss_a: np.ndarray, loss_b: np.ndarray, h: int, name: str = "DM"
) -> TestResult:
    """Diebold-Mariano for a **non-nested** pair — iTransformer vs DLinear etc.

    Do not use this on K=1 vs K=8, on anything vs Naive-RW, or on Ridge-K1 vs
    Ridge-K8: those pairs are nested, and there the statistic is not
    asymptotically ``N(0,1)`` (Clark & McCracken 2001; McCracken 2007). Use
    :func:`clark_west_test`.
    """
    return _hln_and_p(
        np.asarray(loss_a) - np.asarray(loss_b), h, name, one_sided=False
    )


def clark_west_test(
    y: np.ndarray,
    pred_small: np.ndarray,
    pred_large: np.ndarray,
    h: int,
    name: str = "Clark-West",
) -> TestResult:
    """Clark-West (2007) for a **nested** pair — the comparisons that carry the paper.

    ``f_t = (y - y_small)^2 - (y - y_large)^2 + (y_small - y_large)^2``

    The third term is the adjustment. Under the null of equal population
    predictive ability the larger model's extra estimation noise makes it look
    worse, so the unadjusted differential has a mean shifted away from zero and
    standard DM is systematically undersized **against the alternative this
    study exists to establish**. The Stage 5 gate turns a title decision on this
    statistic, which is why the choice is not left to the caller.

    One-sided by construction: the alternative is that the larger model helps.
    """
    y = np.asarray(y, dtype=np.float64)
    s = np.asarray(pred_small, dtype=np.float64)
    lg = np.asarray(pred_large, dtype=np.float64)
    f = np.square(y - s) - np.square(y - lg) + np.square(s - lg)
    return _hln_and_p(f, h, name, one_sided=True)


def per_origin_loss(frame: pl.DataFrame) -> pl.DataFrame:
    """Mean squared error per forecast origin — the ``d_t`` series DM consumes.

    Root §9.2 pins the DM sample: **per (origin, block)**, on the overlapping
    hourly loss differential, ``T ~ 720``, ``h = 24``, truncation lag 23. Block
    level statistics are combined across cells by a stated method and **never**
    by concatenating ``d_t`` across origins: the model changes at each origin, so
    the DM null has no interpretation across that boundary.
    """
    return (
        frame.with_columns((pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"))
        .group_by(["block", "timestamp"])
        .agg(pl.col("_se").mean().alias("loss"))
        .sort(["block", "timestamp"])
    )


# -- RQ2's core regression: beta1 with origin FE and a wild cluster bootstrap -


@dataclass(frozen=True, slots=True)
class Beta1Result:
    """``A(i,b) = alpha_i + beta1 b + eps`` with clustered inference (`D06`, `D42`)."""

    beta1: float
    t_statistic: float
    cluster_se: float
    p_rademacher: float
    p_webb: float
    n_clusters: int
    n_observations: int
    within_slopes: np.ndarray
    B: int

    @property
    def headline_p(self) -> float:
        """The more conservative of the two weight schemes, as root §9.2 requires."""
        return max(self.p_rademacher, self.p_webb)

    def __str__(self) -> str:
        return (
            f"beta1 = {self.beta1:+.6f}   t = {self.t_statistic:+.3f}   "
            f"G = {self.n_clusters}   N = {self.n_observations}\n"
            f"WCR one-sided p (H1: beta1 < 0): Rademacher {self.p_rademacher:.4f}, "
            f"Webb {self.p_webb:.4f}  ->  headline {self.headline_p:.4f}\n"
            f"Effective independence is bounded near 4 by the training-window "
            f"overlap (root §8.1), well below G = {self.n_clusters}."
        )


def _weights(kind: str, shape: tuple[int, int], rng: np.random.Generator) -> np.ndarray:
    if kind == "rademacher":
        return rng.choice(np.array([-1.0, 1.0]), size=shape)
    if kind == "webb":
        # Webb's 6-point distribution. At G = 15 Rademacher already admits
        # 2^15 = 32,768 distinct draws, a minimum two-sided p of about 6e-5, so
        # the original small-G justification for preferring Webb no longer
        # binds — both are reported and the more conservative is the headline.
        atoms = np.array([
            -math.sqrt(1.5), -1.0, -math.sqrt(0.5),
            math.sqrt(0.5), 1.0, math.sqrt(1.5),
        ])
        return rng.choice(atoms, size=shape)
    raise ValueError(f"unknown weight scheme {kind!r}")


def _balanced_matrix(panel: pl.DataFrame, value: str) -> tuple[np.ndarray, np.ndarray]:
    """``(G x B)`` outcome matrix and the block axis, or a loud failure.

    Built by hand rather than with ``pivot`` so the code does not depend on which
    polars major version the Kaggle image happens to ship.
    """
    origins = sorted(set(panel.get_column("origin").to_list()))
    blocks = sorted(set(int(b) for b in panel.get_column("block").to_list()))
    index = {(o, b): i for i, (o, b) in enumerate([(o, b) for o in origins for b in blocks])}

    out = np.full(len(index), np.nan)
    for origin, block, val in zip(
        panel.get_column("origin").to_list(),
        panel.get_column("block").to_list(),
        panel.get_column(value).to_list(),
    ):
        out[index[(str(origin), int(block))]] = float(val)

    matrix = out.reshape(len(origins), len(blocks))
    if np.isnan(matrix).any():
        raise ValueError(
            "unbalanced panel: beta1's reduction to the mean of within-slopes "
            "holds only when every origin carries every block"
        )
    return matrix, np.array(blocks, dtype=np.float64)


def panel_beta1(
    panel: pl.DataFrame,
    value: str = "A",
    B: int = 99_999,
    seed: int = 42,
) -> Beta1Result:
    """Fit ``A(i,b) = alpha_i + beta1 b + eps`` and test ``H1: beta1 < 0``.

    Args:
        panel: Long frame with ``origin``, ``block`` and ``value``. Must be
            balanced — every origin carries the same block set.
        value: Dependent variable column.
        B: Bootstrap replications. 99,999 as pre-registered.
        seed: Bootstrap seed, recorded so the p-value is regenerable (root §12).

    **Without ``alpha_i``, beta1 absorbs origin-level difficulty** (`D06`). With
    origin fixed effects and a balanced panel, ``beta1`` reduces algebraically to
    the simple mean of the origin-specific within-slopes, so inference on the
    paper's core claim is a one-sample test on **G** numbers. Citing "15 x 6 = 90
    observations" invites the reader to infer power that does not exist; both
    counts are reported, and effective independence is bounded near 4 by the
    training-window overlap (root §8.1).

    The bootstrap is **restricted** (WCR — the null imposed when generating
    samples), bootstraps the **cluster-robust t-statistic** rather than beta-hat,
    and is **one-sided at alpha = 0.05 declared in advance**. WCU is severely
    size-distorted at small G (MacKinnon, Nielsen & Webb 2023), and the
    asymptotic refinement comes from bootstrapping *t* (Cameron, Gelbach &
    Miller 2008). A side chosen after seeing the sign is not pre-registered.
    """
    a, x = _balanced_matrix(panel, value)
    g, n_blocks = a.shape
    xd = x - x.mean()
    sxx = float(xd @ xd)

    within = a - a.mean(axis=1, keepdims=True)
    beta = float((within * xd).sum() / (g * sxx))
    resid = within - beta * xd
    score = resid @ xd
    variance = float((score @ score) / (g * sxx) ** 2)
    se = math.sqrt(variance) if variance > 0 else float("nan")
    t_obs = beta / se if se == se and se > 0 else float("nan")

    # Restricted residuals: with beta1 = 0 imposed the fitted value is the origin
    # mean, so u_tilde is exactly the within-origin demeaned outcome. Because
    # each row of u_tilde already sums to zero, the bootstrap origin means are
    # unchanged and the whole replication collapses to s = u_tilde @ xd.
    s = within @ xd

    def _p(kind: str) -> float:
        rng = np.random.default_rng(seed)
        weights = _weights(kind, (B, g), rng)
        beta_star = (weights @ s) / (g * sxx)
        score_star = weights * s[None, :] - beta_star[:, None] * sxx
        var_star = np.square(score_star).sum(axis=1) / (g * sxx) ** 2
        ok = var_star > 0
        t_star = beta_star[ok] / np.sqrt(var_star[ok])
        # (1 + count) / (1 + B), not count / B (Davison & Hinkley 1997): the
        # observed statistic is one of its own reference distribution, and the
        # naive form returns a literal p = 0, which is not a probability any
        # finite bootstrap can support. At B = 99,999 the floor it reports is
        # 1e-5, and at G = 15 Rademacher's own granularity bounds it at ~3e-5
        # anyway — so the floor is honest rather than conservative padding.
        below = int(np.sum(t_star <= t_obs))  # H1: beta1 < 0, left tail
        return (1.0 + below) / (1.0 + int(ok.sum()))

    return Beta1Result(
        beta1=beta,
        t_statistic=t_obs,
        cluster_se=se,
        p_rademacher=_p("rademacher"),
        p_webb=_p("webb"),
        n_clusters=g,
        n_observations=g * n_blocks,
        within_slopes=(within * xd).sum(axis=1) / sxx,
        B=B,
    )


@dataclass(frozen=True, slots=True)
class EquivalenceResult:
    """TOST verdict on a rung `D49` pre-registers as flat."""

    mean_delta: float
    margin: float
    p_lower: float
    p_upper: float
    n: int

    @property
    def equivalent(self) -> bool:
        return max(self.p_lower, self.p_upper) < 0.05

    def __str__(self) -> str:
        verdict = "EQUIVALENT (flat)" if self.equivalent else "NOT shown equivalent"
        return (
            f"TOST: mean delta = {self.mean_delta:+.6f}, margin = +/-{self.margin:.6f}, "
            f"p = ({self.p_lower:.4f}, {self.p_upper:.4f}), G = {self.n}  ->  {verdict}"
        )


def tost_equivalence(
    deltas: np.ndarray, margin: float, alpha: float = 0.05
) -> EquivalenceResult:
    """Two one-sided tests — RQ1's pre-registered equivalence check (`D49`).

    RQ1's claim that the 8->12 rung is flat is an assertion of **no effect**, and
    a non-significant ΔMSE is a failure to reject, not evidence of equivalence.
    The margin is fixed in advance at ``0.25 x ΔMSE(4->8)``: choosing it after
    seeing the rung is the same p-hacking root §3 forbids for tau.

    Args:
        deltas: One within-origin ΔMSE per cluster. The inferential unit is the
            origin, never the (origin, block) cell.
        margin: ``Δ_eq``, positive.
    """
    d = np.asarray(deltas, dtype=np.float64)
    n = len(d)
    if n < 2:
        raise ValueError("TOST needs at least two clusters")
    se = float(np.std(d, ddof=1) / math.sqrt(n))
    if se <= 0:
        raise ValueError("zero dispersion across clusters; TOST is undefined")
    mean = float(d.mean())
    return EquivalenceResult(
        mean_delta=mean,
        margin=abs(margin),
        p_lower=_upper_tail((mean + abs(margin)) / se, n - 1),   # H0: mu <= -margin
        p_upper=_upper_tail(-(mean - abs(margin)) / se, n - 1),  # H0: mu >= +margin
        n=n,
    )


def j_test(
    y: np.ndarray, x_a: np.ndarray, x_b: np.ndarray, groups: np.ndarray
) -> tuple[float, float]:
    """Davidson-MacKinnon J-test of model A against model B (`D32`).

    RQ1 is a **non-nested** comparison: "benefit tracks K" and "benefit tracks
    K_eff" are two different regressors for the same outcome, and neither nests
    the other. Fitting both and comparing R-squared answers nothing; the J-test
    augments A with B's fitted values and asks whether they still carry
    information.

    All three inputs are within-transformed by ``groups`` first — the (origin x
    block) fixed effects of §9.1's specification — so the comparison is
    identified from within-cell variation across rungs, which is the only
    variation that distinguishes the two theories.

    Returns:
        ``(t statistic on B's fitted values, two-sided p)``. A significant t
        means A alone is inadequate. Run it both ways: if both reject, neither
        explanation is sufficient; if neither does, the data cannot separate
        them, which at ``corr(K, K_eff) ~ 0.97`` is the outcome to expect and to
        report plainly.
    """
    def _demean(v: np.ndarray) -> np.ndarray:
        v = np.asarray(v, dtype=np.float64)
        out = v.astype(np.float64).copy()
        for g in np.unique(groups):
            mask = groups == g
            out[mask] = v[mask] - v[mask].mean()
        return out

    yd, ad, bd = _demean(y), _demean(x_a), _demean(x_b)
    fitted_b = bd * (float(bd @ yd) / float(bd @ bd))

    design = np.column_stack([ad, fitted_b])
    coef, *_ = np.linalg.lstsq(design, yd, rcond=None)
    resid = yd - design @ coef
    dof = len(yd) - design.shape[1] - len(np.unique(groups))
    if dof <= 0:
        return float("nan"), float("nan")
    sigma2 = float(resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(design.T @ design)
    se = math.sqrt(max(cov[1, 1], 0.0))
    if se <= 0:
        return float("nan"), float("nan")
    t = float(coef[1] / se)
    return t, 2.0 * _upper_tail(abs(t), dof)


def minimum_detectable_beta1(
    within_slopes: np.ndarray, alpha: float = 0.05, power: float = 0.80
) -> float:
    """The MDE root §13.2 calls the most damaging omission on its list.

    Every design choice in this study implies someone reasoned about precision,
    and no number was written down. If the MDE exceeds the plausible magnitude
    of ``A``, RQ2 must be repositioned as descriptive **before** the grid runs —
    otherwise a non-significant beta1 is indistinguishable from a design that
    could never have detected decay.

    Computed from the between-origin dispersion of the within-slope, which is
    exactly what the Stage 5 pilot estimates. Returned negative, since the
    alternative is one-sided and downward.
    """
    g = len(within_slopes)
    if g < 2:
        return float("nan")
    se = float(np.std(within_slopes, ddof=1) / math.sqrt(g))
    return -(_normal_quantile(1 - alpha) + _normal_quantile(power)) * se


# -- drivers for the paper's tables (`D62a`) ---------------------------------
#
# Everything below reads what the grid already wrote. None of it trains and none
# of it needs a GPU. Each existed as a function nobody called, or as a number
# that lived only in `CLAUDE.md` prose and therefore fell outside §12's
# regenerability contract.


def directional_accuracy_table(run_ids: list[str], roots: list[Path]) -> pl.DataFrame:
    """DA at all three horizons for many runs (`D21`) -- an input to Table 4.

    :func:`directional_accuracy` has existed and been tested since the model
    plane was built and **was never called**: no DA figure appears in
    ``paper_numbers.json`` or anywhere in the session log. This is the driver.

    The three variants do not share a testing regime and the table keeps that
    visible rather than tidying it away. ``da_h1`` carries a Pesaran-Timmermann
    p-value on hourly spacing. ``da_hH`` and ``da_cum`` carry one **only** on the
    non-overlapping sample; their ``*_overlapping`` twins are descriptive and have
    no p-value at all, because on hourly spacing those targets overlap by 23 of
    24 hours, giving lag-1 autocorrelation near 23/24 -- PT's variance is then far
    too small and the test over-rejects badly. The resulting power loss is
    **stated** as ``n_non_overlapping``, never recovered by using the invalid
    sample.

    Args:
        run_ids: Runs to measure.
        roots: Artifact roots, working directory first.

    Returns:
        One row per run: its identity, all eight DA figures, and both sample sizes.
    """
    rows: list[dict[str, float | str | int]] = []
    for run_id in run_ids:
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        da = directional_accuracy(load_predictions(run_id, roots))
        rows.append(
            {
                "run_id": run_id,
                "model": str(parts["model"]),
                "origin": str(meta["origin"]),
                "origin_index": int(parts["origin_index"]),
                "k": int(parts["k"]),
                "pred_len": int(parts["pred_len"]),
                "seed": int(parts["seed"]),
                "da_h1": da.da_h1,
                "p_h1": da.p_h1,
                "da_hH": da.da_hH,
                "p_hH": da.p_hH,
                "da_hH_overlapping": da.da_hH_overlapping,
                "da_cum": da.da_cum,
                "p_cum": da.p_cum,
                "da_cum_overlapping": da.da_cum_overlapping,
                "n_h1": da.n_h1,
                "n_non_overlapping": da.n_non_overlapping,
            }
        )
    return pl.DataFrame(rows)


def raw_scale_table(seed_avg: pl.DataFrame) -> pl.DataFrame:
    """Add RMSE in raw log-return units -- root §9.1's second metric scale.

    "RMSE 0.0043 on hourly log-returns" tells a reader far more than "MSE 0.187
    on normalized data". Both scales are reported and ``sigma_g`` is stated, which
    is what lets the two reconcile. :func:`raw_rmse` has existed all along and,
    like :func:`directional_accuracy`, was never called.
    """
    return seed_avg.with_columns(
        (pl.col("mse").sqrt() * pl.col("sigma_g")).alias("rmse_raw")
    )


def falsification_relmse(seed_avg: pl.DataFrame) -> pl.DataFrame:
    """``aged - fresh`` on **RelMSE**, per (origin, block) -- `D60i`.

    Root §8.1's falsification arm is the only design in the study that identifies
    decay directly, and the number reported for it was a units artefact. The
    notebook printed ``mean(aged - fresh) = -0.053341`` as a raw scaler-space MSE
    difference. The two arms are fitted at origins 90 days apart and therefore
    carry **different sigma_g** -- 0.009151 against 0.007297 at origin 1 -- so
    that difference compares numbers in different units. The matching naive
    baselines differ by -0.053196, i.e. about **99.7% of it is scaler drift**, and
    the sign reads backwards, appearing to say the aged model beat the fresh one.

    Root §9.1 already forbade the comparison by requiring RelMSE "to control for
    period difficulty". The arm was simply never brought under the rule, and the
    corrected figure lived only in prose. **The raw-MSE figure must not appear in
    the manuscript**, and the general rule this defect bought is that any
    cross-origin model comparison is on RelMSE or ``R2_oos``, never on
    scaler-space MSE.

    Returns:
        ``origin_index, origin, block, rel_aged, rel_fresh, gap_rel_mse`` over the
        cells the arm covers -- blocks 4-6 at each origin, which are the same
        calendar hours the aged model was scored on.
    """
    sel = seed_avg.filter(pl.col("pred_len") == 24)
    aged = (
        sel.filter((pl.col("model") == "itr") & (pl.col("k") == 8))
        .select(["origin_index", "origin", "block", "rel_mse"])
        .rename({"rel_mse": "rel_aged"})
    )
    fresh = (
        sel.filter(pl.col("model") == "itrf")
        .select(["origin_index", "block", "rel_mse"])
        .rename({"rel_mse": "rel_fresh"})
    )
    return (
        aged.join(fresh, on=["origin_index", "block"], how="inner")
        .with_columns((pl.col("rel_aged") - pl.col("rel_fresh")).alias("gap_rel_mse"))
        .sort(["origin_index", "block"])
    )


def beta1_with_coverage(
    panel: pl.DataFrame,
    min_coverage: float = 0.9,
    B: int = 99_999,
    seed: int = 42,
) -> tuple[Beta1Result, Beta1Result | None]:
    """beta1 on the full panel, and on well-covered blocks only (`D45`).

    Test-window survival is conditioned on **future** gaps -- whether a forecast
    issued at *s* is evaluated depends on whether the next 120 hours contain an
    outage, information unavailable at *s* -- and Binance outages cluster on
    stress. So within an origin the surviving sample composition trends, the
    dropped targets are systematically the high-volatility ones, and beta1 would
    absorb that trend as though it were decay. Root §9.2 requires either a
    coverage covariate or a re-estimate on well-covered blocks; this is the
    second.

    Restricting usually leaves an **unbalanced** panel, and
    :func:`_balanced_matrix` refuses one by design: beta1's reduction to the mean
    of within-slopes holds only when every origin carries every block. ``None``
    comes back in that case, and that is the honest report -- the check could not
    be run, not that it passed. Only a restriction that removes whole origins
    leaves something estimable.

    Args:
        panel: :func:`amplification`'s output, carrying ``n_large``.
        min_coverage: Surviving windows as a fraction of :data:`BLOCK_HOURS`.
        B: Bootstrap draws, passed through to :func:`panel_beta1`.
        seed: Bootstrap seed, passed through.

    Returns:
        ``(full, restricted_or_None)``.
    """
    full = panel_beta1(panel, B=B, seed=seed)
    restricted = panel.filter((pl.col("n_large") / float(BLOCK_HOURS)) >= min_coverage)
    try:
        return full, panel_beta1(restricted, B=B, seed=seed)
    except ValueError:
        # Unbalanced after the restriction. Loosening the estimator to produce a
        # number here would answer a different question than the one asked.
        return full, None


In [ ]:
# ═══ baselines.py ════════
"""Ridge, DLinear and PatchTST — the comparators root §7 calls mandatory (`D56`).

Until the 534-run grid finished on 2026-08-08 this module did not exist, and
neither did any other baseline class. Root §7 says DLinear and PatchTST are "not
optional", root §10.2 budgets 255 baseline runs, and
:func:`itransformer_btc.metrics.dm_nonnested` has been sitting ready for input
that was never produced — so §10.2's 789 was never executable and Table 6 had no
inputs. The consequence is not bookkeeping: with Naive-RW the only comparator,
"iTransformer has no edge" rests on one contrast, and §6.2/`D38` says the
hyperparameters were adopted unchanged and never tuned. A referee reads that null
as an immature configuration rather than a finding. These three models are the
minimum that answers the question deciding what the negative result means:
**did iTransformer fail, or did the whole LTSF class fail here?**

What is built, and what each one is for:

=========  ==========  =====================================================
Model      K           Question it answers
=========  ==========  =====================================================
Ridge      1, 4, 8, 12 `D17` — is a transformer needed at all? Linear and
                       genuinely multivariate, so it separates *does the
                       information help* from *does attention help*
DLinear    8           root §7 — the first thing an LTSF-literate reviewer
                       looks for. Linear, decomposition-based, ~4.7k weights
PatchTST   8           root §7 — SOTA and channel-independent, the other side
                       of the debate §13.1 makes a Related Work pillar
=========  ==========  =====================================================

**What "K = 8" means for a channel-independent model, stated because it is not
what it means elsewhere in this study.** DLinear and PatchTST forecast each
channel from that channel's own history; that is the architecture's claim, not a
shortcoming of this implementation. They are therefore trained with their
published **all-channel** objective and their weights are **shared across
channels**, which is the only route by which the other seven variates reach the
target's forecast at all. Trained on the target channel alone they would be K=1
wearing a K=8 label — the exact collapse `D40` was written to prevent — and the
paper's central architectural comparison would quietly become
univariate-versus-multivariate again. Both facts are recorded as fields in every
``meta/*.json`` (``loss_channels``, ``channel_independent``) so no reader has to
infer them. Ridge carries no such caveat: every one of its ``L x K`` inputs
enters the target's forecast, so its K label means what K means in §5.2.

**Capacity is held fixed rather than tuned.** PatchTST takes iTransformer's
``d_model``, ``d_ff``, ``e_layers``, ``n_heads`` and ``dropout`` and reuses
:class:`itransformer_btc.model.EncoderLayer` itself, so the two models differ in
**what a token is** — a patch of one variate against the whole lookback of each
variate — and in nothing else. That is the cleanest available form of the
contrast, and it extends §6.2/`D38`'s no-tuning posture to the baselines instead
of quietly exempting them.

**Scale space is identical to the ladder's** (root §6.3, §11). Every model reads
the same ``StandardScaler`` output fitted on the same 21-month sub-block. What
differs is internal normalisation, and it differs the way the published models
do: PatchTST normalises per window (RevIN — the same operation ``use_norm=True``
applies), DLinear and ridge do not, which is precisely the case §6.3 says the
outer scaler exists to serve.

**Not built here, and not silently dropped.** ARIMA, LSTM, naive-persist and
seasonal-naive are deferred from this minimal set. Naive-RW needs no run at all —
:func:`itransformer_btc.metrics.block_metrics` computes it from ``naive_rw_z`` on
exactly the rows the model was scored on. If the other four are eventually cut,
root §7 must be edited with a written reason rather than left standing over
models nobody built.
"""

from __future__ import annotations

import time
import warnings
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch import Tensor, nn



class BaselineModule(nn.Module):
    """Shared plumbing for the two channel-independent baselines.

    ``forward`` returns all N channels because that is what the published
    objective supervises; ``forecast_target`` is the projection root §10.4's
    prediction file actually holds. Channel ``TARGET_INDEX`` is ``r`` at every
    rung — the ladder pins it there, see
    :data:`itransformer_btc.features.VARIATE_ORDER` — so this is one constant
    rather than a lookup.
    """

    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, H)`` on the target channel."""
        return self(x)[:, :, TARGET_INDEX]


# -- ridge -------------------------------------------------------------------


#: Root §11: ridge alpha is selected on the validation sub-block, and with ARIMA
#: outside the minimal set it is the **only** hyperparameter selected anywhere in
#: this study (`D38`). The solve is unnormalised — ``(X'X + a I) W = X'Y`` — so
#: the scale that matters is ``diag(X'X) ~ n``, about 1.4e4 at these origins; the
#: grid spans five orders below it and two above.
RIDGE_ALPHAS: tuple[float, ...] = (1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6)


@dataclass(frozen=True, slots=True)
class RidgeConfig:
    """L2-regularised linear map from the flattened window to the H-step target.

    `D17`: K=1 iTransformer controls for *architecture* — it answers "does
    cross-variate attention help?" It does not answer "is a transformer needed at
    all?" Ridge on the same K features separates *does the information help* from
    *does attention help*, at seconds per run, and closes a question a reviewer
    asks otherwise.

    ``k`` is a field here and nowhere else among the study's configs.
    iTransformer's parameter count is identical at every rung because K changes
    the token count and not a weight shape; ridge's weight matrix is
    ``(L*K, H)``, so K is part of its geometry and belongs in ``meta['config']``.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    k: int = 8
    alphas: tuple[float, ...] = RIDGE_ALPHAS
    #: Chosen by :meth:`fit` on the validation sub-block. ``None`` in an unfitted
    #: config and never in a written ``meta/*.json`` — root §12 cannot regenerate
    #: a number whose only free parameter went unrecorded.
    alpha: float | None = None

    def build(self) -> "RidgeForecaster":
        return RidgeForecaster(self)

    def loss_target(self) -> str:
        """``"target"``: ridge predicts the target channel and nothing else."""
        return "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["RidgeForecaster", "RidgeConfig", TrainOutcome]:
        """Solve the normal equations once, then pick alpha on validation.

        The Gram matrix and the right-hand side are built **once** and reused for
        every alpha, so the sweep costs one solve per candidate rather than a
        refit. In ``float64``: at ``L*K = 1152`` the design is conditioned badly
        enough that a ``float32`` Gram would make the smallest alphas report
        noise, and showing what an essentially unregularised linear map does is
        the whole reason the small alphas are in the grid.

        The intercept is fitted by centring and is **not** penalised. Shrinking
        it toward zero would shrink the forecast toward zero *in scaler space*,
        which is ``r = mu_g`` — the constant-drift model `D31` spent a section
        removing from the Naive-RW baseline.
        """
        device = device or pick_device()
        # A solve consumes no RNG. Seeded anyway, so a ridge run and an
        # iTransformer run of the same cell are reproducible under one rule
        # (root §16) rather than two.
        set_seed(spec.seed)
        started = time.perf_counter()

        model = self.build().to(device)
        x_tr = self._design(tensors.train.x, device)
        y_tr = torch.from_numpy(tensors.train.y).to(device).double()
        x_va = self._design(tensors.val.x, device)
        y_va = torch.from_numpy(tensors.val.y).to(device).double()

        x_mean, y_mean = x_tr.mean(0), y_tr.mean(0)
        x_tr -= x_mean
        y_tr -= y_mean
        gram = x_tr.T @ x_tr
        rhs = x_tr.T @ y_tr
        eye = torch.eye(gram.shape[0], dtype=gram.dtype, device=gram.device)

        best: tuple[float, float, Tensor] | None = None
        for alpha in self.alphas:
            weight = torch.linalg.solve(gram + alpha * eye, rhs)
            residual = (x_va - x_mean) @ weight + y_mean - y_va
            val_mse = float(residual.pow(2).mean())
            if best is None or val_mse < best[0]:
                best = (val_mse, float(alpha), weight)
        if best is None:
            raise ValueError("no ridge alpha to select; `alphas` is empty")

        val_mse, alpha, weight = best
        if len(self.alphas) > 1 and alpha in (self.alphas[0], self.alphas[-1]):
            # Not a failure. An alpha pinned at the top of the grid says the
            # least-squares fit is worthless and the best linear predictor is the
            # training mean, which is a finding. It is warned about because a
            # boundary selection is also what an unbracketed grid looks like, and
            # the two are indistinguishable from the number alone.
            warnings.warn(
                f"{spec.run_id}: ridge alpha {alpha:g} sits at the edge of "
                f"{self.alphas}; the grid may not bracket the optimum",
                stacklevel=2,
            )

        with torch.no_grad():
            model.weight.copy_(weight.to(torch.float32))
            model.bias.copy_((y_mean - x_mean @ weight).to(torch.float32))
        train_mse = float((x_tr @ weight - y_tr).pow(2).mean())

        return (
            model,
            replace(self, alpha=alpha),
            TrainOutcome(
                run_id=spec.run_id,
                # A solve, not a loop. Zero is the honest number, and it is what
                # tells a reader of Table 3 why this row has no epochs-to-stop.
                epochs_run=0,
                best_val_mse=val_mse,
                train_loss=train_mse,
                wall_time_s=time.perf_counter() - started,
                n_parameters=model.n_parameters(),
                device=str(device),
            ),
        )

    @staticmethod
    def _design(x: np.ndarray, device: torch.device) -> Tensor:
        """``(n, L, K) -> (n, L*K)`` in float64, on the device.

        Row-major, so a column is one (lag, variate) pair. Nothing depends on
        which ordering it is, only that this and
        :meth:`RidgeForecaster.forward` agree — which they do by both being
        ``reshape``.
        """
        return torch.from_numpy(x).to(device).reshape(len(x), -1).double()


class RidgeForecaster(nn.Module):
    """``y_hat = vec(x) @ W + b``, fitted in closed form.

    ``W`` and ``b`` are **buffers**, not parameters: nothing here is trained by
    gradient descent, and registering them as parameters would put them in front
    of an optimiser that must never see them. That is why :meth:`n_parameters`
    counts them explicitly — the usual sum over ``self.parameters()`` would
    report zero, and root §12 would record a model with no coefficients.
    """

    def __init__(self, cfg: RidgeConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.register_buffer(
            "weight",
            torch.zeros(cfg.seq_len * cfg.k, cfg.pred_len, dtype=torch.float32),
        )
        self.register_buffer("bias", torch.zeros(cfg.pred_len, dtype=torch.float32))

    def forward(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)``."""
        return x.reshape(len(x), -1) @ self.weight + self.bias

    def forecast_target(self, x: Tensor) -> Tensor:
        return self(x)

    def n_parameters(self) -> int:
        return self.weight.numel() + self.bias.numel()


# -- DLinear -----------------------------------------------------------------


@dataclass(frozen=True, slots=True)
class DLinearConfig:
    """Trend-seasonal decomposition plus two linear maps (Zeng et al., 2023).

    Root §7 calls it mandatory for a reason worth stating: a missing DLinear is
    the first thing a reviewer familiar with the LTSF literature flags, because
    it is the model that showed a linear map beating several transformers on the
    standard benchmarks. Against a null result it does more than that — if ~4.7k
    weights also fail here, the failure belongs to the problem and not to
    attention.

    Weights are **shared across channels**, never per-channel. The published
    implementation offers both; only the shared form lets the all-channel
    objective carry information from the other seven variates into the target's
    forecast, which is what makes the K=8 label true (see this module's header).
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    #: Odd, so the decomposition's padding is symmetric. 25 is the published
    #: default and is not tuned here (`D38`, extended to the baselines).
    moving_avg: int = 25
    #: Recorded in ``meta/*.json`` rather than left to be inferred: see the
    #: header. A reader who does not know the objective cannot read
    #: ``best_val_mse``, which is an all-channel figure for this model and a
    #: target-channel one for the ladder.
    loss_channels: str = "all"
    channel_independent: bool = True

    def build(self) -> "DLinear":
        return DLinear(self)

    def loss_target(self) -> str:
        return "all" if self.loss_channels == "all" else "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["DLinear", "DLinearConfig", TrainOutcome]:
        """Root §6.2's schedule; nothing is selected, so the config returns as given."""
        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


class SeriesDecomposition(nn.Module):
    """Moving-average trend and the residual seasonal component.

    **This is a rolling window inside a model, and root §5.3's ban is on rolling
    *features*. The distinction is not a technicality, so here is the argument.**
    The ban exists because a rolling feature computed over the full series can let
    a later bar reach an earlier feature value — the ``center=True`` leak class —
    and root §8.3's no-embargo justification rests on no feature having one. This
    average is computed at inference time from the 96 bars of the window itself,
    every one of which precedes the first forecast hour, and the padding
    replicates the window's own endpoints rather than reaching outside it. No
    test-period bar can therefore influence any training-set value, which is the
    property §8.3 actually needs, and it holds even though the average is centred
    **within** the window, as the published DLinear's is. Reproducing the
    published decomposition matters: a causal variant would be a different model,
    and the question this baseline exists to answer is about DLinear.
    """

    def __init__(self, kernel: int) -> None:
        super().__init__()
        self.kernel = kernel
        self.average = nn.AvgPool1d(kernel, stride=1, padding=0)

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        """``(B, L, N) -> (seasonal, trend)``, both ``(B, L, N)``."""
        front_pad = (self.kernel - 1) // 2
        padded = torch.cat(
            [
                x[:, :1, :].repeat(1, front_pad, 1),
                x,
                x[:, -1:, :].repeat(1, self.kernel - 1 - front_pad, 1),
            ],
            dim=1,
        )
        trend = self.average(padded.permute(0, 2, 1)).permute(0, 2, 1)
        return x - trend, trend


class DLinear(BaselineModule):
    """``(B, L, N) -> (B, H, N)``: decompose, map each part linearly, add.

    No instance normalisation, as published — which is exactly the case root §6.3
    says the outer ``StandardScaler`` exists to serve, so this model reads the
    same scaler space as every other and needs nothing of its own.
    """

    def __init__(self, cfg: DLinearConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.decomposition = SeriesDecomposition(cfg.moving_avg)
        self.seasonal = nn.Linear(cfg.seq_len, cfg.pred_len)
        self.trend = nn.Linear(cfg.seq_len, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        seasonal, trend = self.decomposition(x)
        out = self.seasonal(seasonal.permute(0, 2, 1)) + self.trend(
            trend.permute(0, 2, 1)
        )
        return out.permute(0, 2, 1)


# -- PatchTST ----------------------------------------------------------------


@dataclass(frozen=True, slots=True)
class PatchTSTConfig:
    """Patched, channel-independent transformer (Nie et al., 2023).

    The other side of the channel-independence debate root §13.1 makes a Related
    Work pillar, and the comparison `D40` says an LTSF-literate reviewer wants:
    iTransformer at K=8 against PatchTST at K=8, on the same information, the same
    windows and the same scale space.

    Capacity is iTransformer's, field for field, and the encoder block is
    literally :class:`itransformer_btc.model.EncoderLayer`. The two models
    therefore differ in **what a token is** — a patch of one variate here, one
    variate's entire lookback there — and in nothing else. Root §6.2/`D38`'s
    no-tuning rule applies unchanged.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    #: Root §7's committed geometry. ``(96 - 16) / 8 + 1 = 11`` patches, with no
    #: end-padding patch: the published option that adds one is a convenience for
    #: lookbacks the stride does not divide evenly, and 96 is not one of those.
    patch_len: int = 16
    stride: int = 8
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    #: Reversible instance normalisation, as published. It is the **same**
    #: operation ``use_norm=True`` applies in :class:`ITransformer` — per window,
    #: per channel — so the two models are normalised alike and root §6.3's
    #: cross-model scale consistency holds.
    revin: bool = True
    loss_channels: str = "all"
    channel_independent: bool = True

    @property
    def n_patches(self) -> int:
        return (self.seq_len - self.patch_len) // self.stride + 1

    def build(self) -> "PatchTST":
        return PatchTST(self)

    def loss_target(self) -> str:
        return "all" if self.loss_channels == "all" else "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["PatchTST", "PatchTSTConfig", TrainOutcome]:
        """Root §6.2's schedule; nothing is selected, so the config returns as given."""
        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


class PatchTST(BaselineModule):
    """``(B, L, N) -> (B, H, N)``, each channel processed as its own sequence."""

    def __init__(self, cfg: PatchTSTConfig) -> None:
        super().__init__()
        self.cfg = cfg
        if (cfg.seq_len - cfg.patch_len) % cfg.stride:
            raise ValueError(
                f"seq_len {cfg.seq_len} and patch_len {cfg.patch_len} leave "
                f"{(cfg.seq_len - cfg.patch_len) % cfg.stride} bars uncovered at "
                f"stride {cfg.stride}. Dropping the tail of every window would "
                f"make this model's lookback shorter than the ladder's, and the "
                f"comparison would no longer be on identical information."
            )
        # EncoderLayer reads d_model, d_ff, n_heads, dropout and
        # uniform_attention; its remaining fields are inert here and stay at
        # their defaults. Reusing the block rather than reimplementing it is what
        # makes "same capacity, different tokenisation" a fact and not a claim.
        block = ITransformerConfig(
            d_model=cfg.d_model,
            d_ff=cfg.d_ff,
            n_heads=cfg.n_heads,
            dropout=cfg.dropout,
        )
        self.embedding = nn.Linear(cfg.patch_len, cfg.d_model)
        self.position = nn.Parameter(torch.zeros(cfg.n_patches, cfg.d_model))
        nn.init.normal_(self.position, std=0.02)
        self.dropout = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList(EncoderLayer(block) for _ in range(cfg.e_layers))
        self.head = nn.Linear(cfg.n_patches * cfg.d_model, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        b, length, n = x.shape
        mean = std = None
        if self.cfg.revin:
            mean = x.mean(dim=1, keepdim=True)
            x = x - mean
            std = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5)
            x = x / std

        # (B, L, N) -> (B*N, L). Folding the channels into the batch **is**
        # channel independence: from here on nothing in the network sees two
        # variates at once, which is the property under test.
        series = x.permute(0, 2, 1).reshape(b * n, length)
        patches = series.unfold(1, self.cfg.patch_len, self.cfg.stride)
        h = self.dropout(self.embedding(patches) + self.position)
        for layer in self.layers:
            h = layer(h)
        out = self.head(h.reshape(b * n, -1)).reshape(b, n, self.cfg.pred_len)
        out = out.permute(0, 2, 1)

        if self.cfg.revin:
            out = out * std[:, 0, :].unsqueeze(1) + mean[:, 0, :].unsqueeze(1)
        return out


# -- the `D45` assertion -----------------------------------------------------


def assert_baseline_alignment(
    baseline_run_id: str, reference_run_id: str, roots: list[Path]
) -> None:
    """`D45` — a baseline may only be scored on its comparator's exact windows.

    Root §7: "Baselines are scored on exactly the same surviving windows." Unless
    that holds, RelMSE is a ratio across two samples rather than a ratio, and the
    two samples would differ systematically rather than randomly: test-window
    survival is conditioned on *future* gaps (root §4.3) and Binance outages
    cluster on stress, so the windows one model kept and the other dropped are
    disproportionately the high-volatility ones.

    Here the sets are equal by construction — both come from
    :func:`itransformer_btc.splits.window_starts` with the same origin, span and
    ``"origin"`` semantics — and that is exactly why the assertion is cheap, and
    why it is the only thing that would notice if it ever stopped being true.
    Root §4.3 names positional-index drift after a row drop as the
    highest-probability silent bug in this pipeline; this is its detector on the
    cross-model axis.

    Raises:
        ValueError: If the evaluated ``(block, timestamp)`` sets differ.
        FileNotFoundError: If either run is absent from ``roots``.
    """
    assert_same_windows(
        load_predictions(baseline_run_id, roots),
        load_predictions(reference_run_id, roots),
        f"{baseline_run_id} vs {reference_run_id}",
    )


In [ ]:
# ═══ runner.py ════════
"""The run manifest, resume, the budget guard, and the two-GPU launcher.

Root §10. This module is what a Kaggle notebook calls; the notebook itself
installs, discovers inputs, calls :func:`launch_workers`, and saves. Logic in a
notebook is a defect (``notebooks/CLAUDE.md``), and a run queue is logic.

**Two GPUs are two independent run *processes*, not two threads and not
``nn.DataParallel``.** Root §10.3 rejects DataParallel on cost grounds — at
batch 32 the scatter/gather transfer costs more than the split saves, and
parallelism belongs at the *run* level because the grid is many small runs
rather than one large one. Threads are rejected for a second, sharper reason:
``torch.manual_seed`` seeds **every** CUDA device, so two threads seeding
concurrently would clobber each other's generator mid-run and root §12's
reproducibility contract would be unenforceable. One process per GPU with
``CUDA_VISIBLE_DEVICES`` pinned gives each worker its own global RNG, its own
interpreter lock and crash isolation, at the cost of rebuilding the feature
frame once per worker — seconds against hours.

**Work is split statically, by group.** A *group* is one ``(arm, origin, K, H)``
cell, whose seeds share a tensor build; shards take groups round-robin. Root
§10.5's idempotence rule makes this safe with no coordination: a run is complete
only when both artifacts exist and ``status == "complete"``, so a worker that
finishes early drains whatever is still pending regardless of which shard owned
it.
"""

from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
import time
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path

import polars as pl

# Names, not the module. In the flattened notebook there is no ``baselines``
# module object to attribute off — every definition lands in one namespace — so
# ``baselines.RidgeConfig`` would be a NameError hours into a Kaggle session
# while passing every parse-level check here (root §15, `D58`).

#: Arm to the ``model`` component of ``run_id``. Distinct tags mean a changed
#: arm **orphans** prior outputs rather than silently reusing a mismatched
#: result (root §10.4).
ARM_MODEL_TAG: dict[str, str] = {
    "main": "itr",       # 15 origins x 4 K x 5 seeds, H=24
    "uniform": "itru",   # `D50` — attention forced uniform, K=8
    "fresh": "itrf",     # root §8.1 falsification arm, trained at o_i + 90 d
    "horizon": "itr",    # `D08`/`D48` — 4 named origins x 4 K x 4 H x 3 seeds
    "ridge": "rdg",      # `D17` — is a transformer needed at all? K = 1,4,8,12
    "dlinear": "dlin",   # root §7 — "not optional", K=8
    "patchtst": "ptst",  # root §7 — SOTA channel-independent, K=8
}

#: The §7 comparators (`D56`). Two things key off this set, and both follow from
#: these arms being a *different model* rather than a different configuration of
#: the same one: the `D45` window-alignment assertion runs for them, and their
#: ``run_id`` prefix keeps them apart in every table.
BASELINE_ARMS: tuple[str, ...] = ("ridge", "dlinear", "patchtst")

#: Every arm, in execution order. iTransformer first, so that a session cut short
#: leaves the ladder — which RQ1, RQ2 and RQ3 all read — complete before the
#: comparators, and so a baseline's alignment assertion finds its reference on
#: disk rather than reporting itself unchecked.
ALL_ARMS: tuple[str, ...] = ("main", "uniform", "fresh", "horizon", *BASELINE_ARMS)

#: Seeds for the horizon sweep. Three, not five: root §10.2 budgets 192 runs for
#: it, and root §10.3 says to cut the sweep before cutting seed counts if the
#: grid ever stops fitting, because `D30` and `D49` depend on the seed counts.
SWEEP_SEEDS: tuple[int, ...] = SEEDS[:3]

#: Seeds for the stochastic baselines. Three, matching root §10.2's baseline
#: budget of "3 stochastic x 3 seeds". `D49`'s five-seed rule is about the
#: **rungs of the ladder**, where the 8->12 contrast cannot be the one carrying
#: the fewest; a baseline is a single cell rather than a rung, and a fourth and
#: fifth seed there would buy precision on a number no hypothesis is stated about.
BASELINE_SEEDS: tuple[int, ...] = SEEDS[:3]

#: Root §10.5. Checked at **run boundaries**, not epoch boundaries: runs are
#: short, epochs are shorter, and the checkpoint granularity is the run.
SESSION_BUDGET_H: float = 11.0
RESERVE_H: float = 0.5


@dataclass(frozen=True, slots=True)
class RunCell:
    """One (arm, origin, K, H, seed) cell of the grid."""

    arm: str
    origin_index: int
    k: int
    pred_len: int
    seed: int

    @property
    def model_tag(self) -> str:
        return ARM_MODEL_TAG[self.arm]

    @property
    def spec(self) -> RunSpec:
        return RunSpec(
            model=self.model_tag,
            origin_index=self.origin_index,
            k=self.k,
            pred_len=self.pred_len,
            seed=self.seed,
        )

    @property
    def run_id(self) -> str:
        return self.spec.run_id

    @property
    def group(self) -> tuple[str, int, int, int]:
        """The **shard** key. Seeds inside a group stay with one worker."""
        return (self.arm, self.origin_index, self.k, self.pred_len)

    @property
    def tensor_key(self) -> tuple[bool, int, int, int]:
        """The **tensor-build** key, which is coarser than :attr:`group`.

        :func:`build_origin_tensors` reads the origin, K and H and nothing else,
        and only the falsification arm changes the origin object. So ridge at
        (origin 7, K=8, H=24) consumes byte-for-byte the tensors the main arm
        already built there, and keying the cache by arm would rebuild them —
        150 redundant builds across the baseline arms. Sharding still keys on
        :attr:`group`, because that partition must be a function of the arm.
        """
        return (self.arm == "fresh", self.origin_index, self.k, self.pred_len)

    def origin(self) -> OriginLike:
        base = ORIGINS[self.origin_index - 1]
        return FalsificationOrigin(base) if self.arm == "fresh" else base

    def model_config(self) -> Architecture:
        """The arm's configuration — hyperparameters fixed a priori in every case.

        **No per-rung tuning** (`D38`): holding capacity fixed is what makes the
        rungs comparable, and tuning per rung would confound the ladder with
        model selection. The only field an iTransformer arm may move is
        ``uniform_attention``, which *is* the arm.

        The rule extends to the §7 baselines rather than exempting them. Ridge's
        alpha is the single exception root §11 names, and it is not chosen here:
        :meth:`RidgeConfig.fit` selects it on the validation sub-block and
        returns the resolved config, which is what reaches ``meta['config']``.
        """
        if self.arm == "ridge":
            return RidgeConfig(pred_len=self.pred_len, k=self.k)
        if self.arm == "dlinear":
            return DLinearConfig(pred_len=self.pred_len)
        if self.arm == "patchtst":
            return PatchTSTConfig(pred_len=self.pred_len)
        return ITransformerConfig(
            pred_len=self.pred_len,
            uniform_attention=(self.arm == "uniform"),
        )

    def reference_run_id(self) -> str:
        """The iTransformer run this cell is compared against (`D45`).

        Same origin, same K, same horizon, first seed — the main-grid cell whose
        evaluated window set this run's must equal exactly before any RelMSE or
        DM statistic is formed across the two.
        """
        return RunSpec(
            ARM_MODEL_TAG["main"], self.origin_index, self.k, self.pred_len, SEEDS[0]
        ).run_id


def manifest(arms: tuple[str, ...] = ALL_ARMS) -> list[RunCell]:
    """Every run in the study, deduplicated and ordered.

    Root §10.2's accounting, arm by arm:

    ==========  =====  =========================================================
    Arm         Runs   Composition
    ==========  =====  =========================================================
    main          300  15 origins x 4 K x 5 seeds (`D49` — 5 at *every* rung,
                       because the 8->12 rung is RQ1's designed contrast and
                       cannot carry the fewest)
    uniform        75  15 x K=8 x 5 seeds (`D50`)
    fresh          15  one fresh model per origin at ``o_i + 90 d``
    horizon       192  4 named origins x 4 K x 4 H x 3 seeds (`D08`, `D48`)
    ridge          60  15 x 4 K, deterministic (`D17`)
    dlinear        45  15 x K=8 x 3 seeds (root §7)
    patchtst       45  15 x K=8 x 3 seeds (root §7)
    ==========  =====  =========================================================

    **48 of those cells are literally the same run.** The sweep's ``H=24`` slice
    at seeds 42-44 carries the same ``run_id`` as the corresponding main-grid
    cells, so the iTransformer union is **534 unique runs**, not 582, and the
    whole manifest is **684**. Deduplicating is not a saving quietly banked: root
    §10.4 makes ``run_id`` the identity of a run, so executing one twice would
    mean two files racing for one path.

    **The three baseline arms are new, and their absence was `D56`.** Root §7
    calls DLinear and PatchTST "not optional" and §10.2 budgets 255 baseline
    runs, but no baseline class existed and this manifest never contained one —
    so §10.2's 789 was never executable, and the study's central architectural
    comparison had no data. 150 of that 255 are built: the deferred remainder is
    ARIMA, LSTM, naive-persist and seasonal-naive, listed in ``baselines.py``
    rather than left silently unbuilt. Naive-RW needs no run at all, being
    computed inside :func:`itransformer_btc.metrics.block_metrics` on exactly the
    rows its comparator was scored on.

    Ordering is by group, so the seeds of a cell reuse one tensor build, and
    groups are emitted arm by arm so a shard split stays balanced across the
    heavy ``H=168`` cells.
    """
    cells: list[RunCell] = []

    if "main" in arms:
        cells += [
            RunCell("main", o.index, k, PRED_LEN, s)
            for o in ORIGINS for k in K_LADDER for s in SEEDS
        ]
    if "uniform" in arms:
        cells += [
            RunCell("uniform", o.index, 8, PRED_LEN, s) for o in ORIGINS for s in SEEDS
        ]
    if "fresh" in arms:
        # One seed. The arm asks whether the aged-minus-fresh gap is zero, and
        # that contrast is between two models, not between five initialisations.
        cells += [RunCell("fresh", o.index, 8, PRED_LEN, SEEDS[0]) for o in ORIGINS]
    if "horizon" in arms:
        cells += [
            RunCell("horizon", i, k, h, s)
            for i in SWEEP_ORIGIN_INDICES
            for k in K_LADDER
            for h in HORIZONS
            for s in SWEEP_SEEDS
        ]
    if "ridge" in arms:
        # One seed. Ridge is a solve, not an optimisation: a second seed would
        # reproduce the first to the last bit. The seed component of ``run_id``
        # is carried only because root §10.4 fixes the format.
        cells += [
            RunCell("ridge", o.index, k, PRED_LEN, SEEDS[0])
            for o in ORIGINS for k in K_LADDER
        ]
    if "dlinear" in arms:
        cells += [
            RunCell("dlinear", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]
    if "patchtst" in arms:
        cells += [
            RunCell("patchtst", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]

    seen: set[str] = set()
    unique: list[RunCell] = []
    for cell in cells:
        if cell.run_id not in seen:
            seen.add(cell.run_id)
            unique.append(cell)
    return unique


def discover_roots(working: Path = ARTIFACTS) -> list[Path]:
    """Artifact roots to search, working directory first.

    Root §10.5: discover by **globbing** ``/kaggle/input/*/``, never a hard-coded
    dataset slug, so the Kaggle Dataset can be renamed without editing code. Any
    input directory holding a ``preds`` folder counts, whatever it is called, and
    one nesting level is searched because Kaggle wraps some dataset uploads in an
    extra folder.
    """
    roots = [Path(working)]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        roots += sorted(p for p in kaggle_input.iterdir() if (p / "preds").is_dir())
        roots += sorted(p.parent for p in kaggle_input.glob("*/*/preds") if p.is_dir())
    seen: set[str] = set()
    return [r for r in roots if not (str(r) in seen or seen.add(str(r)))]


def completed_run_ids(roots: list[Path]) -> set[str]:
    """Run ids complete under root §10.5: both artifacts present, status complete.

    A prediction file without its meta, or a meta whose status is anything else,
    is **not** complete and the run is redone from scratch. Intra-run
    checkpointing is deliberately omitted: at ~30 s per run measured (`D57`) it
    costs far more complexity than it saves.
    """
    done: set[str] = set()
    for root in roots:
        meta_dir = Path(root) / "meta"
        if not meta_dir.is_dir():
            continue
        for meta_path in meta_dir.glob("*.json"):
            run_id = meta_path.stem
            if not (Path(root) / "preds" / f"{run_id}.parquet").exists():
                continue
            try:
                if json.loads(meta_path.read_text()).get("status") == "complete":
                    done.add(run_id)
            except (json.JSONDecodeError, OSError):
                continue
    return done


def pending(cells: list[RunCell], roots: list[Path]) -> list[RunCell]:
    """Manifest minus what is already complete, order preserved."""
    done = completed_run_ids(roots)
    return [c for c in cells if c.run_id not in done]


def shard(cells: list[RunCell], index: int, count: int) -> list[RunCell]:
    """Round-robin by **group**, so a cell's seeds share one tensor build.

    Sharding by cell instead would send consecutive seeds to different workers
    and make both build the same tensors — correct, but paying the build cost
    twice for nothing.
    """
    groups: list[tuple[str, int, int, int]] = []
    for cell in cells:
        if cell.group not in groups:
            groups.append(cell.group)
    owned = {g for i, g in enumerate(groups) if i % count == index}
    return [c for c in cells if c.group in owned]


class BudgetGuard:
    """Root §10.5's session budget, checked at run boundaries.

    Hitting Kaggle's own 12 h wall interactively loses ``/kaggle/working``
    entirely, so the guard stops early enough that Save Version still runs. It
    also refuses to *start* a run it does not expect to finish, using the
    observed mean wall time: stopping at 10.9 h and then beginning a 98 s run is
    the failure mode a naive elapsed-only check has.
    """

    def __init__(
        self, budget_h: float = SESSION_BUDGET_H, reserve_h: float = RESERVE_H
    ) -> None:
        self.deadline = time.perf_counter() + (budget_h - reserve_h) * 3600.0
        self.durations: list[float] = []

    def record(self, seconds: float) -> None:
        self.durations.append(seconds)

    @property
    def mean_run_s(self) -> float:
        return sum(self.durations) / len(self.durations) if self.durations else 120.0

    @property
    def remaining_s(self) -> float:
        return self.deadline - time.perf_counter()

    def may_start(self) -> bool:
        return self.remaining_s > self.mean_run_s


class _TensorCache:
    """Small LRU over ``(arm, origin, K, H)`` builds.

    Bounded because one build is up to 70.12 MB of training tensor plus its
    validation and test blocks (root §10.3 / `D25`); a few is comfortable in
    Kaggle's RAM, an unbounded cache across 154 groups is not.
    """

    def __init__(self, features: pl.DataFrame, size: int = 3) -> None:
        self.features = features
        self.size = size
        self._store: OrderedDict[tuple, OriginTensors] = OrderedDict()

    def get(self, cell: RunCell) -> OriginTensors:
        key = cell.tensor_key
        if key in self._store:
            self._store.move_to_end(key)
            return self._store[key]
        tensors = build_origin_tensors(
            self.features, cell.origin(), cell.k, pred_len=cell.pred_len
        )
        self._store[key] = tensors
        while len(self._store) > self.size:
            self._store.popitem(last=False)
        return tensors


@dataclass(frozen=True, slots=True)
class ExecutionSummary:
    """What one worker did, and what is left.

    ``remaining`` and ``estimated_sessions`` are printed on exit because a
    session that ends without saying how much is left forces the next one to
    re-derive it (``notebooks/CLAUDE.md``).
    """

    completed: int
    skipped: int
    failed: int
    #: Pending **in this shard**, not in the whole manifest. The notebook prints
    #: the global figure; a worker only knows its own queue.
    remaining: int
    wall_time_s: float
    mean_run_s: float

    @property
    def estimated_sessions(self) -> float:
        if self.remaining == 0:
            return 0.0
        usable = (SESSION_BUDGET_H - RESERVE_H) * 3600.0
        return self.remaining * self.mean_run_s / usable

    def __str__(self) -> str:
        return (
            f"completed {self.completed}  skipped {self.skipped}  "
            f"failed {self.failed}  remaining {self.remaining}\n"
            f"wall {self.wall_time_s / 3600:.2f} h  mean run {self.mean_run_s:.1f} s  "
            f"estimated sessions left {self.estimated_sessions:.2f}"
        )


def _assert_alignment(cell: RunCell, roots: list[Path], log) -> None:
    """`D45`, enforced when the file is written rather than when the table is built.

    Root §7 requires every baseline to be scored on **exactly** the surviving
    window set of the run it is compared against. The two sets are equal by
    construction — both come from :func:`window_starts` with the same origin,
    span and semantics — which is why this costs microseconds and why it is the
    only thing that would notice if that ever stopped holding.

    A missing comparator is **reported, never swallowed**. The check is then
    unrun, and an unrun check that prints nothing is indistinguishable from a
    passing one; :data:`ALL_ARMS` orders the ladder first precisely so this stays
    the rare case rather than the normal one.
    """
    reference = cell.reference_run_id()
    try:
        assert_baseline_alignment(cell.run_id, reference, roots)
    except FileNotFoundError:
        log(
            f"  {cell.run_id}: `D45` alignment UNCHECKED — comparator "
            f"{reference} is not on disk in {[str(r) for r in roots]}"
        )


def execute(
    cells: list[RunCell],
    features: pl.DataFrame,
    *,
    out_root: Path = ARTIFACTS,
    roots: list[Path] | None = None,
    guard: BudgetGuard | None = None,
    device=None,
    log=print,
) -> ExecutionSummary:
    """Run a shard to completion or to the budget, whichever comes first.

    A failing run is logged and skipped rather than aborting the shard: with 684
    runs, losing the rest of a session to one bad cell is worse than finishing
    the others and letting root §10.5's resume pick that cell up next session.
    A failing *invariant* is the opposite case and does end the shard — see
    :func:`_assert_alignment`.
    """
    guard = guard or BudgetGuard()
    roots = roots or discover_roots(out_root)
    device = device or pick_device()
    cache = _TensorCache(features)

    started = time.perf_counter()
    done = completed_run_ids(roots)
    completed = skipped = failed = 0
    queue = list(cells)

    for position, cell in enumerate(queue, start=1):
        if cell.run_id in done or is_complete(cell.run_id, out_root):
            skipped += 1
            continue
        if not guard.may_start():
            log(
                f"budget guard: {guard.remaining_s / 60:.1f} min left, mean run "
                f"{guard.mean_run_s:.0f} s — stopping cleanly so the version saves"
            )
            break

        began = time.perf_counter()
        try:
            tensors = cache.get(cell)
            # The config comes back **resolved**: identical for every
            # iTransformer arm (`D38` — nothing is tuned), and carrying the
            # chosen alpha for ridge, which is the one selection root §11 admits.
            # Writing the config that went in would lose it.
            model, cfg, outcome = cell.model_config().fit(
                tensors, cell.spec, device=device
            )
            write_artifacts(
                model, tensors, cell.spec, cfg, outcome, device, root=out_root,
            )
        except Exception as exc:  # noqa: BLE001 - one bad cell must not end the shard
            failed += 1
            log(f"[{position}/{len(queue)}] {cell.run_id} FAILED: {exc!r}")
            continue

        # Outside the try, and deliberately fatal. A window-set mismatch between
        # a baseline and its comparator is the defect class root §11 calls
        # fatal — RelMSE across two samples is not a ratio — and continuing would
        # fill Table 6 with statistics that mean nothing. One bad *cell* must not
        # end a shard; one broken *invariant* must.
        if cell.arm in BASELINE_ARMS:
            _assert_alignment(cell, roots, log)

        elapsed = time.perf_counter() - began
        guard.record(elapsed)
        completed += 1
        log(
            f"[{position}/{len(queue)}] {cell.run_id}  "
            f"epochs={outcome.epochs_run}  val={outcome.best_val_mse:.6f}  "
            f"{elapsed:.1f}s  n_train={len(tensors.train)}"
        )

    return ExecutionSummary(
        completed=completed,
        skipped=skipped,
        failed=failed,
        remaining=len(pending(queue, discover_roots(out_root))),
        wall_time_s=time.perf_counter() - started,
        mean_run_s=guard.mean_run_s,
    )


@dataclass(frozen=True, slots=True)
class PilotResult:
    """Root §8.5's Stage 5 gate. Note what it does **not** touch: the test blocks."""

    val_mse: dict[int, float]
    clark_west: object
    n_val: int
    passed: bool

    def __str__(self) -> str:
        rungs = "  ".join(f"K={k}: {v:.6f}" for k, v in sorted(self.val_mse.items()))
        verdict = (
            "PASS — K=8 beats K=1 on validation; keep the framing as written"
            if self.passed
            else "FAIL — reposition the title to the descriptive variant NOW, "
                 "not in week nine (root §8.5)"
        )
        return f"validation MSE  {rungs}\n{self.clark_west}\n{verdict}"


def stage5_pilot(
    features: pl.DataFrame,
    *,
    origin_index: int = 1,
    rungs: tuple[int, ...] = K_LADDER,
    seeds: tuple[int, ...] = SEEDS[:3],
    out_root: Path = ARTIFACTS,
    device=None,
    log=print,
) -> PilotResult:
    """Origin 1, 4 K x 3 seeds, scored on the **validation** sub-block (`D27`).

    §11's final item requires the test blocks be opened once, after the design is
    frozen; a gate that repositions the title on a test-block result cannot
    coexist with it. The validation sub-block is the leak-free instrument for a
    go/no-go on architecture.

    The twelve cells are ordinary main-grid ``run_id``s and their artifacts are
    written, so the pilot costs the grid nothing: §10.5's resume finds them
    complete and the main run skips them. That is deliberate and is *why* the
    gate must run on validation — with a test-block gate, the origin that decided
    the paper's framing would end up back inside the evidence for it.

    The gate statistic is **Clark-West, not DM** (`D29`): K=1's feature set is a
    strict subset of K=8's under the same architecture and sample, so the pair is
    nested and standard DM is undersized against exactly the alternative being
    tested. Predictions are averaged across seeds before the test, matching §9.1's
    order of operations.
    """
    # The *name*, not the module, for the reason given at the top of this file.
    # ``from itransformer_btc import metrics`` binds a module **object**, and the
    # flattened notebook has no such object — so ``metrics.clark_west_test`` was a
    # NameError six minutes into a Kaggle session while satisfying every check the
    # repository had (`D59`).

    device = device or pick_device()
    origin = ORIGINS[origin_index - 1]
    cache = _TensorCache(features, size=1)

    val_mse: dict[int, float] = {}
    val_pred: dict[int, "object"] = {}
    y_val = None

    import numpy as _np
    import torch as _torch

    for k in rungs:
        cell = RunCell("main", origin_index, k, PRED_LEN, seeds[0])
        tensors = cache.get(cell)
        y_val = tensors.val.y
        stacked = []
        for seed in seeds:
            spec = RunCell("main", origin_index, k, PRED_LEN, seed).spec
            model, outcome = train_one(tensors, spec, cell.model_config(), device=device)
            write_artifacts(model, tensors, spec, cell.model_config(), outcome,
                            device, root=out_root)
            stacked.append(
                predict(model, _torch.from_numpy(tensors.val.x).to(device))
            )
            log(f"pilot {spec.run_id}  val={outcome.best_val_mse:.6f}  "
                f"{outcome.wall_time_s:.1f}s")
        mean_pred = _np.mean(_np.stack(stacked), axis=0)
        val_pred[k] = mean_pred
        val_mse[k] = float(_np.mean((y_val - mean_pred) ** 2))

    small, large = min(rungs), 8
    # One loss value per forecast origin: the DM/CW series is indexed by the
    # moment the forecast was issued, not by the (origin, step) pair.
    cw = clark_west_test(
        y_val.mean(axis=1),
        val_pred[small].mean(axis=1),
        val_pred[large].mean(axis=1),
        h=PRED_LEN,
        name=f"Clark-West K={small} vs K={large} (validation)",
    )
    return PilotResult(
        val_mse=val_mse,
        clark_west=cw,
        n_val=len(y_val),
        passed=bool(cw.p_value < 0.05 and val_mse[large] < val_mse[small]),
    )


def build_feature_frame(parquet: Path = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the immutable artifact and compute the twelve variates."""

    return build_features(usable_mask(load_bars(parquet)))








In [ ]:
MODULE_NAMES = [
    'config.py',
    '__init__.py',
    'segments.py',
    'windows.py',
    'budget.py',
    'features.py',
    'splits.py',
    'model.py',
    'train.py',
    'keff.py',
    'efficiency.py',
    'metrics.py',
    'baselines.py',
    'runner.py',
]


In [ ]:
# Root section 12 asks a run to name the code that produced it, and names the git
# sha as the way. There is no git repository on Kaggle and — the cells above
# being definitions rather than files — nothing on disk to hash either. So
# the digest is taken from src/itransformer_btc/ when this notebook is generated
# and pinned here (D54b). It is the SAME number a local checkout of the same
# source reports, which is the point: a run from the notebook and a run from the
# repository must not look like different code vintages.
CODE_SHA256_OVERRIDE = "4a8f8ca09dadbf03281a2f51cffce9da97a970a4aa7971b508e9f2d984f7eee5"

# These cells are EXECUTED, not written, so a skipped one leaves a hole rather
# than a stale file — and the hole surfaces hours later, inside the grid. One
# sentinel per module, in MODULE_ORDER: cheap here, unbounded there.
_sentinels = (
    "ORIGINS", "__all__", "build_segments", "count_windows", "budget_table",
    "build_features", "build_origin_tensors", "ITransformer", "code_sha256",
    "keff_table", "seed_average", "PatchTST", "manifest",
)
_missing = [name for name in _sentinels if name not in globals()]
assert not _missing, (
    f"{len(_missing)} of {len(_sentinels)} definition cells have not run: "
    f"{_missing}. Run the Definitions cells above in order, top to bottom."
)
assert "itransformer_btc" not in sys.modules, (
    "an installed or on-path itransformer_btc package was imported. This notebook "
    "must run its OWN definitions, or every number it produces is traceable to "
    "code that is not in the cells above — exactly the dependency this format "
    "exists to remove."
)

print(f"modules defined in-kernel: {len(MODULE_NAMES)}  {MODULE_NAMES}")
print(f"code_sha256 {code_sha256()}")
print("\nThat digest goes into every meta/*.json. There is no git repository on "
      "Kaggle, so it is what the traceability contract has to name the code with "
      "— and it is the better half of the pair anyway: it identifies the code "
      "that ran, not the commit someone was standing on with a dirty tree.")


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 4px solid #ffb703; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffb703; margin: 0 0 8px 0;">&#128202; 1 &middot; Data &amp; integrity &mdash; Stage 2</h2>
  <p style="color: #b8c7e0; margin: 0;">Load the immutable artifact and assert the window budget per origin, by exact equality.</p>
  <ul style="color: #e0c68a; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li><strong>No imputation anywhere.</strong> When the exchange is down no price forms, so
    imputation is <em>undefined</em>, not merely risky &mdash; Rubin's taxonomy applies to values
    that exist but went unobserved.</li>
    <li><strong>Per origin, exact equality</strong> (<code>D45</code>). Asserted against the pooled
    4.9% it fires spuriously at fourteen of fifteen origins, gets loosened until it passes, and then
    can no longer distinguish positional drift from ordinary between-origin variation.</li>
    <li>Test blocks hold <strong>720</strong> forecast origins, not 601 (<code>D51b</code>): a test
    window's lookback may cross backwards, a training window's target may not cross forwards.</li>
    </ul>
</div>

In [ ]:
bars = usable_mask(load_bars(PARQUET))
print(f"bars {bars.height:,}  usable {int(bars['usable'].sum()):,}  "
      f"unusable {int((~bars['usable']).sum())}")

# D51c: the same 3 bars are zero-volume, zero-trade and H == L. No volume means
# no trades, and no trades means high and low never separate.
print(bars.filter(~pl.col("usable")).select(
    ["open_time", "zero_volume", "flat_bar", "zero_trades"]))

budgets = budget_table(bars)
drift = [
    (b.label, b.summary.break_runs, b.summary.excluded_positions, b.windows_measured,
     COMMITTED_TRAIN_BUDGET[b.label])
    for b in budgets
    if (b.summary.break_runs, b.summary.excluded_positions, b.windows_measured)
    != COMMITTED_TRAIN_BUDGET[b.label]
]
assert not drift, f"budget drift against the committed table: {drift}"
print(f"\nwindow budget matches docs/ORIGIN_WINDOW_BUDGET.md at all "
      f"{len(budgets)} origins (exact equality, D45)")

coverage = pl.DataFrame([
    {"origin": b.label, "train_windows": b.windows_measured,
     "loss_pct": round(b.loss_pct, 2), "closed_form_agrees": b.closed_form_agrees,
     **{f"B{i}": n for i, n in enumerate(b.test_block_starts, start=1)}}
    for b in budgets
])
print(coverage)
print(f"\ntraining range {coverage['train_windows'].min():,} … "
      f"{coverage['train_windows'].max():,} windows (raw-bar frame)")
print("The closed form disagrees wherever a segment is shorter than one window "
      "(D51a) and is kept only as an upper bound.")


<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #48cae4; margin: 0 0 8px 0;">&#129514; 2 &middot; The twelve variates</h2>
  <p style="color: #b8c7e0; margin: 0;">Per-bar functions of the current bar, except r, which uses the current and previous close. No rolling window anywhere.</p>
  <p style="color: #a5e8f0; margin: 10px 0 0 0; font-size: 0.92em;">That is a structural safety
    property, not a style choice: with no rolling window in the pipeline, the
    <code>center=True</code> leak class is <em>unrepresentable</em>. Column order is ladder order, so
    rung K is exactly the first K columns and <code>r</code> is channel 0 at every rung &mdash; which
    makes the single-channel loss one constant rather than a lookup.</p>
</div>

In [ ]:
features = build_features(bars)
print(f"feature frame {features.height:,} rows x {len(VARIATE_ORDER)} variates")
print(f"dropped {int(bars['usable'].sum()) - features.height} rows = one per segment "
      f"(D52c: r is per segment, so each segment's first bar has no predecessor)")

for k in (1, 4, 8, 12):
    print(f"  K={k:>2}: {ladder_columns(k)}")

print(features.select(VARIATE_ORDER).describe().filter(
    pl.col("statistic").is_in(["mean", "std", "min", "max"])))

# D52a: Rogers-Satchell is NOT strictly positive — it vanishes on a shadowless
# (marubozu) bar, of which 33 exist. log(RS + 1e-9) puts log kappa = -20.7 inside
# the measured support rather than leaving 33 out-of-support spikes that would
# distort the instance normalisation of every window containing one.
rs = features["log_rogers_satchell"]
print(f"\nlog_rogers_satchell: min {rs.min():.3f}  q0.1% {rs.quantile(0.001):.3f}  "
      f"median {rs.median():.3f}  at-floor {int((rs <= np.log(1e-9) + 1e-9).sum())}")
assert features.select([pl.col(c).is_finite().all() for c in VARIATE_ORDER]).row(0) \
    == tuple([True] * 12), "a variate is non-finite; the segment law did not run"


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #c77dff; margin: 0 0 8px 0;">&#128209; 3 &middot; Effective dimensionality &mdash; Stage 3b</h2>
  <p style="color: #b8c7e0; margin: 0;">K_eff is RQ1's independent variable and it is measured before a single epoch runs.</p>
  <ul style="color: #d8b4fe; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li><strong>Per origin, on that origin's own 21-month training sub-block</strong>
    (<code>D44</code>). A full-sample PR would be estimated on the same data as the outcome, making
    RQ1 partly circular &mdash; the one leakage path that survived every checklist item, because
    &sect;11 audits only the gate.</li>
    <li><strong>The gate reads the pre-first-origin span alone</strong> (<code>D02</code>), trigger
    pre-registered at PR &lt; 5.0. Its action is <em>disclosure, not a re-cut</em>
    (<code>D48</code>): <code>D01</code> leaves no second consistent cut over F1&ndash;F5, so
    "re-cut the ladder" named no reachable alternative.</li>
    <li>Reported on <strong>window-normalised</strong> features too (<code>D04</code>) &mdash;
    <code>use_norm</code> strips volatility <em>level</em>, so the 8&rarr;12 rung can flatten for a
    reason that has nothing to do with redundancy.</li>
    </ul>
</div>

In [ ]:
gate = gate_pr(features, k=8)
print(gate_verdict(gate))

t0 = time.perf_counter()
keff_tbl = keff_table(features)          # 15 origins x 4 rungs, training spans only
print(f"\nmeasured in {time.perf_counter() - t0:.0f}s")

rung_view = (
    keff_tbl.group_by("k")
    .agg(
        pl.col("pr_raw").mean().alias("PR_raw"),
        pl.col("pr_raw").std().alias("PR_raw_sd"),
        pl.col("pr_window_norm").mean().alias("PR_windownorm"),
        pl.col("stable_rank_lookback").mean().alias("stable_rank"),
        pl.col("pr_lookback_ratio").mean().alias("crosslag_share"),
        pl.col("divergence").mean().alias("divergence"),
    )
    .sort("k")
)
print(rung_view)
print(f"\ncorr(K, K_eff) = {corr_k_keff(keff_tbl):.4f}")
print("A reader is entitled to that before reading the K-vs-K_eff horse race: near "
      "1 means the two theories are close to collinear and there is little to "
      "separate, whatever the p-value says.")
print("Section 5.2 expected 1 / ~3.5 / ~6.5 / ~7, reasoned from family structure "
      "and not measured. Fix the hypothesis to the measurement, never the reverse.")

keff_tbl.write_parquet(ARTIFACTS / "keff_table.parquet")


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 4px solid #7ae582; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #7ae582; margin: 0 0 8px 0;">&#128295; 4 &middot; Pre-flight invariants &mdash; Stage 4</h2>
  <p style="color: #b8c7e0; margin: 0;">Three checks that must pass before the grid. Each failed the first time it ran.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li><code>MSE(c&middot;x)/c&sup2; == MSE(x)</code>, <strong>not</strong>
    <code>MSE(c&middot;x) == MSE(x)</code> (<code>D03</code>) &mdash; the target is a channel of the
    same array, so it scales too and the loss scales by c&sup2;. The source specification's version
    cannot pass.</li>
    <li>Single-batch overfit with <strong><code>dropout=0.0</code></strong> (<code>D52d</code>). With
    the configured 0.1 still on, the loss floors near 7e-2 and a reader following the instruction
    literally concludes the plumbing is broken when it is not.</li>
    <li>The <strong>Naive-RW baseline is computed first</strong>, before any model trains, and it is
    <code>&#375;<sub>z</sub> = &minus;&mu;<sub>g</sub>/&sigma;<sub>g</sub></code> &mdash; never 0
    (<code>D31</code>), which would silently be a constant-drift model wearing the EMH baseline's
    name.</li>
    </ul>
</div>

In [ ]:
device = pick_device()
print(f"device {device}")

set_seed(42)
probe = ITransformer(ITransformerConfig()).to(device).eval()
base, scaled = scale_invariance_check(
    probe,
    torch.randn(64, 96, 8, device=device),
    torch.randn(64, 24, device=device),
    c=100.0,
)
rel = abs(base - scaled) / base
print(f"use_norm invariance: {base:.8f} vs {scaled:.8f}  rel={rel:.2e}")
assert rel < 1e-3, "FATAL (D03): use_norm inactive, or the scaler no longer cancels"
print(f"parameters {probe.n_parameters():,} — identical at every rung by construction")

set_seed(42)
plumb = ITransformer(ITransformerConfig(dropout=0.0)).to(device).train()
xs, ys = torch.randn(8, 96, 8, device=device), torch.randn(8, 24, device=device)
opt = torch.optim.Adam(plumb.parameters(), lr=1e-3)
for _ in range(200):
    opt.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(plumb(xs), ys)
    loss.backward()
    opt.step()
print(f"single-batch overfit (dropout=0.0): {loss.item():.3e}")
assert loss.item() < 1e-3, "plumbing broken"

print("\nNaive-RW in scaler space, per origin (D31 / D52b):")
naive = pl.DataFrame([
    {"origin": o.label, **{
        "mu_g": (t := build_origin_tensors(features, o, 1)).scaler.mean[0],
        "sigma_g": t.scaler.std[0],
        "mu_over_sigma": t.scaler.target_mu_over_sigma,
        "naive_rw_z": t.naive_rw_z,
        "n_train": len(t.train),
    }}
    for o in ORIGINS
])
print(naive)
print(f"mu_g/sigma_g spans {naive['mu_over_sigma'].min():+.5f} … "
      f"{naive['mu_over_sigma'].max():+.5f} and CHANGES SIGN, so the tilt is not a "
      f"constant a reader could subtract. It tracks the same bull/bear cycle H2 "
      f"invokes as its own mechanism, which is why it is confounded with the "
      f"effect of interest and does not wash out.")
naive.write_parquet(ARTIFACTS / "naive_rw_by_origin.parquet")


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 4px solid #ff7b54; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ff7b54; margin: 0 0 8px 0;">&#128737; 5 &middot; Stage 5 gate &mdash; validation only</h2>
  <p style="color: #b8c7e0; margin: 0;">Origin 1, 4 K x 3 seeds, scored on the validation sub-block. The test blocks stay shut.</p>
  <p style="color: #ffc4a3; margin: 10px 0 0 0; font-size: 0.92em;">
    &sect;11 requires the test blocks be opened once, after the design is frozen, so a gate that
    repositions the title on a test-block result cannot coexist with it (<code>D27</code>). The
    statistic is <strong>Clark&ndash;West, not DM</strong> (<code>D29</code>): K=1's feature set is a
    strict subset of K=8's under the same architecture and sample, and standard DM is systematically
    undersized against exactly the alternative being tested. The gate is
    <strong>K=1 vs K=8, never K=12</strong> &mdash; K=12 is built to be redundant, and gating on it
    would kill a viable paper for the wrong reason. The twelve cells are ordinary main-grid
    <code>run_id</code>s, so the grid below skips them.</p>
</div>

In [ ]:
pilot = stage5_pilot(features, out_root=ARTIFACTS, device=device)
print(pilot)

if not pilot.passed:
    print("\n*** Reposition the title to the descriptive variant NOW, not in week "
          "nine (root section 8.5). ***")
print("\nDisclose the pilot in section 13.2 as a SELECTION EVENT, stated separately "
      "from the DSR trial count — the DSR does not correct for selection over a "
      "paper's conclusion.")


<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 4px solid #4cc9f0; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #4cc9f0; margin: 0 0 8px 0;">&#128640; 6 &middot; The grid</h2>
  <p style="color: #b8c7e0; margin: 0;">684 unique runs across seven arms, executed in this kernel. Resume automatic, budget guard at run boundaries.</p>
  <ul style="color: #a9d6f5; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li><strong>main 300</strong> &middot; <strong>uniform 75</strong> (<code>D50</code>) &middot;
    <strong>fresh 15</strong> (falsification) &middot; <strong>horizon 144</strong>. The sweep's
    H=24 slice shares 48 <code>run_id</code>s with the main grid, so 582 nominal cells are 534 real
    runs &mdash; executing one twice would mean two files racing for one path.</li>
    <li><strong>ridge 60</strong> (<code>D17</code>, K=1/4/8/12) &middot;
    <strong>dlinear 45</strong> &middot; <strong>patchtst 45</strong> &mdash; the &sect;7 comparators,
    absent from every earlier manifest (<code>D56</code>). Without them "iTransformer has no edge"
    rests on Naive-RW alone, and a referee reads the null as an untuned configuration rather than a
    finding. They run <em>after</em> the ladder so a short session leaves RQ1&ndash;RQ3's inputs
    complete, and so each baseline's <code>D45</code> window-alignment assertion finds its comparator
    already on disk.</li>
    <li><strong>One process, one GPU, and a second T4 idles.</strong> A real cost, stated rather than
    buried: it roughly doubles wall time against the two-worker form. It is what the notebook format
    costs &mdash; definitions live in this namespace and a subprocess inherits none of it. The
    measured arithmetic says it still fits: <strong>534 &times; ~30 s &asymp; 4.5 h</strong> for the
    ladder (<code>D57</code>).</li>
    <li><strong>PatchTST is the expensive arm, by a factor nobody guessed.</strong> Measured on CPU at
    origin 1, K=8: iTransformer 113 s over 10 epochs, ridge 0.5 s, DLinear 24 s, PatchTST
    <strong>1810 s</strong>. Per epoch that is 5.3&times; iTransformer &mdash; it folds channels into
    the batch, so a step processes B&times;N=256 sequences rather than 32 &mdash; and both
    channel-independent baselines run the full 30 epochs because early stopping never fires. Scaling
    <code>D57</code>'s 30 s/run gives ridge and DLinear ~10 min combined and PatchTST
    <strong>~6 h</strong>, putting the whole manifest near 11 h. <strong>Two sessions is therefore the
    expected case, not the exception.</strong> That is survivable precisely because the baselines run
    last: an overrun costs comparators, never RQ1&ndash;RQ3's inputs, and they resume by
    <code>run_id</code> like anything else.
    Threads are <em>not</em> the way to reclaim the second GPU:
    <code>torch.manual_seed</code> seeds <em>every</em> CUDA device, so two threads would clobber
    each other's generator mid-run and &sect;12's reproducibility contract would be
    unenforceable.</li>
    <li><strong>No <code>DataLoader</code>.</strong> At ~280k parameters the run is dominated by data
    movement and Python overhead, which a per-item loader maximises &mdash; roughly 10&times; worse,
    which puts the grid outside the 30 h weekly quota outright.</li>
    <li>Run <em>Save Version &rarr; Save &amp; Run All</em>, never the editor: the 20-minute idle
    timeout kills interactive sessions, and hitting the 12 h wall interactively loses
    <code>/kaggle/working</code> entirely.</li>
    <li><strong>Resume granularity is one run, ~30 s.</strong> A run counts as complete only when
    both <code>preds/</code> and <code>meta/</code> exist and <code>meta.status ==
    "complete"</code>, so a session cut short at run 200 of 684 loses at most the one run in flight.
    The next session subtracts what is done and continues &mdash; there is no bookkeeping to do by
    hand and no state beyond the files themselves. Intra-run checkpointing is deliberately omitted:
    at ~30 s per run it costs more complexity than it saves.</li>
    <li><strong>The budget guard bounds the session, not the grid call.</strong> Kaggle's 12 h wall
    starts at cell 0, so the prelude &mdash; data, K<sub>eff</sub>, invariants, the twelve pilot
    runs &mdash; is subtracted before the guard is built. Counting from the grid's own start would
    let the two clocks drift apart by exactly however long the prelude took.</li>
  </ul>
</div>

In [ ]:
import gc

# The pre-flight probe and the pilot allocated on this same device, and the grid
# is about to. Hand the memory back before it starts rather than carrying two
# dead models through 684 runs.
for _name in ("probe", "plumb", "xs", "ys", "opt", "loss"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ALL = manifest()
roots = discover_roots(ARTIFACTS)
todo = pending(ALL, roots)

by_arm = {}
for cell in ALL:
    by_arm[cell.arm] = by_arm.get(cell.arm, 0) + 1
already_done = len(ALL) - len(todo)
print(f"manifest {len(ALL)} unique runs {by_arm}")
print(f"roots searched: {[str(r) for r in roots]}")
print(f"already complete: {already_done}   pending: {len(todo)}")
print("Completeness is per run_id: both preds/ and meta/ present and "
      "meta.status == 'complete'. A run interrupted mid-training leaves no meta, "
      "so it is redone — losing at most one run, ~30 s (root §10.5).")

# The budget is what is LEFT of the session, not a fresh 11 h.
SESSION_BUDGET_H = 11.0
elapsed_h = (time.perf_counter() - SESSION_T0) / 3600.0
budget_h = max(0.25, SESSION_BUDGET_H - elapsed_h)
print(f"\nprelude took {elapsed_h * 60:.0f} min -> grid gets {budget_h:.2f} h "
      f"of the {SESSION_BUDGET_H:.1f} h session budget, guard reserves 0.5 h more")

# In-kernel and sequential. The definitions live in THIS namespace and nowhere
# else, so a subprocess could not reach them — and at ~30 s per run measured
# (D57) the 534 iTransformer cells are ~4.5 h, which fits without a second
# worker. The 150 baseline cells (D56) are not measured on a T4; they run last,
# so an overrun costs the comparators and never RQ1-RQ3's inputs. The second GPU
# idles; that is the price of the format, and it is stated rather than hidden.
if torch.cuda.device_count() > 1:
    print(f"NOTE: {torch.cuda.device_count()} GPUs visible, using {device} only. "
          f"Threads are not the fix — torch.manual_seed seeds EVERY CUDA device, "
          f"so two threads would clobber each other's generator mid-run.")

t0 = time.perf_counter()
summary = execute(
    todo, features,
    out_root=ARTIFACTS,
    roots=roots,
    guard=BudgetGuard(budget_h, 0.5),
    device=device,
    log=lambda msg: print(msg, flush=True),
)
print(summary)
print(f"grid finished after {(time.perf_counter() - t0) / 3600:.2f} h")

left = pending(ALL, discover_roots(ARTIFACTS))
GRID_COMPLETE = not left
print(f"remaining after this session: {len(left)} of {len(ALL)}")
if left:
    # The evaluation below is GATED on this. A partial grid is an unbalanced
    # panel, and §9.1's estimators refuse one by design — `amplification` raises
    # rather than silently comparing K=1 at eleven origins against K=8 at ten.
    # Letting that exception reach Kaggle would mark the version failed at the
    # exact moment its output is the only thing worth keeping.
    print("\nEvaluation is SKIPPED this session — the panel is incomplete, and a")
    print("half-panel beta1 is not a smaller answer but a different estimand.")
    print("Nothing is lost: preds/ and meta/ are on disk and resume is by run_id.")
    print("\n  1. Save Version now (its output IS the session's work)")
    print("  2. Attach that output as the next session's input Dataset")
    print("  3. Run this notebook again — completed runs are skipped automatically")

    ran = len(ALL) - len(left) - already_done
    if ran > 0:
        mean_s = (time.perf_counter() - t0) / ran
        print(f"\n  {ran} runs this session at ~{mean_s:.0f}s each -> about "
              f"{len(left) * mean_s / 3600:.1f} h of wall still to do, "
              f"{len(left) * mean_s / 3600 / 10.5:.1f} more sessions")


<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #e0aaff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0;">&#128200; 7 &middot; Evaluation &mdash; RQ1, RQ2, RQ3</h2>
  <p style="color: #b8c7e0; margin: 0;">Every number below resolves to a persisted prediction file and a config hash, or it does not enter the manuscript.</p>
  <p style="color: #d8b4fe; margin: 10px 0 0 0; font-size: 0.92em;">Ratio metrics are formed
    from <strong>seed-averaged MSEs</strong>, never from an average of per-seed ratios
    (<code>D42</code>): the two differ by Jensen, and the second would require pairing seed 42 at K=1
    with seed 42 at K=8 &mdash; independent training runs of different models, where any of 5!
    orderings gives a different answer.</p>
</div>

In [ ]:
if not GRID_COMPLETE:
    print("RQ1: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    done = sorted(completed_run_ids(discover_roots(ARTIFACTS)))
    grid = gather_grid(done, discover_roots(ARTIFACTS))
    seed_avg = seed_average(grid)
    print(f"gathered {len(done)} runs -> {grid.height} run-block rows -> "
          f"{seed_avg.height} seed-averaged cells")

    main = seed_avg.filter((pl.col("model") == "itr") & (pl.col("pred_len") == 24))

    # Root section 9.2 / D30: any number aggregated across origins carries the SE
    # ACROSS ORIGINS, never the seed std. Seed dispersion measures re-initialisation
    # noise on one fixed dataset; origin dispersion measures the sampling variability
    # of the estimand, and in walk-forward crypto evaluation the second is typically
    # an order of magnitude larger. Reporting the first as "+/-" on an aggregated row
    # understates the headline uncertainty by roughly that factor — reintroducing,
    # through the reporting convention, the overstated precision the wild cluster
    # bootstrap was added to prevent.
    per_origin = main.group_by(["origin", "k"]).agg(
        pl.col("mse").mean().alias("mse"), pl.col("r2_oos").mean().alias("r2_oos")
    )
    rung = (
        per_origin.group_by("k")
        .agg(
            pl.col("mse").mean().alias("MSE"),
            (pl.col("mse").std() / pl.col("mse").count().sqrt()).alias("SE_across_origins"),
            pl.col("r2_oos").mean().alias("R2_oos"),
            pl.col("mse").count().alias("n_origins"),
        )
        .sort("k")
    )
    print("\n--- RQ1: free rung effects ---")
    print(rung)
    print(f"seed std, a Monte-Carlo diagnostic only: {main['mse_seed_std'].mean():.6f} "
          f"mean across cells at n={main['n_seeds'][0]} seeds per cell")

    wide = {int(k): per_origin.filter(pl.col("k") == k).sort("origin")["mse"].to_numpy()
            for k in (1, 4, 8, 12)}
    d_4_8, d_8_12 = wide[4] - wide[8], wide[8] - wide[12]
    margin = 0.25 * abs(float(d_4_8.mean()))
    print(f"\ndelta MSE 4->8 = {d_4_8.mean():+.6f}   8->12 = {d_8_12.mean():+.6f}")
    print("D49's margin is 0.25 x delta(4->8), fixed in advance: a non-significant "
          "delta is a failure to reject, not evidence of equivalence, and choosing the "
          "margin after seeing the rung is the p-hacking section 3 forbids for tau.")
    print(tost_equivalence(d_8_12, margin))

    # D32: RQ1 is a NON-NESTED comparison, not an OLS on three points. Four rungs give
    # three deltas, and stacking 360 rows creates no information about a slope that
    # varies only between rungs. K_eff measured per origin is what makes the regressor
    # vary at all — and it is leak-free because the span is training-only.
    keff_join = keff_tbl.select(["origin", "k", "pr_raw"]).rename({"pr_raw": "k_eff"})
    race = main.join(keff_join, on=["origin", "k"], how="inner")
    groups = race["origin_index"].to_numpy() * 100 + race["block"].to_numpy()
    t_ab, p_ab = j_test(race["mse"].to_numpy(),
                        race["k"].to_numpy().astype(float),
                        race["k_eff"].to_numpy(), groups)
    t_ba, p_ba = j_test(race["mse"].to_numpy(), race["k_eff"].to_numpy(),
                        race["k"].to_numpy().astype(float), groups)
    print(f"\nJ-test  K augmented by K_eff: t={t_ab:+.3f} p={p_ab:.4f}   |   "
          f"K_eff augmented by K: t={t_ba:+.3f} p={p_ba:.4f}")
    print("Both reject -> neither explanation alone suffices. Neither rejects -> the "
          "data cannot separate them, which at corr(K, K_eff) near 1 is the outcome to "
          "expect and to report plainly rather than to spin.")


In [ ]:
if not GRID_COMPLETE:
    print("RQ2: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    print("--- RQ2: does the multivariate gap narrow with model age? ---")
    amp = amplification(seed_avg, k_small=1, k_large=8)
    print(amp.select(["origin", "block", "mse_small", "mse_large", "A"]).head(12))

    beta = panel_beta1(amp, value="A", B=99_999, seed=42)
    print(f"\n{beta}")
    mde = minimum_detectable_beta1(beta.within_slopes)
    print(f"\nminimum detectable beta1 at 80% power, alpha=0.05: {mde:+.6f}")
    print(f"observed {beta.beta1:+.6f} is "
          f"{'INSIDE (undetectable)' if abs(beta.beta1) < abs(mde) else 'outside'} it")
    print("If the MDE exceeds the plausible magnitude of A, RQ2 must be repositioned as "
          "descriptive BEFORE the grid: a non-significant beta1 is otherwise "
          "indistinguishable from a design that could never have detected decay.")

    # D28: consecutive origins share 79.2% of their training data, so the clusters are
    # NOT independent draws and the bootstrapped p is anticonservative by an
    # unquantified amount. Windows become disjoint only at stride 5.
    print("\ntraining-window overlap: 79.2% at stride 1, 58.3% at 2, 37.5% at 3, "
          "16.7% at 4. Disjoint only at stride 5, which leaves G=3.")
    for offset in range(5):
        triple = [ORIGINS[i].label for i in range(offset, len(ORIGINS), 5)]
        subset = amp.filter(pl.col("origin").is_in(triple))
        if subset.height == len(triple) * 6:
            sub = panel_beta1(subset, value="A", B=9_999, seed=42)
            print(f"  {triple}: beta1={sub.beta1:+.6f}  p={sub.headline_p:.4f}  (G=3)")
    print("At G=3 these will very likely be inconclusive, and THAT IS THE FINDING — it "
          "bounds what the full-panel p-value can honestly claim.")

    # D50: K=1 vs K=8 differs in information AND in whether attention is active, at the
    # same time. This holds information fixed and varies only what attention selects.
    try:
        attn = attention_amplification(seed_avg, k=8)
        print(f"\nA_attn (uniform-attention control, D50):")
        print(panel_beta1(attn, value="A_attn", B=99_999, seed=42))
        print("Reporting both decompositions answers information-versus-attention "
              "directly, at runs Figure 5 needs anyway.")
    except (ValueError, KeyError) as exc:
        print(f"\nuniform-attention arm not complete yet: {exc}")

    # The falsification arm: the only design that identifies decay directly.
    aged = main.filter((pl.col("k") == 8) & (pl.col("block") >= 4)).select(
        ["origin_index", "block", "mse"]).rename({"mse": "mse_aged"})
    fresh = seed_avg.filter(pl.col("model") == "itrf").select(
        ["origin_index", "block", "mse"]).rename({"mse": "mse_fresh"})
    falsify = aged.join(fresh, on=["origin_index", "block"], how="inner")
    if falsify.height:
        gap = (falsify["mse_aged"] - falsify["mse_fresh"]).to_numpy()
        print(f"\nfalsification arm: mean(aged - fresh) = {gap.mean():+.6f} over "
              f"{len(gap)} (origin, block) cells")
        print("Positive means the fresh model really is better, so decay is age. Near "
              "zero while beta1 < 0 means beta1 is CALENDAR, not age, and RQ2's "
              "headline is an artefact.")
    else:
        print("\nfalsification arm not complete yet")


In [ ]:
if not GRID_COMPLETE:
    print("RQ3: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    print("--- RQ3: what retraining cadence? ---")
    dec = decay(seed_avg, k=8)
    print(dec.table)
    if dec.excluded_origins:
        print(f"\nEXCLUDED, mean R2_oos <= 0 so there is no edge to lose a proportion "
              f"of: {list(dec.excluded_origins)}")
        print("Named, never silently dropped. Root section 10.3's first measured run "
              "returned R2_oos = -0.0183, so this guard may be the common case rather "
              "than the edge case section 9.1 assumed.")

    print("\ntau sensitivity — headline 5%, pre-registered before the curve was seen:")
    b_star_rows = []

    if not dec.table.height:
        # Every origin failed the R2_oos > 0 guard, so D(i,b) has no denominator
        # anywhere and b* has nothing to estimate (D55). This is NOT censoring: a
        # censored origin has an edge that never decays past tau within 180 days,
        # whereas here there is no edge to lose a proportion of. Reporting the two
        # in one wording would claim skill the grid never found.
        for tau in TAU_SENSITIVITY:
            flag = "   <-- HEADLINE" if abs(tau - TAU_HEADLINE) < 1e-9 else ""
            print(f"  tau={tau:>6.1%}  UNDEFINED — no origin has positive mean skill{flag}")
            b_star_rows.append({"tau": tau, "status": "undefined", "median_b_star": None,
                                "ci_low": None, "ci_high": None, "events": 0,
                                "censored": 0, "n_origins": 0})
        print(f"\nRQ3 RETURNS NO ANSWER, and that is the finding. D(i,b) is a proportion "
              f"of skill lost; all {len(dec.excluded_origins)} origins have mean "
              f"R2_oos <= 0, so the proportion is undefined rather than large or small.")
        print("Root section 9.1's guard was written for an edge case and is here the "
              "ONLY case. Report it as 'the decay estimand is undefined under "
              "non-positive out-of-sample skill' — never as 'no decay detected within "
              "180 days', which is the right-censored wording and asserts an edge.")
    else:
        for tau in TAU_SENSITIVITY:
            bs = dec.b_star(tau)
            km = kaplan_meier(bs["b_star"].to_numpy(), bs["event"].to_numpy())
            lo, hi = km.median_interval
            median = "censored >6" if km.median == float("inf") else f"{km.median:.0f}"
            interval = "censored" if lo == float("inf") else f"[{lo:.0f}, {hi:.0f}]"
            flag = "   <-- HEADLINE" if abs(tau - TAU_HEADLINE) < 1e-9 else ""
            print(f"  tau={tau:>6.1%}  crossings {km.n_events}/"
                  f"{km.n_events + km.n_censored}  median b* {median}  CI {interval}{flag}")
            b_star_rows.append({"tau": tau, "status": "estimated",
                                "median_b_star": km.median, "ci_low": lo,
                                "ci_high": hi, "events": km.n_events,
                                "censored": km.n_censored, "n_origins": bs.height})

        print("\nb* resolves only to 30-day granularity and only out to 180 days. If no "
              "block crosses tau, the honest answer is 'no decay detected within 180 days' "
              "— a right-censored result, not a missing one. Say it in those words, and put "
              "the INTERVAL in the abstract, never a bare integer.")

    # H3: larger K decays faster. Needs surviving origins AND at least one crossing:
    # with zero events in both arms the log-rank variance is zero and the statistic
    # is 0/0, which prints as nan and reads like a computed result. Section 12 calls
    # a number that cannot be regenerated a documented failure, so say why instead.
    a = dec.b_star(TAU_HEADLINE)
    b = decay(seed_avg, k=1).b_star(TAU_HEADLINE)
    events = (int(a["event"].sum()) if a.height else 0,
              int(b["event"].sum()) if b.height else 0)
    if a.height and b.height and sum(events):
        chi2, p = logrank(a["b_star"].to_numpy(), a["event"].to_numpy(),
                          b["b_star"].to_numpy(), b["event"].to_numpy())
        print(f"\nlog-rank K=8 vs K=1 at tau=5%: chi2={chi2:.3f}  p={p:.4f}  "
              f"(H3; crossings K=8 {events[0]}, K=1 {events[1]})")
    elif not (a.height and b.height):
        print(f"\nlog-rank K=8 vs K=1 UNAVAILABLE: surviving origins K=8 {a.height}, "
              f"K=1 {b.height}. H3 compares decay RATES, so it needs an edge in both "
              "arms; with none, H3 is untestable rather than rejected.")
    else:
        print(f"\nlog-rank K=8 vs K=1 UNAVAILABLE: zero crossings in both arms "
              f"({a.height} and {b.height} origins, all censored at 6). The statistic "
              "is 0/0 here, not a large p-value — H3 is untestable, and reporting a "
              "nan as though it were computed would be the same defect as D55.")


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #bf5af2; margin: 0 0 8px 0;">&#128190; 8 &middot; Save</h2>
  <p style="color: #b8c7e0; margin: 0;">Every table and figure is generated FROM paper_numbers.json, never transcribed.</p>
  <p style="color: #c77dff; margin: 10px 0 0 0; font-size: 0.92em;">Numbers produced under
    different input-artifact hashes are not comparable and must not share a table, so the parquet
    digest travels with them &mdash; and so does <code>code_sha256</code>, which is what identifies
    the code off-repo. A number that cannot be regenerated is a documented failure, not a
    footnote.</p>
</div>

In [ ]:
if not GRID_COMPLETE:
    print("paper_numbers.json: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    _digest, _provenance = _input_sha256(PARQUET)

    paper_numbers = {
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "input_parquet": str(PARQUET),
        "input_sha256": _digest,
        "input_sha256_source": _provenance,
        "code_sha256": code_sha256(),
        "runs_complete": len(done),
        "runs_in_manifest": len(ALL),
        "keff": {
            "gate_pr_k8_pre_first_origin": gate,
            "gate_floor": GATE_PR_FLOOR,
            "corr_k_keff": corr_k_keff(keff_tbl),
            "per_rung": rung_view.to_dicts(),
        },
        "rq1": {
            "rung_effects": rung.to_dicts(),
            "delta_4_to_8": float(d_4_8.mean()),
            "delta_8_to_12": float(d_8_12.mean()),
            "tost_margin": margin,
            "tost": str(tost_equivalence(d_8_12, margin)),
            "j_test_k_augmented_by_keff": {"t": t_ab, "p": p_ab},
            "j_test_keff_augmented_by_k": {"t": t_ba, "p": p_ba},
        },
        "rq2": {
            "beta1": beta.beta1, "t": beta.t_statistic, "cluster_se": beta.cluster_se,
            "p_rademacher": beta.p_rademacher, "p_webb": beta.p_webb,
            "headline_p": beta.headline_p, "G": beta.n_clusters,
            "N": beta.n_observations, "B": beta.B,
            "minimum_detectable_beta1": mde,
            "within_slopes": beta.within_slopes.tolist(),
            "effective_independent_training_sets": 4,
            "consecutive_origin_overlap_pct": 79.2,
        },
        "rq3": {
            "tau_headline": TAU_HEADLINE,
            "b_star": b_star_rows,
            "excluded_origins": list(dec.excluded_origins),
        },
    }

    out = ARTIFACTS / "paper_numbers.json"
    out.write_text(json.dumps(paper_numbers, indent=2, default=float))
    seed_avg.write_parquet(ARTIFACTS / "seed_averaged_cells.parquet")
    grid.write_parquet(ARTIFACTS / "run_block_metrics.parquet")
    amp.write_parquet(ARTIFACTS / "amplification_panel.parquet")
    dec.table.write_parquet(ARTIFACTS / "decay_panel.parquet")

    print(f"wrote {out}")
    for path in sorted(ARTIFACTS.glob("*.parquet")):
        print(f"  {path.name}  {path.stat().st_size / 1e6:.2f} MB")
    print(f"\npreds {len(list((ARTIFACTS / 'preds').glob('*.parquet')))} files  |  "
          f"meta {len(list((ARTIFACTS / 'meta').glob('*.json')))} files")
    print(f"remaining runs: {len(pending(ALL, discover_roots(ARTIFACTS)))}")
    print("\nSave Version now, then attach this output as the next session's input "
          "Dataset. Nothing else needs doing by hand.")
